<a href="https://colab.research.google.com/github/AchuchoNoel237/-AI-Powered-Cattle-Grazing-Route-Optimization-and-Pasture-Suitability-System/blob/main/Pasture_transhumance_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Phase 1

###Install and authenticate Earth Engine and Define the Adamawa region boundary

In [ ]:
# ============================================================
# PHASE 1 — CELL 1
# Setup, GEE Authentication, and Study Area Definition
# Project: Dynamic Pasture Suitability Mapping — Adamawa, Cameroon
# ============================================================

# --- Install/verify required packages (Colab usually has most pre-installed) ---
!pip install -q geemap earthengine-api

import ee
import geemap
import os

# --- Authenticate and Initialize Earth Engine ---
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize(project='pasture-mapping-project')

print("✅ Earth Engine initialized successfully")

# ============================================================
# SESSION CONSTANTS — these will be redefined in every session
# restore cell going forward so we never lose them on disconnect
# ============================================================

# --- Google Drive export folder ---
DRIVE_FOLDER = "pasture-mapping-adamawa"

# --- Target CRS for all analysis ---
TARGET_CRS = "EPSG:32633"  # UTM Zone 33N

# --- Study period ---
START_YEAR = 2016
END_YEAR = 2025  # current year — CHIRPS/MODIS 2025 may be incomplete, handled later

# --- Study area: FAO GAUL 2015, Level 1, Adamaoua Region ---
gaul_level1 = ee.FeatureCollection("FAO/GAUL/2015/level1")

study_area = gaul_level1.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Cameroon"),
        ee.Filter.eq("ADM1_NAME", "Adamaoua")  # exact GAUL spelling
    )
)

# Sanity check — must return exactly 1 feature
count = study_area.size().getInfo()
print(f"Study area features found: {count}")
if count != 1:
    print("⚠️ WARNING: Expected exactly 1 feature for Adamaoua. Check ADM1_NAME spelling or GAUL dataset.")
else:
    print("✅ Adamaoua region successfully located in FAO GAUL 2015 level 1")

# --- Extract geometry and bounds (bounds used for all exports to avoid transform errors) ---
study_geom = study_area.geometry()
study_bounds = study_geom.bounds()

# --- Get approximate area for sanity check (km²) ---
area_km2 = study_geom.area().divide(1e6).getInfo()
print(f"Adamawa region area: {area_km2:,.0f} km²")

# ============================================================
# QUICK VISUAL CHECK (static, avoids the geemap tile 'referrer'
# harmless error — uses ROADMAP basemap instead of default)
# ============================================================
Map = geemap.Map(center=[7.3, 13.0], zoom=7, basemap="ROADMAP")
Map.addLayer(study_area, {"color": "red"}, "Adamaoua Boundary")
Map.centerObject(study_area, 7)
Map

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 67.9 MB/s eta 0:00:00


MessageError: Error: credential propagation was unsuccessful

###MODIS NDVI monthly export (2016–2025), one multi-band GeoTIFF per year.

In [ ]:
# ============================================================
# PHASE 1 — CELL 2
# MODIS NDVI Monthly Export (MOD13A3) — 2016 to 2025
# 1 km resolution, exported year-by-year as multi-band GeoTIFF
# NOTE: NO scale factor applied here — applied later in
# Phase 2 preprocessing to avoid Can't transform (0.0, -1200.0)
# ============================================================

# --- Session restore check (in case this is a fresh session) ---
# If you restarted Colab, re-run Cell 1 first before this cell.
assert 'study_bounds' in dir(), "⚠️ Run Cell 1 first — study_bounds not defined."

# --- Load MODIS NDVI monthly collection ---
modis_ndvi = ee.ImageCollection("MODIS/061/MOD13A3").select("NDVI")

def get_year_ndvi_stack(year):
    """
    Build a multi-band image for a given year where each band
    is one month's NDVI (raw, unscaled — Int16 as provided by MODIS).
    Bands are named NDVI_01 ... NDVI_12.
    """
    months = ee.List.sequence(1, 12)

    def month_image(m):
        m = ee.Number(m)
        start = ee.Date.fromYMD(year, m, 1)
        end = start.advance(1, "month")
        monthly = modis_ndvi.filterDate(start, end).mean()
        band_name = ee.String("NDVI_").cat(ee.Number(m).format("%02d"))
        return monthly.rename([band_name]).toFloat()  # cast to Float32-compatible

    monthly_images = ee.ImageCollection.fromImages(months.map(month_image))
    stack = monthly_images.toBands()

    # toBands() prefixes band names with index — rename cleanly
    clean_names = [f"NDVI_{m:02d}" for m in range(1, 13)]
    stack = stack.rename(clean_names)

    return stack.clip(study_bounds).set("year", year)


# --- Export loop: one task per year, 2016–2025 ---
ndvi_tasks = []

for year in range(START_YEAR, END_YEAR + 1):
    yearly_stack = get_year_ndvi_stack(year)

    # Use clipToCollection equivalent behavior: clip to study_area boundary
    yearly_stack = yearly_stack.clipToCollection(study_area)

    task = ee.batch.Export.image.toDrive(
        image=yearly_stack,
        description=f"NDVI_monthly_{year}",
        folder=DRIVE_FOLDER,
        fileNamePrefix=f"NDVI_monthly_{year}",
        region=study_bounds,          # bounds, not raw geometry
        scale=1000,                   # 1 km resolution
        crs=TARGET_CRS,
        maxPixels=1e13,
        fileFormat="GeoTIFF"
    )
    task.start()
    ndvi_tasks.append((year, task))
    print(f"🚀 Started export task: NDVI_monthly_{year}")

print("\n✅ All NDVI export tasks submitted.")
print("⏳ Monitor progress at: https://code.earthengine.google.com/tasks")
print(f"   {len(ndvi_tasks)} tasks running (years {START_YEAR}–{END_YEAR})")

# --- Optional: check task status programmatically ---
def check_ndvi_task_status():
    for year, task in ndvi_tasks:
        status = task.status()
        print(f"{year}: {status['state']}")

# Run this anytime later to check progress:
# check_ndvi_task_status()

###CHIRPS rainfall monthly export (2016–2025), aggregated from the PENTAD collection.

In [ ]:
# ============================================================
# PHASE 1 — CELL 3
# CHIRPS Rainfall Monthly Export — 2016 to 2025
# Aggregated from UCSB-CHG/CHIRPS/PENTAD to monthly totals
# 5 km resolution
# ============================================================

# --- Session restore check ---
assert 'study_bounds' in dir(), "⚠️ Run Cell 1 first — study_bounds not defined."

# --- Load CHIRPS PENTAD collection (monthly collection ID is deprecated/unreliable) ---
chirps_pentad = ee.ImageCollection("UCSB-CHG/CHIRPS/PENTAD").select("precipitation")

# --- Check data availability for the current year (2025) before exporting ---
current_year = END_YEAR  # 2025
latest_available = chirps_pentad.sort("system:time_start", False).first()
latest_date = ee.Date(latest_available.get("system:time_start")).format("YYYY-MM-dd").getInfo()
print(f"📅 Latest CHIRPS PENTAD image available: {latest_date}")
print("   (If this date is well before Dec 2025, later 2025 months will be padded")
print("    with the 2016–2024 climatology mean during preprocessing — see error note 11)")

def get_year_rainfall_stack(year):
    """
    Build a multi-band image for a given year where each band
    is one month's total rainfall (sum of pentads within that month).
    Bands named RAIN_01 ... RAIN_12. Missing months (e.g. late 2025)
    will simply be absent from the stack — handled in Phase 2.
    """
    months = ee.List.sequence(1, 12)

    def month_image(m):
        m = ee.Number(m)
        start = ee.Date.fromYMD(year, m, 1)
        end = start.advance(1, "month")
        monthly_total = chirps_pentad.filterDate(start, end).sum()
        band_name = ee.String("RAIN_").cat(ee.Number(m).format("%02d"))
        return monthly_total.rename([band_name]).toFloat().set(
            "month", m, "n_images", chirps_pentad.filterDate(start, end).size()
        )

    monthly_images = ee.ImageCollection.fromImages(months.map(month_image))
    return monthly_images


# --- Export loop: one task per year, 2016–2025 ---
chirps_tasks = []

for year in range(START_YEAR, END_YEAR + 1):
    monthly_collection = get_year_rainfall_stack(year)

    # Only keep months that actually have pentad data (avoids blank bands for
    # not-yet-occurred 2025 months)
    monthly_list = monthly_collection.toList(12)
    n_months = monthly_collection.size().getInfo()

    if n_months == 0:
        print(f"⚠️ Skipping {year} — no CHIRPS data available yet.")
        continue

    # Build stack only from months with n_images > 0
    valid_images = []
    for i in range(12):
        img = ee.Image(monthly_list.get(i))
        n_imgs = img.get("n_images").getInfo()
        if n_imgs and n_imgs > 0:
            valid_images.append(img)
        else:
            month_num = i + 1
            print(f"   ⚠️ {year}-{month_num:02d}: no pentad data — will be padded in Phase 2")

    if len(valid_images) == 0:
        print(f"⚠️ Skipping {year} — zero valid months.")
        continue

    stack = ee.ImageCollection(valid_images).toBands()

    band_names = [f"RAIN_{int(img.get('month').getInfo()):02d}" for img in valid_images]
    stack = stack.rename(band_names)
    stack = stack.clip(study_bounds)

    task = ee.batch.Export.image.toDrive(
        image=stack,
        description=f"CHIRPS_monthly_{year}",
        folder=DRIVE_FOLDER,
        fileNamePrefix=f"CHIRPS_monthly_{year}",
        region=study_bounds,
        scale=5000,                 # 5 km resolution
        crs=TARGET_CRS,
        maxPixels=1e13,
        fileFormat="GeoTIFF"
    )
    task.start()
    chirps_tasks.append((year, task, band_names))
    print(f"🚀 Started export task: CHIRPS_monthly_{year} ({len(band_names)} months)")

print("\n✅ All CHIRPS export tasks submitted.")
print("⏳ Monitor progress at: https://code.earthengine.google.com/tasks")

# --- Optional: check task status programmatically ---
def check_chirps_task_status():
    for year, task, bands in chirps_tasks:
        status = task.status()
        print(f"{year}: {status['state']} — bands exported: {bands}")

# Run this anytime later to check progress:
# check_chirps_task_status()

###CHIRPS Rainfall Monthly Export — 2016 to 2025

In [ ]:
# ============================================================
# PHASE 1 — CELL 3
# CHIRPS Rainfall Monthly Export — 2016 to 2025
# Aggregated from UCSB-CHG/CHIRPS/PENTAD to monthly totals
# 5 km resolution
# ============================================================

# --- Session restore check ---
assert 'study_bounds' in dir(), "⚠️ Run Cell 1 first — study_bounds not defined."

# --- Load CHIRPS PENTAD collection (monthly collection ID is deprecated/unreliable) ---
chirps_pentad = ee.ImageCollection("UCSB-CHG/CHIRPS/PENTAD").select("precipitation")

# --- Check data availability for the current year (2025) before exporting ---
current_year = END_YEAR  # 2025
latest_available = chirps_pentad.sort("system:time_start", False).first()
latest_date = ee.Date(latest_available.get("system:time_start")).format("YYYY-MM-dd").getInfo()
print(f"📅 Latest CHIRPS PENTAD image available: {latest_date}")
print("   (If this date is well before Dec 2025, later 2025 months will be padded")
print("    with the 2016–2024 climatology mean during preprocessing — see error note 11)")

def get_year_rainfall_stack(year):
    """
    Build a multi-band image for a given year where each band
    is one month's total rainfall (sum of pentads within that month).
    Bands named RAIN_01 ... RAIN_12. Missing months (e.g. late 2025)
    will simply be absent from the stack — handled in Phase 2.
    """
    months = ee.List.sequence(1, 12)

    def month_image(m):
        m = ee.Number(m)
        start = ee.Date.fromYMD(year, m, 1)
        end = start.advance(1, "month")
        monthly_total = chirps_pentad.filterDate(start, end).sum()
        band_name = ee.String("RAIN_").cat(ee.Number(m).format("%02d"))
        return monthly_total.rename([band_name]).toFloat().set(
            "month", m, "n_images", chirps_pentad.filterDate(start, end).size()
        )

    monthly_images = ee.ImageCollection.fromImages(months.map(month_image))
    return monthly_images


# --- Export loop: one task per year, 2016–2025 ---
chirps_tasks = []

for year in range(START_YEAR, END_YEAR + 1):
    monthly_collection = get_year_rainfall_stack(year)

    # Only keep months that actually have pentad data (avoids blank bands for
    # not-yet-occurred 2025 months)
    monthly_list = monthly_collection.toList(12)
    n_months = monthly_collection.size().getInfo()

    if n_months == 0:
        print(f"⚠️ Skipping {year} — no CHIRPS data available yet.")
        continue

    # Build stack only from months with n_images > 0
    valid_images = []
    for i in range(12):
        img = ee.Image(monthly_list.get(i))
        n_imgs = img.get("n_images").getInfo()
        if n_imgs and n_imgs > 0:
            valid_images.append(img)
        else:
            month_num = i + 1
            print(f"   ⚠️ {year}-{month_num:02d}: no pentad data — will be padded in Phase 2")

    if len(valid_images) == 0:
        print(f"⚠️ Skipping {year} — zero valid months.")
        continue

    stack = ee.ImageCollection(valid_images).toBands()

    band_names = [f"RAIN_{int(img.get('month').getInfo()):02d}" for img in valid_images]
    stack = stack.rename(band_names)
    stack = stack.clip(study_bounds)

    task = ee.batch.Export.image.toDrive(
        image=stack,
        description=f"CHIRPS_monthly_{year}",
        folder=DRIVE_FOLDER,
        fileNamePrefix=f"CHIRPS_monthly_{year}",
        region=study_bounds,
        scale=5000,                 # 5 km resolution
        crs=TARGET_CRS,
        maxPixels=1e13,
        fileFormat="GeoTIFF"
    )
    task.start()
    chirps_tasks.append((year, task, band_names))
    print(f"🚀 Started export task: CHIRPS_monthly_{year} ({len(band_names)} months)")

print("\n✅ All CHIRPS export tasks submitted.")
print("⏳ Monitor progress at: https://code.earthengine.google.com/tasks")

# --- Optional: check task status programmatically ---
def check_chirps_task_status():
    for year, task, bands in chirps_tasks:
        status = task.status()
        print(f"{year}: {status['state']} — bands exported: {bands}")

# Run this anytime later to check progress:
# check_chirps_task_status()

###SRTM DEM + slope export, both bands cast to Float32.

In [ ]:
# ============================================================
# PHASE 1 — CELL 4
# SRTM DEM + Slope Export — 90 m resolution
# Both bands cast to Float32 to avoid the
# "Exported bands must have compatible data types Int16 and Float32" error
# ============================================================

# --- Session restore check ---
assert 'study_bounds' in dir(), "⚠️ Run Cell 1 first — study_bounds not defined."

# --- Load SRTM DEM ---
srtm = ee.Image("USGS/SRTMGL1_003").select("elevation")

# --- Compute slope in degrees ---
slope = ee.Terrain.slope(srtm)

# --- CRITICAL: cast BOTH bands to Float32 before stacking ---
elevation_f32 = srtm.rename("elevation").toFloat()
slope_f32 = slope.rename("slope").toFloat()

# --- Stack into a single 2-band image ---
dem_stack = ee.Image.cat([elevation_f32, slope_f32])

# Sanity check: confirm both bands report Float32/Float64 type before export
band_types = dem_stack.bandTypes().getInfo()
print("Band data types before export:")
for band, dtype in band_types.items():
    print(f"  {band}: {dtype.get('precision', dtype)}")

# --- Clip to study area ---
dem_stack_clipped = dem_stack.clip(study_bounds)

# --- Export ---
dem_task = ee.batch.Export.image.toDrive(
    image=dem_stack_clipped,
    description="SRTM_DEM_slope",
    folder=DRIVE_FOLDER,
    fileNamePrefix="SRTM_DEM_slope",
    region=study_bounds,
    scale=90,                 # 90 m resolution
    crs=TARGET_CRS,
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)
dem_task.start()
print("🚀 Started export task: SRTM_DEM_slope (elevation + slope, Float32)")

# --- Optional: check task status programmatically ---
def check_dem_task_status():
    status = dem_task.status()
    print(f"SRTM_DEM_slope: {status['state']}")
    if status['state'] == 'FAILED':
        print(f"  Error: {status.get('error_message', 'unknown')}")

# Run this anytime later to check progress:
# check_dem_task_status()

print("\n✅ SRTM DEM + slope export task submitted.")
print("⏳ Monitor progress at: https://code.earthengine.google.com/tasks")

###ESA WorldCover 2021 + JRC Global Surface Water + WorldPop 2020

In [ ]:
# ============================================================
# PHASE 1 — CELL 5
# ESA WorldCover 2021 + JRC Global Surface Water + WorldPop 2020
# Three static single-date layers exported in one cell
# ============================================================

# --- Session restore check ---
assert 'study_bounds' in dir(), "⚠️ Run Cell 1 first — study_bounds not defined."

static_tasks = []

# ------------------------------------------------------------
# 5a. ESA WorldCover 2021 — land cover, 100 m resolution
# ------------------------------------------------------------
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")

# WorldCover classes are natively uint8 — keep as-is (no cast needed)
worldcover_clipped = worldcover.clip(study_bounds)

worldcover_task = ee.batch.Export.image.toDrive(
    image=worldcover_clipped,
    description="ESA_WorldCover_2021",
    folder=DRIVE_FOLDER,
    fileNamePrefix="ESA_WorldCover_2021",
    region=study_bounds,
    scale=100,
    crs=TARGET_CRS,
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)
worldcover_task.start()
static_tasks.append(("ESA_WorldCover_2021", worldcover_task))
print("🚀 Started export task: ESA_WorldCover_2021 (100 m)")

# ------------------------------------------------------------
# 5b. JRC Global Surface Water — permanent water (occurrence >= 50%)
# ------------------------------------------------------------
jrc_gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence")

# Permanent water mask: occurrence >= 50%
permanent_water = jrc_gsw.gte(50).selfMask().rename("permanent_water").toByte()

permanent_water_clipped = permanent_water.clip(study_bounds)

water_task = ee.batch.Export.image.toDrive(
    image=permanent_water_clipped,
    description="JRC_permanent_water",
    folder=DRIVE_FOLDER,
    fileNamePrefix="JRC_permanent_water",
    region=study_bounds,
    scale=30,
    crs=TARGET_CRS,
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)
water_task.start()
static_tasks.append(("JRC_permanent_water", water_task))
print("🚀 Started export task: JRC_permanent_water (30 m, occurrence >= 50%)")

# ------------------------------------------------------------
# 5c. WorldPop 2020 — population density, 100 m resolution
# ------------------------------------------------------------
worldpop = ee.ImageCollection("WorldPop/GP/100m/pop") \
    .filter(ee.Filter.eq("year", 2020)) \
    .filter(ee.Filter.eq("country", "CMR")) \
    .mosaic() \
    .rename("population") \
    .toFloat()

worldpop_clipped = worldpop.clip(study_bounds)

worldpop_task = ee.batch.Export.image.toDrive(
    image=worldpop_clipped,
    description="WorldPop_2020",
    folder=DRIVE_FOLDER,
    fileNamePrefix="WorldPop_2020",
    region=study_bounds,
    scale=100,
    crs=TARGET_CRS,
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)
worldpop_task.start()
static_tasks.append(("WorldPop_2020", worldpop_task))
print("🚀 Started export task: WorldPop_2020 (100 m)")

# ------------------------------------------------------------
# Summary + status checker
# ------------------------------------------------------------
print(f"\n✅ All {len(static_tasks)} static layer export tasks submitted.")
print("⏳ Monitor progress at: https://code.earthengine.google.com/tasks")

def check_static_task_status():
    for name, task in static_tasks:
        status = task.status()
        print(f"{name}: {status['state']}")
        if status['state'] == 'FAILED':
            print(f"   Error: {status.get('error_message', 'unknown')}")

# Run this anytime later to check progress:
# check_static_task_status()

In [ ]:
# ============================================================
# PHASE 1 — PATCH CELL
# Re-export missing NDVI year: 2017
# ============================================================

# --- Redefine session constants (in case of restart) ---
DRIVE_FOLDER = "pasture-mapping-adamawa"
TARGET_CRS = "EPSG:32633"

gaul_level1 = ee.FeatureCollection("FAO/GAUL/2015/level1")
study_area = gaul_level1.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Cameroon"),
        ee.Filter.eq("ADM1_NAME", "Adamaoua")
    )
)
study_geom = study_area.geometry()
study_bounds = study_geom.bounds()

# ------------------------------------------------------------
# STEP 1 — Check if a 2017 task already exists and what happened to it
# ------------------------------------------------------------
tasks = ee.batch.Task.list()
print("Searching task history for NDVI_monthly_2017...\n")

found = False
for t in tasks:
    config = t.config if hasattr(t, "config") else {}
    if "NDVI_monthly_2017" in str(t.status().get("description", "")):
        found = True
        status = t.status()
        print(f"Task ID: {status.get('id')}")
        print(f"State: {status.get('state')}")
        print(f"Description: {status.get('description')}")
        if status.get("state") == "FAILED":
            print(f"Error message: {status.get('error_message')}")
        print("-" * 50)

if not found:
    print("No matching task found in history (may have scrolled out of the list, "
          "or it never actually started). Proceeding to re-submit 2017 export.\n")

# ------------------------------------------------------------
# STEP 2 — Check MODIS NDVI data actually exists for 2017
# (sanity check before re-exporting)
# ------------------------------------------------------------
modis_ndvi = ee.ImageCollection("MODIS/061/MOD13A3").select("NDVI")
n_images_2017 = modis_ndvi.filterDate("2017-01-01", "2018-01-01").size().getInfo()
print(f"MODIS MOD13A3 images available for 2017: {n_images_2017} (expect 12)")

if n_images_2017 < 12:
    print("⚠️ WARNING: fewer than 12 monthly images found for 2017 in MOD13A3. "
          "Investigate before re-exporting.")
else:
    print("✅ Source data confirmed available for 2017.")

# ------------------------------------------------------------
# STEP 3 — Rebuild and re-submit the 2017 export
# (identical logic to Phase 1 Cell 2, single year only)
# ------------------------------------------------------------

def get_year_ndvi_stack(year):
    months = ee.List.sequence(1, 12)

    def month_image(m):
        m = ee.Number(m)
        start = ee.Date.fromYMD(year, m, 1)
        end = start.advance(1, "month")
        monthly = modis_ndvi.filterDate(start, end).mean()
        band_name = ee.String("NDVI_").cat(ee.Number(m).format("%02d"))
        return monthly.rename([band_name]).toFloat()

    monthly_images = ee.ImageCollection.fromImages(months.map(month_image))
    stack = monthly_images.toBands()
    clean_names = [f"NDVI_{m:02d}" for m in range(1, 13)]
    stack = stack.rename(clean_names)
    return stack.set("year", year)

year = 2017
yearly_stack = get_year_ndvi_stack(year)
yearly_stack = yearly_stack.clipToCollection(study_area)

retry_task = ee.batch.Export.image.toDrive(
    image=yearly_stack,
    description=f"NDVI_monthly_{year}_retry",
    folder=DRIVE_FOLDER,
    fileNamePrefix=f"NDVI_monthly_{year}",   # same filename as original attempt
    region=study_bounds,
    scale=1000,
    crs=TARGET_CRS,
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)
retry_task.start()
print(f"\n🚀 Re-submitted export task: NDVI_monthly_{year}_retry")
print("⏳ Monitor at: https://code.earthengine.google.com/tasks")

def check_retry_status():
    status = retry_task.status()
    print(f"NDVI_monthly_{year}_retry: {status['state']}")
    if status['state'] == 'FAILED':
        print(f"  Error: {status.get('error_message', 'unknown')}")

# Run later: check_retry_status()

##Phase 2

###Session Restore + Drive Mount + File Inventory

In [ ]:
# ============================================================
# PHASE 2 — CELL 1
# Session Restore + Google Drive Mount + Input File Inventory
# ============================================================

import os
import glob
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask as rio_mask
from rasterio.enums import Resampling as ResamplingEnum
import geopandas as gpd
from scipy import ndimage

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ============================================================
# SESSION CONSTANTS (redefined here since Colab may have restarted
# since Phase 1 — these do NOT require ee.Initialize() for Phase 2)
# ============================================================

DRIVE_FOLDER = "pasture-mapping-adamawa"
TARGET_CRS = "EPSG:32633"          # UTM Zone 33N
START_YEAR = 2016
END_YEAR = 2025

DRIVE_BASE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
PROCESSED_DIR = "/content/data/processed/final"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Intermediate working directories
os.makedirs("/content/data/interim", exist_ok=True)

print(f"📂 Drive source folder: {DRIVE_BASE}")
print(f"📂 Output folder: {PROCESSED_DIR}")

# ============================================================
# INVENTORY: check what actually landed in Drive from Phase 1
# ============================================================

expected_patterns = {
    "NDVI (MODIS)": "NDVI_monthly_*.tif",
    "CHIRPS rainfall": "CHIRPS_monthly_*.tif",
    "SRTM DEM+slope": "SRTM_DEM_slope.tif",
    "ESA WorldCover": "ESA_WorldCover_2021.tif",
    "JRC permanent water": "JRC_permanent_water.tif",
    "WorldPop": "WorldPop_2020.tif",
}

print("\n" + "=" * 60)
print("FILE INVENTORY CHECK")
print("=" * 60)

inventory = {}
all_ok = True

for label, pattern in expected_patterns.items():
    matches = sorted(glob.glob(os.path.join(DRIVE_BASE, pattern)))
    inventory[label] = matches
    status = "✅" if len(matches) > 0 else "❌"
    print(f"{status} {label}: {len(matches)} file(s) found")
    for m in matches:
        size_mb = os.path.getsize(m) / (1024 * 1024)
        print(f"      - {os.path.basename(m)} ({size_mb:.1f} MB)")
    if len(matches) == 0:
        all_ok = False

print("=" * 60)

# Specific check: NDVI and CHIRPS should have ~10 files each (2016-2025)
n_ndvi = len(inventory["NDVI (MODIS)"])
n_chirps = len(inventory["CHIRPS rainfall"])
expected_years = END_YEAR - START_YEAR + 1

if n_ndvi < expected_years:
    print(f"⚠️ NDVI: expected {expected_years} yearly files, found {n_ndvi}. "
          f"Check GEE tasks — some exports may still be RUNNING or FAILED.")
if n_chirps < expected_years:
    print(f"⚠️ CHIRPS: expected up to {expected_years} yearly files, found {n_chirps}. "
          f"2025 may be partial (expected, per error note 11) — that's OK, "
          f"but missing full years for 2016-2024 is NOT OK.")

if all_ok and n_ndvi == expected_years:
    print("\n✅ All expected files present. Ready to proceed with preprocessing.")
else:
    print("\n⚠️ Some files missing — resolve before continuing, or tell me which "
          "GEE tasks failed and I'll help you re-export just those.")

# --- Quick metadata check on one NDVI file (dtype, CRS, shape) ---
if n_ndvi > 0:
    with rasterio.open(inventory["NDVI (MODIS)"][0]) as src:
        print(f"\n📋 Sample NDVI file metadata ({os.path.basename(inventory['NDVI (MODIS)'][0])}):")
        print(f"   CRS: {src.crs}")
        print(f"   Shape: {src.width} x {src.height}, {src.count} bands")
        print(f"   Dtype: {src.dtypes[0]}")
        print(f"   Resolution: {src.res}")

###Reprojection to EPSG:32633 and resampling all layers to the 1 km NDVI reference grid.

In [ ]:
# ============================================================
# PHASE 2 — CELL 2
# Reproject to EPSG:32633 + Resample to 1 km NDVI Reference Grid
# ============================================================

import os
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

# --- Session restore check ---
assert 'DRIVE_BASE' in dir(), "⚠️ Run Phase 2 Cell 1 first."

REFERENCE_DIR = "/content/data/interim/reference"
REPROJECTED_DIR = "/content/data/interim/reprojected"
os.makedirs(REFERENCE_DIR, exist_ok=True)
os.makedirs(REPROJECTED_DIR, exist_ok=True)

# ============================================================
# STEP 1 — Build the reference grid from one NDVI file
# All other layers will be resampled/reprojected to match this
# exactly (same transform, width, height, CRS)
# ============================================================

ndvi_files = sorted(glob.glob(os.path.join(DRIVE_BASE, "NDVI_monthly_*.tif")))
assert len(ndvi_files) == (END_YEAR - START_YEAR + 1), \
    f"⚠️ Expected {END_YEAR - START_YEAR + 1} NDVI files, found {len(ndvi_files)}. Resolve before proceeding."

reference_file = ndvi_files[0]  # e.g. NDVI_monthly_2016.tif

with rasterio.open(reference_file) as src:
    # NDVI was exported in EPSG:32633 already at export time, but we
    # recompute the transform explicitly here to guarantee an exact,
    # well-defined reference grid (avoids subtle pixel-edge misalignment)
    ref_transform, ref_width, ref_height = calculate_default_transform(
        src.crs, TARGET_CRS, src.width, src.height, *src.bounds
    )
    ref_crs = TARGET_CRS
    ref_meta = src.meta.copy()

print("📐 Reference grid established from:", os.path.basename(reference_file))
print(f"   CRS: {ref_crs}")
print(f"   Width x Height: {ref_width} x {ref_height}")
print(f"   Transform: {ref_transform}")
print(f"   Pixel size: {ref_transform[0]:.2f} m x {abs(ref_transform[4]):.2f} m")

REF_TRANSFORM = ref_transform
REF_WIDTH = ref_width
REF_HEIGHT = ref_height
REF_CRS = ref_crs

# ============================================================
# STEP 2 — Generic reproject+resample function
# ============================================================

def reproject_to_reference(src_path, dst_path, resampling_method=Resampling.bilinear,
                             dst_dtype=None):
    """
    Reprojects and resamples a raster to match the reference grid exactly
    (same CRS, transform, width, height). Preserves band count.
    """
    with rasterio.open(src_path) as src:
        src_dtype = dst_dtype if dst_dtype else src.dtypes[0]

        dst_meta = src.meta.copy()
        dst_meta.update({
            "crs": REF_CRS,
            "transform": REF_TRANSFORM,
            "width": REF_WIDTH,
            "height": REF_HEIGHT,
            "dtype": src_dtype,
            "tiled": True,
            "blockxsize": 256,
            "blockysize": 256,
            "compress": "lzw"
        })

        with rasterio.open(dst_path, "w", **dst_meta) as dst:
            for band_idx in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, band_idx),
                    destination=rasterio.band(dst, band_idx),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=REF_TRANSFORM,
                    dst_crs=REF_CRS,
                    resampling=resampling_method,
                    num_threads=2
                )
    return dst_path


# ============================================================
# STEP 3 — Reproject NDVI files (bilinear — continuous data)
# ============================================================

print("\n" + "=" * 60)
print("REPROJECTING NDVI FILES")
print("=" * 60)

ndvi_reprojected = []
for f in ndvi_files:
    year = os.path.basename(f).replace("NDVI_monthly_", "").replace(".tif", "")
    out_path = os.path.join(REPROJECTED_DIR, f"NDVI_monthly_{year}_reproj.tif")
    reproject_to_reference(f, out_path, resampling_method=Resampling.bilinear, dst_dtype="float32")
    ndvi_reprojected.append(out_path)
    print(f"✅ {os.path.basename(out_path)}")

# ============================================================
# STEP 4 — Reproject CHIRPS files (bilinear — continuous data)
# ============================================================

print("\n" + "=" * 60)
print("REPROJECTING CHIRPS FILES")
print("=" * 60)

chirps_files = sorted(glob.glob(os.path.join(DRIVE_BASE, "CHIRPS_monthly_*.tif")))
chirps_reprojected = []
for f in chirps_files:
    year = os.path.basename(f).replace("CHIRPS_monthly_", "").replace(".tif", "")
    out_path = os.path.join(REPROJECTED_DIR, f"CHIRPS_monthly_{year}_reproj.tif")
    reproject_to_reference(f, out_path, resampling_method=Resampling.bilinear, dst_dtype="float32")
    chirps_reprojected.append(out_path)
    print(f"✅ {os.path.basename(out_path)}")

# ============================================================
# STEP 5 — Reproject DEM + slope (bilinear — continuous elevation/slope)
# ============================================================

print("\n" + "=" * 60)
print("REPROJECTING DEM + SLOPE")
print("=" * 60)

dem_file = os.path.join(DRIVE_BASE, "SRTM_DEM_slope.tif")
dem_reproj_path = os.path.join(REPROJECTED_DIR, "SRTM_DEM_slope_reproj.tif")
reproject_to_reference(dem_file, dem_reproj_path, resampling_method=Resampling.bilinear, dst_dtype="float32")
print(f"✅ {os.path.basename(dem_reproj_path)}")

# ============================================================
# STEP 6 — Reproject ESA WorldCover (NEAREST NEIGHBOR — categorical!)
# ============================================================

print("\n" + "=" * 60)
print("REPROJECTING WORLDCOVER (categorical → nearest neighbor)")
print("=" * 60)

wc_file = os.path.join(DRIVE_BASE, "ESA_WorldCover_2021.tif")
wc_reproj_path = os.path.join(REPROJECTED_DIR, "ESA_WorldCover_2021_reproj.tif")
reproject_to_reference(wc_file, wc_reproj_path, resampling_method=Resampling.nearest, dst_dtype="uint8")
print(f"✅ {os.path.basename(wc_reproj_path)}")

# ============================================================
# STEP 7 — Reproject JRC permanent water (NEAREST NEIGHBOR — binary mask!)
# ============================================================

print("\n" + "=" * 60)
print("REPROJECTING JRC PERMANENT WATER (binary → nearest neighbor)")
print("=" * 60)

water_file = os.path.join(DRIVE_BASE, "JRC_permanent_water.tif")
water_reproj_path = os.path.join(REPROJECTED_DIR, "JRC_permanent_water_reproj.tif")
reproject_to_reference(water_file, water_reproj_path, resampling_method=Resampling.nearest, dst_dtype="uint8")
print(f"✅ {os.path.basename(water_reproj_path)}")

# ============================================================
# STEP 8 — Reproject WorldPop (bilinear — continuous population density)
# ============================================================

print("\n" + "=" * 60)
print("REPROJECTING WORLDPOP")
print("=" * 60)

pop_file = os.path.join(DRIVE_BASE, "WorldPop_2020.tif")
pop_reproj_path = os.path.join(REPROJECTED_DIR, "WorldPop_2020_reproj.tif")
reproject_to_reference(pop_file, pop_reproj_path, resampling_method=Resampling.bilinear, dst_dtype="float32")
print(f"✅ {os.path.basename(pop_reproj_path)}")

# ============================================================
# SUMMARY + verification that everything now shares the same grid
# ============================================================

print("\n" + "=" * 60)
print("GRID ALIGNMENT VERIFICATION")
print("=" * 60)

all_reprojected = ndvi_reprojected + chirps_reprojected + [
    dem_reproj_path, wc_reproj_path, water_reproj_path, pop_reproj_path
]

reference_shape = None
for f in all_reprojected:
    with rasterio.open(f) as src:
        shape = (src.width, src.height)
        crs_match = str(src.crs) == REF_CRS
        if reference_shape is None:
            reference_shape = shape
        shape_match = shape == reference_shape
        flag = "✅" if (crs_match and shape_match) else "❌"
        print(f"{flag} {os.path.basename(f)}: shape={shape}, crs_match={crs_match}")

print(f"\n✅ All {len(all_reprojected)} files reprojected to {REF_CRS} "
      f"at grid size {REF_WIDTH} x {REF_HEIGHT}")
print("Files stored in:", REPROJECTED_DIR)

###Clip to study area boundary, apply dtype-appropriate nodata, apply NDVI scale factor

In [ ]:
boundary_geojson = study_area.geometry().getInfo()
import json
with open('/content/drive/MyDrive/pasture-mapping-adamawa/adamaoua_boundary.geojson', 'w') as f:
  json.dump(boundary_geojson, f)


In [ ]:
# ============================================================
# PHASE 2 — CELL 3
# Clip to Study Area + Dtype-Appropriate Nodata + NDVI Scale Factor
# WITH VISUAL QA AT EACH STEP
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
from rasterio.mask import mask as rio_mask

# --- Session restore check ---
assert 'REPROJECTED_DIR' in dir(), "⚠️ Run Phase 2 Cell 2 first."

CLIPPED_DIR = "/content/data/interim/clipped"
os.makedirs(CLIPPED_DIR, exist_ok=True)

# ============================================================
# STEP 1 — Get study area boundary as a GeoDataFrame (for rasterio.mask)
# We need this as vector geometry in EPSG:32633
# ============================================================

# Since Phase 1's study_area lives in Earth Engine (server-side), and this
# is a fresh preprocessing session, we export the boundary geometry once
# to a local GeoJSON/GPKG for use with rasterio.mask going forward.
#
# If you haven't already saved the boundary locally, run this small
# GEE snippet FIRST (needs ee.Initialize() — separate from this cell):
#
#   boundary_geojson = study_area.geometry().getInfo()
#   import json
#   with open('/content/drive/MyDrive/pasture-mapping-adamawa/adamaoua_boundary.geojson', 'w') as f:
#       json.dump(boundary_geojson, f)
#
# Assuming that file now exists in Drive:

BOUNDARY_PATH = os.path.join(DRIVE_BASE, "adamaoua_boundary.geojson")
assert os.path.exists(BOUNDARY_PATH), \
    "⚠️ Boundary GeoJSON not found. Run the GEE export snippet above first (needs ee.Initialize())."

boundary_gdf = gpd.read_file(BOUNDARY_PATH)
boundary_gdf = boundary_gdf.set_crs("EPSG:4326", allow_override=True).to_crs(REF_CRS)
boundary_geom = [boundary_gdf.geometry.unary_union.__geo_interface__]

print(f"✅ Study area boundary loaded, reprojected to {REF_CRS}")
print(f"   Boundary area: {boundary_gdf.to_crs(REF_CRS).area.sum() / 1e6:,.0f} km²")

# --- Visual: boundary shape check ---
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
boundary_gdf.to_crs(REF_CRS).plot(ax=ax, edgecolor='red', facecolor='none', linewidth=2)
ax.set_title("Adamaoua Boundary (EPSG:32633)")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 2 — Generic clip function with dtype-appropriate nodata
# ============================================================

NODATA_BY_DTYPE = {
    "uint8": 255,
    "int16": -9999,
    "float32": -9999.0,
    "float64": -9999.0,
}

def clip_to_boundary(src_path, dst_path, target_dtype):
    """
    Clips a raster to the study area boundary using rasterio.mask,
    applying the correct nodata value for the target dtype to avoid
    the OverflowError: Python integer -9999 out of bounds for uint8
    """
    nodata_val = NODATA_BY_DTYPE[target_dtype]

    with rasterio.open(src_path) as src:
        out_image, out_transform = rio_mask(
            src, boundary_geom, crop=True, nodata=nodata_val, filled=True
        )
        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "nodata": nodata_val,
            "dtype": target_dtype,
            "tiled": True,
            "blockxsize": 256,
            "blockysize": 256,
            "compress": "lzw"
        })

        out_image = out_image.astype(target_dtype)

        with rasterio.open(dst_path, "w", **out_meta) as dst:
            dst.write(out_image)

    return dst_path, nodata_val

# ============================================================
# STEP 3 — Clip NDVI + apply scale factor (0.0001) HERE, not in GEE export
# ============================================================

print("\n" + "=" * 60)
print("CLIPPING + SCALING NDVI")
print("=" * 60)

ndvi_reproj_files = sorted(glob.glob(os.path.join(REPROJECTED_DIR, "NDVI_monthly_*_reproj.tif")))
ndvi_clipped = []

for f in ndvi_reproj_files:
    year = os.path.basename(f).split("_")[2]
    clip_path = os.path.join(CLIPPED_DIR, f"NDVI_monthly_{year}_clipped_raw.tif")
    clip_to_boundary(f, clip_path, target_dtype="float32")

    # Apply NDVI scale factor now, with proper nodata masking
    with rasterio.open(clip_path) as src:
        data = src.read()
        meta = src.meta.copy()
        nodata = src.nodata

    scaled = np.where(data == nodata, nodata, data * 0.0001)

    final_path = os.path.join(CLIPPED_DIR, f"NDVI_monthly_{year}_final.tif")
    with rasterio.open(final_path, "w", **meta) as dst:
        dst.write(scaled.astype("float32"))

    ndvi_clipped.append(final_path)
    os.remove(clip_path)  # cleanup intermediate
    print(f"✅ {os.path.basename(final_path)}")

# --- Visual: before/after scale factor on one sample month ---
sample_year = "2016"
sample_file = [f for f in ndvi_clipped if sample_year in f][0]
with rasterio.open(sample_file) as src:
    ndvi_sample = src.read(6, masked=True)  # June, band 6

fig, ax = plt.subplots(1, 1, figsize=(7, 6))
im = ax.imshow(ndvi_sample, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
ax.set_title(f"NDVI June {sample_year} — after clip + scale factor (0.0001)\n"
             f"Range: {ndvi_sample.min():.3f} to {ndvi_sample.max():.3f}")
plt.colorbar(im, ax=ax, label="NDVI")
plt.tight_layout()
plt.show()

print(f"NDVI sanity check — should be roughly -0.2 to 0.9, NOT thousands: "
      f"min={ndvi_sample.min():.3f}, max={ndvi_sample.max():.3f}")

# ============================================================
# STEP 4 — Clip CHIRPS (no scale factor needed, already in mm)
# ============================================================

print("\n" + "=" * 60)
print("CLIPPING CHIRPS")
print("=" * 60)

chirps_reproj_files = sorted(glob.glob(os.path.join(REPROJECTED_DIR, "CHIRPS_monthly_*_reproj.tif")))
chirps_clipped = []

for f in chirps_reproj_files:
    year = os.path.basename(f).split("_")[2]
    out_path = os.path.join(CLIPPED_DIR, f"CHIRPS_monthly_{year}_final.tif")
    clip_to_boundary(f, out_path, target_dtype="float32")
    chirps_clipped.append(out_path)
    print(f"✅ {os.path.basename(out_path)}")

# --- Visual: sample CHIRPS month ---
sample_chirps = [f for f in chirps_clipped if "2016" in f][0]
with rasterio.open(sample_chirps) as src:
    rain_sample = src.read(8, masked=True)  # August, band 8 - wet season

fig, ax = plt.subplots(1, 1, figsize=(7, 6))
im = ax.imshow(rain_sample, cmap="Blues")
ax.set_title(f"CHIRPS Rainfall August 2016 (mm/month)\n"
             f"Range: {rain_sample.min():.0f} to {rain_sample.max():.0f} mm")
plt.colorbar(im, ax=ax, label="mm")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 5 — Clip DEM + slope
# ============================================================

print("\n" + "=" * 60)
print("CLIPPING DEM + SLOPE")
print("=" * 60)

dem_clipped_path = os.path.join(CLIPPED_DIR, "SRTM_DEM_slope_final.tif")
clip_to_boundary(dem_reproj_path, dem_clipped_path, target_dtype="float32")
print(f"✅ {os.path.basename(dem_clipped_path)}")

# --- Visual: elevation + slope side by side ---
with rasterio.open(dem_clipped_path) as src:
    elevation = src.read(1, masked=True)
    slope = src.read(2, masked=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
im0 = axes[0].imshow(elevation, cmap="terrain")
axes[0].set_title(f"Elevation (m)\nRange: {elevation.min():.0f}–{elevation.max():.0f} m")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(slope, cmap="YlOrRd", vmin=0, vmax=30)
axes[1].set_title(f"Slope (degrees)\nRange: {slope.min():.1f}–{slope.max():.1f}°")
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

# ============================================================
# STEP 6 — Clip WorldCover (categorical)
# ============================================================

print("\n" + "=" * 60)
print("CLIPPING WORLDCOVER")
print("=" * 60)

wc_clipped_path = os.path.join(CLIPPED_DIR, "ESA_WorldCover_2021_final.tif")
clip_to_boundary(wc_reproj_path, wc_clipped_path, target_dtype="uint8")
print(f"✅ {os.path.basename(wc_clipped_path)}")

# --- Visual: land cover with proper class colors ---
WORLDCOVER_COLORS = {
    10: "#006400", 20: "#ffbb22", 30: "#ffff4c", 40: "#f096ff",
    50: "#fa0000", 60: "#b4b4b4", 70: "#f0f0f0", 80: "#0064c8",
    90: "#0096a0", 95: "#00cf75", 100: "#fae6a0"
}
WORLDCOVER_LABELS = {
    10: "Trees", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare/sparse", 70: "Snow/ice", 80: "Water",
    90: "Wetland", 95: "Mangroves", 100: "Moss/lichen"
}

with rasterio.open(wc_clipped_path) as src:
    wc_data = src.read(1, masked=True)

unique_vals = sorted([v for v in np.unique(wc_data.compressed()) if v in WORLDCOVER_COLORS])
cmap_colors = [WORLDCOVER_COLORS[v] for v in unique_vals]
cmap = mcolors.ListedColormap(cmap_colors)
bounds = unique_vals + [unique_vals[-1] + 10]
norm = mcolors.BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(1, 1, figsize=(8, 7))
im = ax.imshow(wc_data, cmap=cmap, norm=norm)
ax.set_title("ESA WorldCover 2021 — Adamaoua")
cbar = plt.colorbar(im, ax=ax, ticks=[v + 5 for v in unique_vals])
cbar.ax.set_yticklabels([WORLDCOVER_LABELS[v] for v in unique_vals])
plt.tight_layout()
plt.show()

print("Land cover class breakdown:")
for v in unique_vals:
    pct = (wc_data == v).sum() / wc_data.compressed().size * 100
    print(f"   {WORLDCOVER_LABELS[v]} (class {v}): {pct:.1f}%")

# ============================================================
# STEP 7 — Clip JRC water + WorldPop
# ============================================================

print("\n" + "=" * 60)
print("CLIPPING JRC WATER + WORLDPOP")
print("=" * 60)

water_clipped_path = os.path.join(CLIPPED_DIR, "JRC_permanent_water_final.tif")
clip_to_boundary(water_reproj_path, water_clipped_path, target_dtype="uint8")
print(f"✅ {os.path.basename(water_clipped_path)}")

pop_clipped_path = os.path.join(CLIPPED_DIR, "WorldPop_2020_final.tif")
clip_to_boundary(pop_reproj_path, pop_clipped_path, target_dtype="float32")
print(f"✅ {os.path.basename(pop_clipped_path)}")

# --- Visual: water + population side by side ---
with rasterio.open(water_clipped_path) as src:
    water_data = src.read(1, masked=True)
with rasterio.open(pop_clipped_path) as src:
    pop_data = src.read(1, masked=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(water_data, cmap="Blues")
axes[0].set_title(f"Permanent Water Bodies\n{int((water_data == 1).sum())} pixels ("
                   f"{(water_data == 1).sum() * (REF_TRANSFORM[0]**2) / 1e6:.1f} km²)")

im1 = axes[1].imshow(pop_data, cmap="magma", vmax=np.percentile(pop_data.compressed(), 98))
axes[1].set_title(f"Population Density 2020\nMax: {pop_data.max():.0f} people/px, "
                   f"98th pctile: {np.percentile(pop_data.compressed(), 98):.0f}")
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("✅ PHASE 2 CELL 3 COMPLETE — all layers clipped to study area,")
print("   dtype-appropriate nodata applied, NDVI scale factor applied.")
print("=" * 60)

###Distance-to-Water Raster + Extract Slope as Standalone File

In [ ]:
# ============================================================
# PHASE 2 — CELL 4
# Distance-to-Water Raster (scipy distance_transform_edt)
# + Extract Slope as Separate Single-Band File
# WITH VISUAL QA
# ============================================================

import matplotlib.pyplot as plt
from scipy import ndimage

# --- Session restore check ---
assert 'CLIPPED_DIR' in dir(), "⚠️ Run Phase 2 Cell 3 first."

# ============================================================
# STEP 1 — Extract slope as a separate single-band file
# from the 2-band DEM stack (band 1 = elevation, band 2 = slope)
# ============================================================

dem_clipped_path = os.path.join(CLIPPED_DIR, "SRTM_DEM_slope_final.tif")

with rasterio.open(dem_clipped_path) as src:
    elevation_data = src.read(1)
    slope_data = src.read(2)
    dem_meta = src.meta.copy()
    dem_transform = src.transform
    dem_nodata = src.nodata

# Save elevation as standalone file
elevation_path = os.path.join(CLIPPED_DIR, "elevation_final.tif")
elev_meta = dem_meta.copy()
elev_meta.update({"count": 1})
with rasterio.open(elevation_path, "w", **elev_meta) as dst:
    dst.write(elevation_data, 1)
print(f"✅ {os.path.basename(elevation_path)} (extracted from DEM stack)")

# Save slope as standalone file
slope_path = os.path.join(CLIPPED_DIR, "slope_final.tif")
slope_meta = dem_meta.copy()
slope_meta.update({"count": 1})
with rasterio.open(slope_path, "w", **slope_meta) as dst:
    dst.write(slope_data, 1)
print(f"✅ {os.path.basename(slope_path)} (extracted from DEM stack)")

# ============================================================
# STEP 2 — Compute distance-to-permanent-water raster
# using scipy.ndimage.distance_transform_edt
# ============================================================

water_clipped_path = os.path.join(CLIPPED_DIR, "JRC_permanent_water_final.tif")

with rasterio.open(water_clipped_path) as src:
    water_data = src.read(1)
    water_nodata = src.nodata
    water_meta = src.meta.copy()
    pixel_size_x = src.transform[0]
    pixel_size_y = abs(src.transform[4])

# Build binary mask: 1 = water, 0 = not water (nodata treated as "not water")
water_binary = np.where((water_data == 1), 1, 0).astype(bool)

n_water_pixels = water_binary.sum()
print(f"\n💧 Permanent water pixels found: {n_water_pixels} "
      f"({n_water_pixels * pixel_size_x * pixel_size_y / 1e6:.1f} km²)")

if n_water_pixels == 0:
    print("⚠️ WARNING: zero water pixels found. Distance raster will be meaningless "
          "(all distances will default to a large constant). Check JRC water clip.")

# distance_transform_edt computes distance FROM each background (0) pixel
# TO the nearest True (1) pixel — so we invert: we want distance to water,
# and water pixels themselves should have distance 0
# sampling= sets the pixel size in each dimension for correct distances in meters
distance_pixels = ndimage.distance_transform_edt(
    ~water_binary,  # True where NOT water — distance computed FROM here TO nearest water
    sampling=(pixel_size_y, pixel_size_x)  # (row spacing, col spacing) in meters
)

# distance_pixels is already in meters because of `sampling=`
distance_to_water_m = distance_pixels.astype("float32")

# Water pixels themselves get distance 0 (already true since ~water_binary is False there)

print(f"Distance to water range: {distance_to_water_m.min():.0f} m to {distance_to_water_m.max():.0f} m")
print(f"Mean distance to water: {distance_to_water_m.mean():.0f} m "
      f"({distance_to_water_m.mean()/1000:.2f} km)")

# Save distance-to-water raster
dist_water_path = os.path.join(CLIPPED_DIR, "distance_to_water_final.tif")
dist_meta = water_meta.copy()
dist_meta.update({
    "dtype": "float32",
    "nodata": -9999.0,
    "count": 1,
    "tiled": True,
    "blockxsize": 256,
    "blockysize": 256,
    "compress": "lzw"
})
with rasterio.open(dist_water_path, "w", **dist_meta) as dst:
    dst.write(distance_to_water_m, 1)
print(f"✅ {os.path.basename(dist_water_path)}")

# ============================================================
# VISUAL QA — slope, elevation, water mask, and distance-to-water
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(13, 12))

# Elevation
im0 = axes[0, 0].imshow(elevation_data, cmap="terrain")
axes[0, 0].set_title(f"Elevation (extracted)\n{elevation_data.min():.0f}–{elevation_data.max():.0f} m")
plt.colorbar(im0, ax=axes[0, 0], fraction=0.046)

# Slope
im1 = axes[0, 1].imshow(slope_data, cmap="YlOrRd", vmin=0, vmax=30)
axes[0, 1].set_title(f"Slope (extracted)\n{slope_data.min():.1f}–{slope_data.max():.1f}°")
plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)

# Water binary mask
axes[1, 0].imshow(water_binary, cmap="Blues")
axes[1, 0].set_title(f"Permanent Water Mask\n{n_water_pixels} pixels")

# Distance to water
im3 = axes[1, 1].imshow(distance_to_water_m / 1000, cmap="viridis_r",
                          vmax=np.percentile(distance_to_water_m / 1000, 98))
axes[1, 1].set_title(f"Distance to Water (km)\nMean: {distance_to_water_m.mean()/1000:.1f} km, "
                      f"Max: {distance_to_water_m.max()/1000:.1f} km")
plt.colorbar(im3, ax=axes[1, 1], fraction=0.046, label="km")

plt.tight_layout()
plt.show()

# ============================================================
# Sanity check specific to Phase 4's water proximity scoring
# (threshold = 20 km, per your spec — invert so closer = better)
# ============================================================

pct_within_20km = (distance_to_water_m <= 20000).sum() / distance_to_water_m.size * 100
print(f"\n📊 Phase 4 preview check: {pct_within_20km:.1f}% of pixels are within "
      f"the 20 km water-proximity threshold used for suitability scoring.")

if pct_within_20km < 30:
    print("⚠️ Less than 30% of the region is within 20 km of permanent water. "
          "This will heavily penalize most of the study area in Phase 4's "
          "weighted combination — worth knowing now, not after zone delineation.")
elif pct_within_20km > 95:
    print("ℹ️ Nearly the entire region is within 20 km of water — the water-proximity "
          "factor may contribute little discriminating power to the final suitability score.")
else:
    print("✅ Reasonable spread — water proximity should meaningfully differentiate zones.")

print("\n" + "=" * 60)
print("✅ PHASE 2 CELL 4 COMPLETE")
print(f"   Elevation: {os.path.basename(elevation_path)}")
print(f"   Slope: {os.path.basename(slope_path)}")
print(f"   Distance to water: {os.path.basename(dist_water_path)}")
print("=" * 60)

###Final Assembly + Manifest for Phase 3

In [ ]:
# ============================================================
# PHASE 2 — CELL 5
# Final Assembly: copy/organize all layers into data/processed/final/
# + write a manifest JSON that Phase 3 will load
# WITH FULL VISUAL SUMMARY (all layers side by side)
# ============================================================

import shutil
import json
from datetime import datetime

# --- Session restore check ---
assert 'CLIPPED_DIR' in dir(), "⚠️ Run Phase 2 Cells 1-4 first."

# ============================================================
# STEP 1 — Organize final outputs into data/processed/final/
# ============================================================

FINAL_DIR = PROCESSED_DIR  # = /content/data/processed/final (from Cell 1)
os.makedirs(FINAL_DIR, exist_ok=True)
os.makedirs(os.path.join(FINAL_DIR, "ndvi"), exist_ok=True)
os.makedirs(os.path.join(FINAL_DIR, "chirps"), exist_ok=True)
os.makedirs(os.path.join(FINAL_DIR, "static"), exist_ok=True)

# --- Move/copy NDVI yearly files ---
ndvi_final_files = sorted(glob.glob(os.path.join(CLIPPED_DIR, "NDVI_monthly_*_final.tif")))
for f in ndvi_final_files:
    year = os.path.basename(f).split("_")[2]
    dst = os.path.join(FINAL_DIR, "ndvi", f"NDVI_{year}.tif")
    shutil.copy2(f, dst)
print(f"✅ Copied {len(ndvi_final_files)} NDVI yearly files → {FINAL_DIR}/ndvi/")

# --- Move/copy CHIRPS yearly files ---
chirps_final_files = sorted(glob.glob(os.path.join(CLIPPED_DIR, "CHIRPS_monthly_*_final.tif")))
for f in chirps_final_files:
    year = os.path.basename(f).split("_")[2]
    dst = os.path.join(FINAL_DIR, "chirps", f"CHIRPS_{year}.tif")
    shutil.copy2(f, dst)
print(f"✅ Copied {len(chirps_final_files)} CHIRPS yearly files → {FINAL_DIR}/chirps/")

# --- Move/copy static layers ---
static_files = {
    "elevation_final.tif": "elevation.tif",
    "slope_final.tif": "slope.tif",
    "distance_to_water_final.tif": "distance_to_water.tif",
    "ESA_WorldCover_2021_final.tif": "landcover.tif",
    "JRC_permanent_water_final.tif": "water_mask.tif",
    "WorldPop_2020_final.tif": "population.tif",
}

for src_name, dst_name in static_files.items():
    src_path = os.path.join(CLIPPED_DIR, src_name)
    dst_path = os.path.join(FINAL_DIR, "static", dst_name)
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        print(f"✅ Copied {dst_name} → {FINAL_DIR}/static/")
    else:
        print(f"❌ MISSING: {src_name} — cannot copy. Check Cells 3-4 completed successfully.")

# ============================================================
# STEP 2 — Build manifest.json (Phase 3 loads this to know what exists)
# ============================================================

with rasterio.open(os.path.join(FINAL_DIR, "static", "elevation.tif")) as src:
    grid_width = src.width
    grid_height = src.height
    grid_transform = list(src.transform)
    grid_crs = str(src.crs)
    grid_bounds = list(src.bounds)

manifest = {
    "created": datetime.now().isoformat(),
    "study_area": "Adamaoua, Cameroon",
    "crs": grid_crs,
    "grid_width": grid_width,
    "grid_height": grid_height,
    "grid_transform": grid_transform,
    "grid_bounds": grid_bounds,
    "pixel_size_m": 1000,
    "years": list(range(START_YEAR, END_YEAR + 1)),
    "ndvi_years_available": sorted([
        os.path.basename(f).replace("NDVI_", "").replace(".tif", "")
        for f in glob.glob(os.path.join(FINAL_DIR, "ndvi", "*.tif"))
    ]),
    "chirps_years_available": sorted([
        os.path.basename(f).replace("CHIRPS_", "").replace(".tif", "")
        for f in glob.glob(os.path.join(FINAL_DIR, "chirps", "*.tif"))
    ]),
    "static_layers": {
        "elevation": "static/elevation.tif",
        "slope": "static/slope.tif",
        "distance_to_water": "static/distance_to_water.tif",
        "landcover": "static/landcover.tif",
        "water_mask": "static/water_mask.tif",
        "population": "static/population.tif",
    },
    "nodata_by_dtype": {
        "uint8": 255,
        "int16": -9999,
        "float32": -9999.0
    },
    "notes": {
        "chirps_2025": "2025 may have fewer than 12 months depending on export date; "
                        "missing months should be padded with 2016-2024 climatology mean in Phase 3",
        "ndvi_scale_factor": "Already applied (x0.0001) — NDVI files are analysis-ready, no further scaling needed"
    }
}

manifest_path = os.path.join(FINAL_DIR, "manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\n✅ Manifest written: {manifest_path}")

# ============================================================
# STEP 3 — Full directory listing + integrity check
# ============================================================

print("\n" + "=" * 60)
print("FINAL DIRECTORY STRUCTURE")
print("=" * 60)

for root, dirs, files in os.walk(FINAL_DIR):
    level = root.replace(FINAL_DIR, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in sorted(files):
        fpath = os.path.join(root, file)
        size_kb = os.path.getsize(fpath) / 1024
        print(f"{subindent}{file} ({size_kb:.0f} KB)")

n_ndvi_final = len(manifest["ndvi_years_available"])
n_chirps_final = len(manifest["chirps_years_available"])
n_static_final = sum(1 for f in static_files.values()
                      if os.path.exists(os.path.join(FINAL_DIR, "static", f)))

print("\n" + "=" * 60)
print("INTEGRITY SUMMARY")
print("=" * 60)
print(f"NDVI years: {n_ndvi_final}/{END_YEAR - START_YEAR + 1} — {manifest['ndvi_years_available']}")
print(f"CHIRPS years: {n_chirps_final}/{END_YEAR - START_YEAR + 1} — {manifest['chirps_years_available']}")
print(f"Static layers: {n_static_final}/6")

all_good = (n_ndvi_final == END_YEAR - START_YEAR + 1) and (n_static_final == 6)
if all_good:
    print("\n✅ Phase 2 preprocessing complete. All data ready for Phase 3.")
else:
    print("\n⚠️ Some data missing — resolve before proceeding to Phase 3.")

# ============================================================
# VISUAL SUMMARY — every static layer + one NDVI/CHIRPS sample, in one grid
# ============================================================

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

def show_layer(ax, path, title, cmap, band=1, vmin=None, vmax=None):
    with rasterio.open(path) as src:
        data = src.read(band, masked=True)
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)

show_layer(axes[0, 0], os.path.join(FINAL_DIR, "static", "elevation.tif"),
           "Elevation (m)", "terrain")
show_layer(axes[0, 1], os.path.join(FINAL_DIR, "static", "slope.tif"),
           "Slope (°)", "YlOrRd", vmin=0, vmax=30)
show_layer(axes[0, 2], os.path.join(FINAL_DIR, "static", "distance_to_water.tif"),
           "Distance to Water (m)", "viridis_r")
show_layer(axes[0, 3], os.path.join(FINAL_DIR, "static", "population.tif"),
           "Population Density", "magma")

show_layer(axes[1, 0], os.path.join(FINAL_DIR, "static", "landcover.tif"),
           "Land Cover (classes)", "tab20")
show_layer(axes[1, 1], os.path.join(FINAL_DIR, "static", "water_mask.tif"),
           "Water Mask", "Blues")
show_layer(axes[1, 2], os.path.join(FINAL_DIR, "ndvi", f"NDVI_{START_YEAR}.tif"),
           f"NDVI June {START_YEAR}", "RdYlGn", band=6, vmin=-0.2, vmax=0.9)
show_layer(axes[1, 3], os.path.join(FINAL_DIR, "chirps", f"CHIRPS_{START_YEAR}.tif"),
           f"Rainfall August {START_YEAR}", "Blues", band=8)

plt.suptitle("Phase 2 Final Output — All Analysis-Ready Layers", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\n📦 All layers are now on the SAME grid, SAME CRS, ready for Phase 3.")

##Phase 3

###Session Restore + Build 4D NDVI Array

In [ ]:
# ============================================================
# PHASE 3 — CELL 1
# Session Restore + Build 4D NDVI Array (years, months, height, width)
# Covering 2016–2025
# ============================================================

import os
import json
import glob
import numpy as np
import rasterio
import matplotlib.pyplot as plt

# --- Session restore: redefine constants + paths (Colab may have restarted) ---
DRIVE_FOLDER = "pasture-mapping-adamawa"
TARGET_CRS = "EPSG:32633"
START_YEAR = 2016
END_YEAR = 2025

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROCESSED_DIR = "/content/data/processed/final"
DRIVE_BASE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"

# --- Load manifest from Phase 2 ---
manifest_path = os.path.join(PROCESSED_DIR, "manifest.json")
assert os.path.exists(manifest_path), \
    "⚠️ manifest.json not found. Re-run Phase 2 Cell 5, or check PROCESSED_DIR path."

with open(manifest_path, "r") as f:
    manifest = json.load(f)

GRID_WIDTH = manifest["grid_width"]
GRID_HEIGHT = manifest["grid_height"]
GRID_TRANSFORM = manifest["grid_transform"]
GRID_CRS = manifest["crs"]
NDVI_YEARS = manifest["ndvi_years_available"]

print("📋 Manifest loaded:")
print(f"   Grid: {GRID_WIDTH} x {GRID_HEIGHT}")
print(f"   CRS: {GRID_CRS}")
print(f"   NDVI years available: {NDVI_YEARS}")

assert len(NDVI_YEARS) == (END_YEAR - START_YEAR + 1), \
    f"⚠️ Expected {END_YEAR - START_YEAR + 1} NDVI years, manifest shows {len(NDVI_YEARS)}. Resolve before proceeding."

# ============================================================
# STEP 1 — Build the 4D NDVI array: (n_years, 12, height, width)
# ============================================================

n_years = len(NDVI_YEARS)
ndvi_4d = np.full((n_years, 12, GRID_HEIGHT, GRID_WIDTH), np.nan, dtype="float32")

NDVI_NODATA = -9999.0

for y_idx, year in enumerate(NDVI_YEARS):
    ndvi_path = os.path.join(PROCESSED_DIR, "ndvi", f"NDVI_{year}.tif")
    with rasterio.open(ndvi_path) as src:
        data = src.read()  # shape (12, height, width)
        nodata_val = src.nodata if src.nodata is not None else NDVI_NODATA

        # Mask nodata as NaN so climatology math (mean/std) ignores it properly
        data_masked = np.where(data == nodata_val, np.nan, data)
        ndvi_4d[y_idx] = data_masked

    print(f"✅ Loaded NDVI {year} → array slice [{y_idx}], "
          f"valid range: {np.nanmin(data_masked):.3f} to {np.nanmax(data_masked):.3f}")

print(f"\n📦 4D NDVI array built: shape = {ndvi_4d.shape} "
      f"(years={n_years}, months=12, height={GRID_HEIGHT}, width={GRID_WIDTH})")
print(f"   Memory footprint: {ndvi_4d.nbytes / 1e9:.2f} GB")

# --- Sanity check: NaN fraction per year (flags months with all-nodata, e.g. sensor gaps) ---
print("\n📊 NaN fraction per year (should mostly reflect boundary/nodata, not full-month gaps):")
for y_idx, year in enumerate(NDVI_YEARS):
    nan_frac = np.isnan(ndvi_4d[y_idx]).mean() * 100
    print(f"   {year}: {nan_frac:.1f}% NaN")

# ============================================================
# VISUAL QA — plot the same month (June) across every year
# to check temporal consistency and catch obviously broken years
# ============================================================

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes_flat = axes.flatten()

for y_idx, year in enumerate(NDVI_YEARS):
    june_ndvi = ndvi_4d[y_idx, 5]  # month index 5 = June (0-indexed)
    im = axes_flat[y_idx].imshow(june_ndvi, cmap="RdYlGn", vmin=-0.1, vmax=0.8)
    axes_flat[y_idx].set_title(f"June {year}", fontsize=11)
    axes_flat[y_idx].axis("off")

plt.suptitle("NDVI — June, All Years (2016–2025) — Visual Consistency Check", fontsize=14, y=1.02)
fig.colorbar(im, ax=axes_flat, fraction=0.02, pad=0.02, label="NDVI")
plt.tight_layout()
plt.show()

# ============================================================
# VISUAL QA — seasonal cycle for a single pixel (center of study area)
# confirms wet/dry season pattern looks physically realistic
# ============================================================

center_row, center_col = GRID_HEIGHT // 2, GRID_WIDTH // 2
pixel_series = ndvi_4d[:, :, center_row, center_col]  # shape (n_years, 12)

fig, ax = plt.subplots(1, 1, figsize=(11, 5))
for y_idx, year in enumerate(NDVI_YEARS):
    ax.plot(range(1, 13), pixel_series[y_idx], marker="o", alpha=0.6, label=year)
ax.set_xlabel("Month")
ax.set_ylabel("NDVI")
ax.set_title(f"NDVI Seasonal Cycle — Center Pixel (row={center_row}, col={center_col})\n"
             "Expect low NDVI in dry season (Dec–Feb), peak in wet season (Aug–Oct)")
ax.set_xticks(range(1, 13))
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ Phase 3 Cell 1 complete — 4D NDVI array ready for climatology computation.")

###NDVI Climatology + Anomaly Detection (Drought Years)

In [ ]:
# ============================================================
# PHASE 3 — CELL 2
# Long-term Monthly NDVI Climatology (mean, max, min, std)
# + NDVI Anomaly per Year (drought detection)
# WITH VISUAL QA
# ============================================================

# --- Session restore check ---
assert 'ndvi_4d' in dir(), "⚠️ Run Phase 3 Cell 1 first — ndvi_4d not defined."

CLIMATOLOGY_DIR = os.path.join(PROCESSED_DIR, "climatology")
os.makedirs(CLIMATOLOGY_DIR, exist_ok=True)

# ============================================================
# STEP 1 — Compute climatology across the years axis (axis=0)
# Result shape for each stat: (12, height, width)
# ============================================================

print("Computing NDVI climatology (2016–2025)...")

ndvi_clim_mean = np.nanmean(ndvi_4d, axis=0)   # (12, H, W)
ndvi_clim_max  = np.nanmax(ndvi_4d, axis=0)
ndvi_clim_min  = np.nanmin(ndvi_4d, axis=0)
ndvi_clim_std  = np.nanstd(ndvi_4d, axis=0)

print(f"✅ Climatology computed — shape per stat: {ndvi_clim_mean.shape}")
print(f"   Mean NDVI overall range: {np.nanmin(ndvi_clim_mean):.3f} to {np.nanmax(ndvi_clim_mean):.3f}")
print(f"   Std NDVI overall range: {np.nanmin(ndvi_clim_std):.3f} to {np.nanmax(ndvi_clim_std):.3f}")

# --- Save climatology stats as multi-band GeoTIFFs (one file per stat, 12 bands each) ---
ref_meta = {
    "driver": "GTiff",
    "height": GRID_HEIGHT,
    "width": GRID_WIDTH,
    "count": 12,
    "dtype": "float32",
    "crs": GRID_CRS,
    "transform": rasterio.Affine(*GRID_TRANSFORM[:6]) if len(GRID_TRANSFORM) >= 6 else rasterio.Affine(*GRID_TRANSFORM),
    "nodata": -9999.0,
    "tiled": True,
    "blockxsize": 256,
    "blockysize": 256,
    "compress": "lzw"
}

def save_climatology_stat(array_3d, name):
    path = os.path.join(CLIMATOLOGY_DIR, f"NDVI_climatology_{name}_2016-2025.tif")
    with rasterio.open(path, "w", **ref_meta) as dst:
        for m in range(12):
            band = np.where(np.isnan(array_3d[m]), -9999.0, array_3d[m]).astype("float32")
            dst.write(band, m + 1)
    print(f"✅ Saved: {os.path.basename(path)}")
    return path

clim_mean_path = save_climatology_stat(ndvi_clim_mean, "mean")
clim_max_path  = save_climatology_stat(ndvi_clim_max, "max")
clim_min_path  = save_climatology_stat(ndvi_clim_min, "min")
clim_std_path  = save_climatology_stat(ndvi_clim_std, "std")

# ============================================================
# VISUAL QA — climatology mean for all 12 months (seasonal cycle map)
# ============================================================

month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, axes = plt.subplots(3, 4, figsize=(20, 13))
axes_flat = axes.flatten()

for m in range(12):
    im = axes_flat[m].imshow(ndvi_clim_mean[m], cmap="RdYlGn", vmin=-0.1, vmax=0.8)
    axes_flat[m].set_title(f"{month_names[m]} — Mean NDVI (2016–2025)", fontsize=11)
    axes_flat[m].axis("off")

plt.suptitle("NDVI Climatology — Long-term Monthly Mean", fontsize=15, y=1.01)
fig.colorbar(im, ax=axes_flat, fraction=0.02, pad=0.02, label="NDVI")
plt.tight_layout()
plt.show()

# --- Std map for wet season peak month (August, index 7) as a "variability hotspot" check ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
im0 = axes[0].imshow(ndvi_clim_mean[7], cmap="RdYlGn", vmin=-0.1, vmax=0.8)
axes[0].set_title("August Mean NDVI (peak wet season)")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(ndvi_clim_std[7], cmap="plasma")
axes[1].set_title("August NDVI Std Dev (interannual variability)")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

# ============================================================
# STEP 2 — Compute NDVI anomaly per year (year − climatology mean)
# Positive = greener than average, Negative = drier/browner than average
# ============================================================

print("\nComputing NDVI anomalies per year...")

ndvi_anomaly = ndvi_4d - ndvi_clim_mean[np.newaxis, :, :, :]  # broadcast over years axis

ANOMALY_DIR = os.path.join(PROCESSED_DIR, "anomaly")
os.makedirs(ANOMALY_DIR, exist_ok=True)

anomaly_meta = ref_meta.copy()

for y_idx, year in enumerate(NDVI_YEARS):
    path = os.path.join(ANOMALY_DIR, f"NDVI_anomaly_{year}.tif")
    with rasterio.open(path, "w", **anomaly_meta) as dst:
        for m in range(12):
            band = np.where(np.isnan(ndvi_anomaly[y_idx, m]), -9999.0, ndvi_anomaly[y_idx, m]).astype("float32")
            dst.write(band, m + 1)
    print(f"✅ Saved: {os.path.basename(path)}")

# ============================================================
# STEP 3 — Drought year detection: mean anomaly per year (spatial average)
# ============================================================

annual_mean_anomaly = np.nanmean(ndvi_anomaly, axis=(1, 2, 3))  # one value per year

print("\n📊 Annual mean NDVI anomaly (spatially averaged, across all months):")
drought_summary = []
for y_idx, year in enumerate(NDVI_YEARS):
    val = annual_mean_anomaly[y_idx]
    flag = "🔴 DROUGHT" if val < -0.03 else ("🟡 Below normal" if val < -0.01 else "🟢 Normal/wet")
    print(f"   {year}: {val:+.4f}  {flag}")
    drought_summary.append({"year": year, "mean_anomaly": float(val), "flag": flag})

# --- Visual: bar chart of annual anomaly, colored by drought severity ---
fig, ax = plt.subplots(1, 1, figsize=(11, 5))
colors = ["#d62728" if v < -0.03 else ("#ff9896" if v < -0.01 else "#2ca02c")
          for v in annual_mean_anomaly]
ax.bar(NDVI_YEARS, annual_mean_anomaly, color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline(-0.03, color="red", linestyle="--", linewidth=0.8, alpha=0.5, label="Drought threshold (-0.03)")
ax.set_xlabel("Year")
ax.set_ylabel("Mean NDVI Anomaly")
ax.set_title("Annual NDVI Anomaly — Drought Year Detection (2016–2025)")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- Spatial anomaly map for the most negative (driest) year found ---
worst_year_idx = int(np.argmin(annual_mean_anomaly))
worst_year = NDVI_YEARS[worst_year_idx]

fig, ax = plt.subplots(1, 1, figsize=(8, 7))
worst_year_spatial_anomaly = np.nanmean(ndvi_anomaly[worst_year_idx], axis=0)  # avg across months
im = ax.imshow(worst_year_spatial_anomaly, cmap="RdBu", vmin=-0.15, vmax=0.15)
ax.set_title(f"Spatial NDVI Anomaly Map — Driest Year Detected: {worst_year}\n"
             f"(Red = drier than normal, Blue = greener than normal)")
plt.colorbar(im, ax=ax, label="NDVI Anomaly")
plt.tight_layout()
plt.show()

print(f"\n🔴 Driest year detected: {worst_year} (mean anomaly = {annual_mean_anomaly[worst_year_idx]:+.4f})")

# --- Save drought summary as JSON for later reference (ML features, reporting) ---
drought_summary_path = os.path.join(CLIMATOLOGY_DIR, "drought_year_summary.json")
with open(drought_summary_path, "w") as f:
    json.dump(drought_summary, f, indent=2)
print(f"✅ Saved: {os.path.basename(drought_summary_path)}")

print("\n" + "=" * 60)
print("✅ PHASE 3 CELL 2 COMPLETE")
print(f"   Climatology files: {CLIMATOLOGY_DIR}/")
print(f"   Anomaly files: {ANOMALY_DIR}/")
print("=" * 60)

###Biomass Estimation + Suitability Classification + Rainfall-NDVI Lag Correlation

In [ ]:
# ============================================================
# PHASE 3 — CELL 3
# Above-Ground Biomass Estimation (Sahelian regression)
# + Grazing Suitability Classification
# + Rainfall-NDVI Lag Correlation (0-3 months)
# + Dry/Wet Season Biomass Export
# WITH VISUAL QA
# ============================================================

# --- Session restore check ---
assert 'ndvi_4d' in dir(), "⚠️ Run Phase 3 Cells 1-2 first."
assert 'ndvi_clim_mean' in dir(), "⚠️ Run Phase 3 Cell 2 first."

BIOMASS_DIR = os.path.join(PROCESSED_DIR, "biomass")
os.makedirs(BIOMASS_DIR, exist_ok=True)

# ============================================================
# STEP 1 — Above-Ground Biomass: AGB = 3500 x NDVI - 250, clipped to min 0
# Applied to the FULL 4D array (years, months, H, W)
# ============================================================

print("Computing above-ground biomass (Sahelian regression formula)...")

biomass_4d = (3500 * ndvi_4d - 250)
biomass_4d = np.where(np.isnan(ndvi_4d), np.nan, np.clip(biomass_4d, 0, None))

print(f"✅ Biomass array computed — shape: {biomass_4d.shape}")
print(f"   Range: {np.nanmin(biomass_4d):.1f} to {np.nanmax(biomass_4d):.1f} kg DM/ha")
print(f"   Mean: {np.nanmean(biomass_4d):.1f} kg DM/ha")

# --- Biomass monthly mean across all years (climatology-style) ---
biomass_monthly_mean = np.nanmean(biomass_4d, axis=0)  # (12, H, W)

biomass_meta = ref_meta.copy()
biomass_mean_path = os.path.join(BIOMASS_DIR, "biomass_monthly_mean_2016-2025.tif")
with rasterio.open(biomass_mean_path, "w", **biomass_meta) as dst:
    for m in range(12):
        band = np.where(np.isnan(biomass_monthly_mean[m]), -9999.0, biomass_monthly_mean[m]).astype("float32")
        dst.write(band, m + 1)
print(f"✅ Saved: {os.path.basename(biomass_mean_path)}")

# --- VISUAL: biomass monthly mean, 12-panel ---
fig, axes = plt.subplots(3, 4, figsize=(20, 13))
axes_flat = axes.flatten()
for m in range(12):
    im = axes_flat[m].imshow(biomass_monthly_mean[m], cmap="YlGn", vmin=0, vmax=1800)
    axes_flat[m].set_title(f"{month_names[m]} — Mean Biomass", fontsize=11)
    axes_flat[m].axis("off")
plt.suptitle("Above-Ground Biomass (kg DM/ha) — Long-term Monthly Mean", fontsize=15, y=1.01)
fig.colorbar(im, ax=axes_flat, fraction=0.02, pad=0.02, label="kg DM/ha")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 2 — Grazing suitability classification from biomass
# High >=1000, Moderate 500-1000, Low 187-500, Poor 50-187, Unsuitable <50
# ============================================================

def classify_biomass(biomass_array):
    """Returns integer class array: 5=High,4=Moderate,3=Low,2=Poor,1=Unsuitable,0=NoData"""
    classified = np.zeros_like(biomass_array, dtype="uint8")
    valid = ~np.isnan(biomass_array)

    classified[valid & (biomass_array >= 1000)] = 5   # High
    classified[valid & (biomass_array >= 500) & (biomass_array < 1000)] = 4   # Moderate
    classified[valid & (biomass_array >= 187) & (biomass_array < 500)] = 3    # Low
    classified[valid & (biomass_array >= 50) & (biomass_array < 187)] = 2     # Poor
    classified[valid & (biomass_array < 50)] = 1                              # Unsuitable
    # 0 remains for NoData (where ~valid)
    return classified

suitability_class_monthly = np.stack(
    [classify_biomass(biomass_monthly_mean[m]) for m in range(12)], axis=0
)  # (12, H, W)

class_meta = ref_meta.copy()
class_meta.update({"dtype": "uint8", "nodata": 0})
class_path = os.path.join(BIOMASS_DIR, "grazing_suitability_class_monthly.tif")
with rasterio.open(class_path, "w", **class_meta) as dst:
    for m in range(12):
        dst.write(suitability_class_monthly[m], m + 1)
print(f"✅ Saved: {os.path.basename(class_path)}")

# --- VISUAL: suitability class map for August (peak season) and January (dry) ---
class_labels = {0: "NoData", 1: "Unsuitable", 2: "Poor", 3: "Low", 4: "Moderate", 5: "High"}
class_colors = {0: "#ffffff", 1: "#8b0000", 2: "#ff8c00", 3: "#ffd700", 4: "#9acd32", 5: "#006400"}

cmap_list = [class_colors[i] for i in range(6)]
cmap_class = mcolors.ListedColormap(cmap_list)
bounds_class = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
norm_class = mcolors.BoundaryNorm(bounds_class, cmap_class.N)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax, month_idx, label in [(axes[0], 7, "August (wet season peak)"), (axes[1], 0, "January (dry season)")]:
    im = ax.imshow(suitability_class_monthly[month_idx], cmap=cmap_class, norm=norm_class)
    ax.set_title(f"Grazing Suitability — {label}")
    ax.axis("off")
cbar = fig.colorbar(im, ax=axes, ticks=range(6), fraction=0.025, pad=0.02)
cbar.ax.set_yticklabels([class_labels[i] for i in range(6)])
plt.tight_layout()
plt.show()

# --- Class distribution table for both months ---
print("\n📊 Suitability class distribution:")
for month_idx, label in [(7, "August"), (0, "January")]:
    print(f"\n  {label}:")
    total_valid = (suitability_class_monthly[month_idx] > 0).sum()
    for c in range(1, 6):
        pct = (suitability_class_monthly[month_idx] == c).sum() / total_valid * 100
        print(f"    {class_labels[c]}: {pct:.1f}%")

# ============================================================
# STEP 3 — Rainfall-NDVI lag correlation (test lags 0 to 3 months)
# ============================================================

print("\n" + "=" * 60)
print("RAINFALL-NDVI LAG CORRELATION")
print("=" * 60)

# --- Load CHIRPS into a matching 4D array ---
chirps_years = manifest["chirps_years_available"]
n_chirps_years = len(chirps_years)

# Build rainfall 4D array — note some years/months may be missing (2025 partial)
rainfall_4d = np.full((n_chirps_years, 12, GRID_HEIGHT, GRID_WIDTH), np.nan, dtype="float32")

for y_idx, year in enumerate(chirps_years):
    chirps_path = os.path.join(PROCESSED_DIR, "chirps", f"CHIRPS_{year}.tif")
    with rasterio.open(chirps_path) as src:
        data = src.read()
        nodata_val = src.nodata if src.nodata is not None else -9999.0
        n_bands = data.shape[0]
        data_masked = np.where(data == nodata_val, np.nan, data)
        rainfall_4d[y_idx, :n_bands] = data_masked  # handles partial 2025 (<12 bands)

print(f"✅ Rainfall 4D array built: shape = {rainfall_4d.shape}")

# --- Flatten to time series per pixel is too expensive at full res for a quick lag test;
#     compute lag correlation on SPATIALLY AVERAGED monthly series instead (region-wide),
#     which is the standard approach for identifying the dominant lag before applying
#     it in per-pixel biomass modeling (Phase 7) ---

# Build a single continuous monthly time series for both variables (only years common to both)
common_years = [y for y in NDVI_YEARS if y in chirps_years]
print(f"Years used for lag correlation (common to NDVI & CHIRPS): {common_years}")

ndvi_ts = []
rain_ts = []
for year in common_years:
    y_idx_ndvi = NDVI_YEARS.index(year)
    y_idx_rain = chirps_years.index(year)
    for m in range(12):
        ndvi_val = np.nanmean(ndvi_4d[y_idx_ndvi, m])
        rain_val = np.nanmean(rainfall_4d[y_idx_rain, m])
        if not np.isnan(rain_val):  # skip months missing from CHIRPS (2025 tail)
            ndvi_ts.append(ndvi_val)
            rain_ts.append(rain_val)

ndvi_ts = np.array(ndvi_ts)
rain_ts = np.array(rain_ts)
print(f"Time series length: {len(ndvi_ts)} months")

lag_correlations = {}
for lag in range(0, 4):
    if lag == 0:
        r = np.corrcoef(rain_ts, ndvi_ts)[0, 1]
    else:
        r = np.corrcoef(rain_ts[:-lag], ndvi_ts[lag:])[0, 1]
    lag_correlations[lag] = r
    print(f"   Lag {lag} month(s): r = {r:.3f}")

best_lag = max(lag_correlations, key=lambda k: lag_correlations[k])
print(f"\n🏆 Strongest correlation at lag = {best_lag} month(s), r = {lag_correlations[best_lag]:.3f}")

# --- Visual: lag correlation bar chart ---
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
lags = list(lag_correlations.keys())
rvals = list(lag_correlations.values())
colors_lag = ["#1f77b4" if l != best_lag else "#d62728" for l in lags]
ax.bar(lags, rvals, color=colors_lag)
ax.set_xlabel("Lag (months)")
ax.set_ylabel("Correlation coefficient (r)")
ax.set_title("Rainfall → NDVI Lag Correlation (region-averaged, 2016-2025)")
ax.set_xticks(lags)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# Save lag correlation results
lag_results_path = os.path.join(BIOMASS_DIR, "rainfall_ndvi_lag_correlation.json")
with open(lag_results_path, "w") as f:
    json.dump({str(k): float(v) for k, v in lag_correlations.items()}, f, indent=2)
print(f"✅ Saved: {os.path.basename(lag_results_path)}")

# ============================================================
# STEP 4 — Dry season and wet season biomass export
# Adamawa: Wet season ~ Apr-Oct (months 4-10), Dry season ~ Nov-Mar (months 11,12,1,2,3)
# ============================================================

WET_MONTHS = [4, 5, 6, 7, 8, 9, 10]   # April-October (1-indexed)
DRY_MONTHS = [11, 12, 1, 2, 3]         # November-March (1-indexed)

wet_month_idx = [m - 1 for m in WET_MONTHS]
dry_month_idx = [m - 1 for m in DRY_MONTHS]

wet_season_biomass = np.nanmean(biomass_monthly_mean[wet_month_idx], axis=0)
dry_season_biomass = np.nanmean(biomass_monthly_mean[dry_month_idx], axis=0)

season_meta = ref_meta.copy()
season_meta.update({"count": 1})

wet_path = os.path.join(BIOMASS_DIR, "biomass_wet_season_mean.tif")
with rasterio.open(wet_path, "w", **season_meta) as dst:
    dst.write(np.where(np.isnan(wet_season_biomass), -9999.0, wet_season_biomass).astype("float32"), 1)
print(f"✅ Saved: {os.path.basename(wet_path)}")

dry_path = os.path.join(BIOMASS_DIR, "biomass_dry_season_mean.tif")
with rasterio.open(dry_path, "w", **season_meta) as dst:
    dst.write(np.where(np.isnan(dry_season_biomass), -9999.0, dry_season_biomass).astype("float32"), 1)
print(f"✅ Saved: {os.path.basename(dry_path)}")

# --- Visual: wet vs dry season biomass side by side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
im0 = axes[0].imshow(wet_season_biomass, cmap="YlGn", vmin=0, vmax=1800)
axes[0].set_title(f"Wet Season Mean Biomass (Apr-Oct)\nMean: {np.nanmean(wet_season_biomass):.0f} kg DM/ha")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(dry_season_biomass, cmap="YlOrBr_r", vmin=0, vmax=1800)
axes[1].set_title(f"Dry Season Mean Biomass (Nov-Mar)\nMean: {np.nanmean(dry_season_biomass):.0f} kg DM/ha")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

wet_dry_ratio = np.nanmean(wet_season_biomass) / np.nanmean(dry_season_biomass)
print(f"\n📊 Wet/Dry season biomass ratio: {wet_dry_ratio:.2f}x "
      f"(expect wet season substantially higher — this drives seasonal migration timing)")

# ============================================================
# STEP 5 — NDVI anomaly biomass export (drought-year biomass impact)
# ============================================================

biomass_anomaly = biomass_4d - biomass_monthly_mean[np.newaxis, :, :, :]

anomaly_biomass_dir = os.path.join(BIOMASS_DIR, "biomass_anomaly")
os.makedirs(anomaly_biomass_dir, exist_ok=True)

for y_idx, year in enumerate(NDVI_YEARS):
    path = os.path.join(anomaly_biomass_dir, f"biomass_anomaly_{year}.tif")
    with rasterio.open(path, "w", **biomass_meta) as dst:
        for m in range(12):
            band = np.where(np.isnan(biomass_anomaly[y_idx, m]), -9999.0, biomass_anomaly[y_idx, m]).astype("float32")
            dst.write(band, m + 1)

print(f"✅ Saved {len(NDVI_YEARS)} biomass anomaly files → {anomaly_biomass_dir}/")

print("\n" + "=" * 60)
print("✅ PHASE 3 CELL 3 COMPLETE")
print(f"   Biomass monthly mean: {biomass_mean_path}")
print(f"   Suitability classification: {class_path}")
print(f"   Lag correlation results: {lag_results_path}")
print(f"   Wet/dry season biomass: {wet_path}, {dry_path}")
print(f"   Biomass anomaly per year: {anomaly_biomass_dir}/")
print("=" * 60)

##Phase 4

###Normalize Input Layers + Build Conflict Exclusion Mask

In [ ]:
# ============================================================
# PHASE 4 — CELL 1
# Normalize All Input Layers (0-1 scale) + Conflict Exclusion Mask
# WITH VISUAL QA
# ============================================================

import os
import json
import glob
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# --- Session restore: redefine constants + paths ---
DRIVE_FOLDER = "pasture-mapping-adamawa"
TARGET_CRS = "EPSG:32633"
START_YEAR = 2016
END_YEAR = 2025

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROCESSED_DIR = "/content/data/processed/final"

manifest_path = os.path.join(PROCESSED_DIR, "manifest.json")
with open(manifest_path, "r") as f:
    manifest = json.load(f)

GRID_WIDTH = manifest["grid_width"]
GRID_HEIGHT = manifest["grid_height"]
GRID_TRANSFORM = manifest["grid_transform"]
GRID_CRS = manifest["crs"]

ref_meta = {
    "driver": "GTiff", "height": GRID_HEIGHT, "width": GRID_WIDTH,
    "count": 1, "dtype": "float32", "crs": GRID_CRS,
    "transform": rasterio.Affine(*GRID_TRANSFORM[:6]),
    "nodata": -9999.0, "tiled": True, "blockxsize": 256, "blockysize": 256, "compress": "lzw"
}

SUITABILITY_DIR = os.path.join(PROCESSED_DIR, "suitability")
os.makedirs(SUITABILITY_DIR, exist_ok=True)

month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

# ============================================================
# STEP 1 — Load required layers
# ============================================================

def load_raster(path, band=1):
    with rasterio.open(path) as src:
        data = src.read(band).astype("float32")
        nodata = src.nodata
        data = np.where(data == nodata, np.nan, data)
    return data

# Static layers
elevation = load_raster(os.path.join(PROCESSED_DIR, "static", "elevation.tif"))
slope = load_raster(os.path.join(PROCESSED_DIR, "static", "slope.tif"))
dist_water = load_raster(os.path.join(PROCESSED_DIR, "static", "distance_to_water.tif"))
landcover = load_raster(os.path.join(PROCESSED_DIR, "static", "landcover.tif"))
population = load_raster(os.path.join(PROCESSED_DIR, "static", "population.tif"))

# Biomass monthly mean (12 bands) — from Phase 3
biomass_monthly_mean = np.stack([
    load_raster(os.path.join(PROCESSED_DIR, "biomass", "biomass_monthly_mean_2016-2025.tif"), band=m+1)
    for m in range(12)
], axis=0)

# NDVI climatology mean (12 bands) — from Phase 3
ndvi_clim_mean = np.stack([
    load_raster(os.path.join(PROCESSED_DIR, "climatology", "NDVI_climatology_mean_2016-2025.tif"), band=m+1)
    for m in range(12)
], axis=0)

# Rainfall monthly mean — build from CHIRPS files (mean across years per month)
chirps_years = manifest["chirps_years_available"]
rainfall_stack = []
for m in range(12):
    monthly_vals = []
    for year in chirps_years:
        path = os.path.join(PROCESSED_DIR, "chirps", f"CHIRPS_{year}.tif")
        with rasterio.open(path) as src:
            if m + 1 <= src.count:
                band = src.read(m + 1).astype("float32")
                nodata = src.nodata
                band = np.where(band == nodata, np.nan, band)
                monthly_vals.append(band)
    rainfall_stack.append(np.nanmean(np.stack(monthly_vals, axis=0), axis=0))
rainfall_monthly_mean = np.stack(rainfall_stack, axis=0)  # (12, H, W)

print("✅ All input layers loaded:")
print(f"   Elevation: {elevation.shape}, range {np.nanmin(elevation):.0f}-{np.nanmax(elevation):.0f} m")
print(f"   Slope: {slope.shape}, range {np.nanmin(slope):.1f}-{np.nanmax(slope):.1f}°")
print(f"   Distance to water: {dist_water.shape}, range {np.nanmin(dist_water):.0f}-{np.nanmax(dist_water):.0f} m")
print(f"   Landcover: {landcover.shape}, classes present: {sorted(np.unique(landcover[~np.isnan(landcover)]))}")
print(f"   Population: {population.shape}, range {np.nanmin(population):.1f}-{np.nanmax(population):.1f}")
print(f"   Biomass monthly mean: {biomass_monthly_mean.shape}")
print(f"   NDVI climatology mean: {ndvi_clim_mean.shape}")
print(f"   Rainfall monthly mean: {rainfall_monthly_mean.shape}, range {np.nanmin(rainfall_monthly_mean):.0f}-{np.nanmax(rainfall_monthly_mean):.0f} mm")

# ============================================================
# STEP 2 — Normalization function (0-1 scale, min-max per layer)
# ============================================================

def normalize_01(array, invert=False, p_low=1, p_high=99):
    """
    Min-max normalize to 0-1 using percentile clipping (1st-99th) to avoid
    outlier pixels compressing the useful range. If invert=True, higher raw
    value -> lower score (used for slope, distance, population).
    """
    valid = array[~np.isnan(array)]
    lo, hi = np.percentile(valid, [p_low, p_high])
    clipped = np.clip(array, lo, hi)
    norm = (clipped - lo) / (hi - lo + 1e-9)
    norm = np.where(np.isnan(array), np.nan, norm)
    if invert:
        norm = np.where(np.isnan(norm), np.nan, 1 - norm)
    return norm

# --- Normalize each monthly layer independently for biomass, NDVI, rainfall ---
biomass_norm = np.stack([normalize_01(biomass_monthly_mean[m]) for m in range(12)], axis=0)
ndvi_norm = np.stack([normalize_01(ndvi_clim_mean[m]) for m in range(12)], axis=0)
rainfall_norm = np.stack([normalize_01(rainfall_monthly_mean[m]) for m in range(12)], axis=0)

# --- Water proximity: invert (closer=better), THRESHOLD at 20 km per spec ---
# Beyond 20km, score = 0 (not just low) — clip distance at 20000m before normalizing
dist_water_thresholded = np.clip(dist_water, 0, 20000)
water_prox_norm = normalize_01(dist_water_thresholded, invert=True, p_low=0, p_high=100)

# --- Slope: invert (flatter=better) ---
slope_norm = normalize_01(slope, invert=True)

# --- Population: invert (less crowded=better) ---
population_norm = normalize_01(population, invert=True)

print("\n✅ Normalization complete — all layers scaled 0-1")
print(f"   Biomass norm range: {np.nanmin(biomass_norm):.3f}-{np.nanmax(biomass_norm):.3f}")
print(f"   Water proximity norm range: {np.nanmin(water_prox_norm):.3f}-{np.nanmax(water_prox_norm):.3f}")
print(f"   Slope norm range: {np.nanmin(slope_norm):.3f}-{np.nanmax(slope_norm):.3f}")
print(f"   Population norm range: {np.nanmin(population_norm):.3f}-{np.nanmax(population_norm):.3f}")

# --- VISUAL: normalized static layers ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
im0 = axes[0].imshow(water_prox_norm, cmap="Blues", vmin=0, vmax=1)
axes[0].set_title("Water Proximity (normalized, inverted)\n1=close to water, 0=far/beyond 20km")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(slope_norm, cmap="Greens", vmin=0, vmax=1)
axes[1].set_title("Slope (normalized, inverted)\n1=flat, 0=steep")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(population_norm, cmap="Purples", vmin=0, vmax=1)
axes[2].set_title("Population Pressure (normalized, inverted)\n1=sparse, 0=dense")
plt.colorbar(im2, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.show()

# --- VISUAL: normalized biomass/NDVI/rainfall for August (peak season) ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
im0 = axes[0].imshow(biomass_norm[7], cmap="YlGn", vmin=0, vmax=1)
axes[0].set_title("Biomass (normalized) — August")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(ndvi_norm[7], cmap="RdYlGn", vmin=0, vmax=1)
axes[1].set_title("NDVI (normalized) — August")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(rainfall_norm[7], cmap="Blues", vmin=0, vmax=1)
axes[2].set_title("Rainfall (normalized) — August")
plt.colorbar(im2, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.show()

# ============================================================
# STEP 3 — Conflict exclusion mask
# Cropland (class 40), Built-up (class 50), Dense forest (class 10),
# Population > 300/km², Slope > 25 degrees
# ============================================================

print("\nBuilding conflict exclusion mask...")

exclusion_mask = np.zeros_like(landcover, dtype=bool)
exclusion_reasons = np.zeros_like(landcover, dtype="uint8")  # for diagnostic breakdown

cropland_mask = (landcover == 40)
builtup_mask = (landcover == 50)
forest_mask = (landcover == 10)
population_mask = (population > 300)
slope_mask = (slope > 25)

exclusion_mask = cropland_mask | builtup_mask | forest_mask | population_mask | slope_mask

# Track which reason(s) triggered exclusion (bitflags for diagnostics)
exclusion_reasons = (
    cropland_mask.astype("uint8") * 1 +
    builtup_mask.astype("uint8") * 2 +
    forest_mask.astype("uint8") * 4 +
    population_mask.astype("uint8") * 8 +
    slope_mask.astype("uint8") * 16
)

valid_pixels = ~np.isnan(landcover)
n_valid = valid_pixels.sum()
n_excluded = (exclusion_mask & valid_pixels).sum()
pct_excluded = n_excluded / n_valid * 100

print(f"✅ Exclusion mask built:")
print(f"   Cropland pixels: {(cropland_mask & valid_pixels).sum()} ({(cropland_mask & valid_pixels).sum()/n_valid*100:.1f}%)")
print(f"   Built-up pixels: {(builtup_mask & valid_pixels).sum()} ({(builtup_mask & valid_pixels).sum()/n_valid*100:.1f}%)")
print(f"   Dense forest pixels: {(forest_mask & valid_pixels).sum()} ({(forest_mask & valid_pixels).sum()/n_valid*100:.1f}%)")
print(f"   Pop > 300/km² pixels: {(population_mask & valid_pixels).sum()} ({(population_mask & valid_pixels).sum()/n_valid*100:.1f}%)")
print(f"   Slope > 25° pixels: {(slope_mask & valid_pixels).sum()} ({(slope_mask & valid_pixels).sum()/n_valid*100:.1f}%)")
print(f"   TOTAL excluded (any reason, union): {n_excluded} ({pct_excluded:.1f}% of valid area)")

if pct_excluded > 60:
    print("⚠️ WARNING: over 60% of the region excluded. This may leave too little "
          "area for realistic zone delineation in the next cell — worth reviewing thresholds.")
elif pct_excluded < 5:
    print("⚠️ WARNING: less than 5% excluded — check that landcover/population/slope "
          "layers are loading correctly (this seems too permissive for a real region).")
else:
    print("✅ Exclusion percentage looks reasonable.")

# Save exclusion mask
exclusion_meta = ref_meta.copy()
exclusion_meta.update({"dtype": "uint8", "nodata": 255})
exclusion_path = os.path.join(SUITABILITY_DIR, "exclusion_mask.tif")
exclusion_out = np.where(~valid_pixels, 255, exclusion_mask.astype("uint8"))
with rasterio.open(exclusion_path, "w", **exclusion_meta) as dst:
    dst.write(exclusion_out, 1)
print(f"✅ Saved: {os.path.basename(exclusion_path)}")

# --- VISUAL: exclusion mask breakdown by reason ---
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Binary exclusion mask
axes[0].imshow(np.where(valid_pixels, exclusion_mask, np.nan), cmap="Reds")
axes[0].set_title(f"Exclusion Mask (union)\n{pct_excluded:.1f}% of area excluded")
axes[0].axis("off")

# Reason breakdown (dominant reason per pixel, for visualization)
reason_display = np.full(landcover.shape, np.nan)
reason_display = np.where(cropland_mask, 1, reason_display)
reason_display = np.where(forest_mask & np.isnan(reason_display), 2, reason_display)
reason_display = np.where(builtup_mask & np.isnan(reason_display), 3, reason_display)
reason_display = np.where(population_mask & np.isnan(reason_display), 4, reason_display)
reason_display = np.where(slope_mask & np.isnan(reason_display), 5, reason_display)

reason_colors = ["#f096ff", "#006400", "#fa0000", "#800080", "#ff8c00"]
reason_cmap = mcolors.ListedColormap(reason_colors)
reason_bounds = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
reason_norm = mcolors.BoundaryNorm(reason_bounds, reason_cmap.N)

im1 = axes[1].imshow(reason_display, cmap=reason_cmap, norm=reason_norm)
axes[1].set_title("Primary Exclusion Reason")
axes[1].axis("off")
cbar = plt.colorbar(im1, ax=axes[1], ticks=[1,2,3,4,5], fraction=0.046)
cbar.ax.set_yticklabels(["Cropland", "Forest", "Built-up", "Pop>300", "Slope>25°"])

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("✅ PHASE 4 CELL 1 COMPLETE")
print(f"   Normalized layers ready in memory for weighted combination")
print(f"   Exclusion mask saved: {exclusion_path}")
print("=" * 60)

###Weighted Linear Combination + Suitability Classification

In [ ]:
# ============================================================
# PHASE 4 — CELL 2
# Weighted Linear Combination -> Final Suitability Score (0-100)
# + Classification (High/Moderate/Low/Unsuitable)
# + Apply Exclusion Mask
# WITH VISUAL QA
# ============================================================

# --- Session restore check ---
assert 'biomass_norm' in dir(), "⚠️ Run Phase 4 Cell 1 first."

# ============================================================
# STEP 1 — Weights (must sum to 1.0 — verify explicitly)
# ============================================================

WEIGHTS = {
    "biomass": 0.35,
    "ndvi": 0.20,
    "rainfall": 0.20,
    "water": 0.15,
    "slope": 0.05,
    "population": 0.05,
}

weight_sum = sum(WEIGHTS.values())
print(f"📊 Weight sum check: {weight_sum:.3f}")
assert abs(weight_sum - 1.0) < 1e-6, f"⚠️ Weights do not sum to 1.0! Sum = {weight_sum}"
print("✅ Weights sum to 1.0 exactly")
for k, v in WEIGHTS.items():
    print(f"   {k}: {v}")

# ============================================================
# STEP 2 — Weighted Linear Combination (computed per month)
# Static layers (water, slope, population) are constant across months;
# biomass/ndvi/rainfall vary by month
# ============================================================

print("\nComputing weighted linear combination per month...")

suitability_score_monthly = np.zeros((12, GRID_HEIGHT, GRID_WIDTH), dtype="float32")

for m in range(12):
    score = (
        WEIGHTS["biomass"]    * np.nan_to_num(biomass_norm[m], nan=0) +
        WEIGHTS["ndvi"]       * np.nan_to_num(ndvi_norm[m], nan=0) +
        WEIGHTS["rainfall"]   * np.nan_to_num(rainfall_norm[m], nan=0) +
        WEIGHTS["water"]      * np.nan_to_num(water_prox_norm, nan=0) +
        WEIGHTS["slope"]      * np.nan_to_num(slope_norm, nan=0) +
        WEIGHTS["population"] * np.nan_to_num(population_norm, nan=0)
    )
    # Re-apply NaN mask where ANY critical input was NaN (avoid false low scores at edges)
    any_nan = (
        np.isnan(biomass_norm[m]) | np.isnan(ndvi_norm[m]) | np.isnan(rainfall_norm[m]) |
        np.isnan(water_prox_norm) | np.isnan(slope_norm) | np.isnan(population_norm)
    )
    score = np.where(any_nan, np.nan, score)

    # Scale 0-1 -> 0-100
    suitability_score_monthly[m] = score * 100

print(f"✅ Suitability score computed for all 12 months")
print(f"   Range: {np.nanmin(suitability_score_monthly):.1f} to {np.nanmax(suitability_score_monthly):.1f}")
print(f"   Mean: {np.nanmean(suitability_score_monthly):.1f}")

# ============================================================
# STEP 3 — Apply exclusion mask (set excluded pixels to 0 / NoData for suitability)
# ============================================================

# exclusion_mask is (H,W), broadcast across all 12 months
exclusion_mask_3d = np.broadcast_to(exclusion_mask, suitability_score_monthly.shape)

suitability_score_final = np.where(exclusion_mask_3d, 0, suitability_score_monthly)
# Preserve NaN where original data was NaN (outside study area / true nodata)
suitability_score_final = np.where(np.isnan(suitability_score_monthly), np.nan, suitability_score_final)

print(f"\n✅ Exclusion mask applied — {(exclusion_mask_3d & ~np.isnan(suitability_score_monthly)).sum()} "
      f"pixel-months forced to score 0")

# --- Save monthly suitability score raster ---
score_meta = ref_meta.copy()
score_meta.update({"count": 12})
score_path = os.path.join(SUITABILITY_DIR, "suitability_score_monthly_0-100.tif")
with rasterio.open(score_path, "w", **score_meta) as dst:
    for m in range(12):
        band = np.where(np.isnan(suitability_score_final[m]), -9999.0, suitability_score_final[m]).astype("float32")
        dst.write(band, m + 1)
print(f"✅ Saved: {os.path.basename(score_path)}")

# ============================================================
# STEP 4 — Classification: High>=65, Moderate 45-65, Low 25-45, Unsuitable<25
# ============================================================

def classify_suitability(score_array):
    classified = np.zeros_like(score_array, dtype="uint8")
    valid = ~np.isnan(score_array)
    classified[valid & (score_array >= 65)] = 4                                  # High
    classified[valid & (score_array >= 45) & (score_array < 65)] = 3            # Moderate
    classified[valid & (score_array >= 25) & (score_array < 45)] = 2            # Low
    classified[valid & (score_array < 25)] = 1                                   # Unsuitable
    return classified

suitability_class_final = np.stack(
    [classify_suitability(suitability_score_final[m]) for m in range(12)], axis=0
)

class_meta = ref_meta.copy()
class_meta.update({"count": 12, "dtype": "uint8", "nodata": 0})
class_final_path = os.path.join(SUITABILITY_DIR, "suitability_classification_monthly.tif")
with rasterio.open(class_final_path, "w", **class_meta) as dst:
    for m in range(12):
        dst.write(suitability_class_final[m], m + 1)
print(f"✅ Saved: {os.path.basename(class_final_path)}")

# ============================================================
# VISUAL QA
# ============================================================

# --- 12-panel suitability score map ---
fig, axes = plt.subplots(3, 4, figsize=(20, 13))
axes_flat = axes.flatten()
for m in range(12):
    im = axes_flat[m].imshow(suitability_score_final[m], cmap="RdYlGn", vmin=0, vmax=100)
    axes_flat[m].set_title(f"{month_names[m]} — Suitability Score", fontsize=11)
    axes_flat[m].axis("off")
plt.suptitle("Grazing Suitability Score (0-100) — Monthly, Exclusions Applied", fontsize=15, y=1.01)
fig.colorbar(im, ax=axes_flat, fraction=0.02, pad=0.02, label="Score")
plt.tight_layout()
plt.show()

# --- Classification maps: August vs January, with proper class colors ---
class_labels_suit = {0: "NoData", 1: "Unsuitable", 2: "Low", 3: "Moderate", 4: "High"}
class_colors_suit = {0: "#ffffff", 1: "#8b0000", 2: "#ffd700", 3: "#9acd32", 4: "#006400"}
cmap_suit = mcolors.ListedColormap([class_colors_suit[i] for i in range(5)])
bounds_suit = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]
norm_suit = mcolors.BoundaryNorm(bounds_suit, cmap_suit.N)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax, month_idx, label in [(axes[0], 7, "August (wet peak)"), (axes[1], 0, "January (dry)")]:
    im = ax.imshow(suitability_class_final[month_idx], cmap=cmap_suit, norm=norm_suit)
    ax.set_title(f"Suitability Classification — {label}")
    ax.axis("off")
cbar = fig.colorbar(im, ax=axes, ticks=range(5), fraction=0.025, pad=0.02)
cbar.ax.set_yticklabels([class_labels_suit[i] for i in range(5)])
plt.tight_layout()
plt.show()

# --- Classification distribution table ---
print("\n📊 Suitability classification distribution:")
for month_idx, label in [(7, "August"), (0, "January")]:
    print(f"\n  {label}:")
    total_valid = (suitability_class_final[month_idx] > 0).sum()
    for c in range(1, 5):
        pct = (suitability_class_final[month_idx] == c).sum() / total_valid * 100
        area_km2 = (suitability_class_final[month_idx] == c).sum() * (1000*1000) / 1e6
        print(f"    {class_labels_suit[c]}: {pct:.1f}% ({area_km2:.0f} km²)")

# --- Sanity check: how much "High+Moderate" area exists per month (grazing-viable land) ---
print("\n📊 High + Moderate suitability area per month (potential zone source area):")
viable_area_by_month = []
for m in range(12):
    viable = ((suitability_class_final[m] == 3) | (suitability_class_final[m] == 4)).sum()
    area_km2 = viable * 1.0  # 1km x 1km pixels
    viable_area_by_month.append(area_km2)
    print(f"   {month_names[m]}: {area_km2:.0f} km²")

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(month_names, viable_area_by_month, marker="o", color="#2ca02c", linewidth=2)
ax.set_ylabel("High+Moderate suitability area (km²)")
ax.set_title("Seasonal Grazing-Viable Area — Adamawa")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("✅ PHASE 4 CELL 2 COMPLETE")
print(f"   Suitability score: {score_path}")
print(f"   Suitability classification: {class_final_path}")
print("=" * 60)

###Zone Delineation

In [ ]:
# ============================================================
# PHASE 4 — CELL 3 (FINAL, CONSOLIDATED)
# Zone Delineation — threshold=35, smoothing=5, min_area=5km²,
# 4-connectivity, no dilation → 104 zones (confirmed)
# Self-contained: reloads all inputs from disk
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.ndimage import uniform_filter
from rasterio.features import shapes
from shapely.geometry import shape
from pyproj import Transformer
import geopandas as gpd

# ============================================================
# SESSION RESTORE
# ============================================================

DRIVE_FOLDER = "pasture-mapping-adamawa"
TARGET_CRS = "EPSG:32633"

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROCESSED_DIR = "/content/data/processed/final"
SUITABILITY_DIR = os.path.join(PROCESSED_DIR, "suitability")
os.makedirs(SUITABILITY_DIR, exist_ok=True)

month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

manifest_path = os.path.join(PROCESSED_DIR, "manifest.json")
with open(manifest_path, "r") as f:
    manifest = json.load(f)

GRID_WIDTH = manifest["grid_width"]
GRID_HEIGHT = manifest["grid_height"]
GRID_TRANSFORM = manifest["grid_transform"]
GRID_CRS = manifest["crs"]
transform_obj = rasterio.Affine(*GRID_TRANSFORM[:6])

ref_meta = {
    "driver": "GTiff", "height": GRID_HEIGHT, "width": GRID_WIDTH,
    "count": 1, "dtype": "float32", "crs": GRID_CRS,
    "transform": transform_obj,
    "nodata": -9999.0, "tiled": True, "blockxsize": 256, "blockysize": 256, "compress": "lzw"
}

PIXEL_AREA_KM2 = 1.0

# --- Load suitability score (12-band, from Phase 4 Cell 2) ---
score_path = os.path.join(SUITABILITY_DIR, "suitability_score_monthly_0-100.tif")
assert os.path.exists(score_path), "⚠️ suitability_score_monthly_0-100.tif not found. Run Phase 4 Cell 2 first."
with rasterio.open(score_path) as src:
    suitability_score_final = np.stack([
        np.where(src.read(m + 1) == src.nodata, np.nan, src.read(m + 1))
        for m in range(12)
    ], axis=0)
print(f"✅ Loaded suitability score: shape {suitability_score_final.shape}")

# --- Load biomass monthly mean (from Phase 3) ---
biomass_path = os.path.join(PROCESSED_DIR, "biomass", "biomass_monthly_mean_2016-2025.tif")
assert os.path.exists(biomass_path), "⚠️ biomass_monthly_mean_2016-2025.tif not found. Run Phase 3 Cell 3 first."
with rasterio.open(biomass_path) as src:
    biomass_monthly_mean = np.stack([
        np.where(src.read(m + 1) == src.nodata, np.nan, src.read(m + 1))
        for m in range(12)
    ], axis=0)
print(f"✅ Loaded biomass monthly mean: shape {biomass_monthly_mean.shape}")

# ============================================================
# STEP 1 — Annual mean suitability score (delineation base layer)
# ============================================================

annual_mean_score = np.nanmean(suitability_score_final, axis=0)
print(f"Annual mean suitability score range: "
      f"{np.nanmin(annual_mean_score):.1f} to {np.nanmax(annual_mean_score):.1f}")

# ============================================================
# STEP 2 — Smoothing (CONFIRMED: size=5), NaN-aware
# ============================================================

FINAL_SMOOTHING_SIZE = 5
FINAL_THRESHOLD = 35
MIN_AREA_KM2 = 5

print(f"Using: smoothing={FINAL_SMOOTHING_SIZE}, threshold={FINAL_THRESHOLD}, min_area={MIN_AREA_KM2}km²")

score_filled = np.nan_to_num(annual_mean_score, nan=0.0)
valid_mask_f = (~np.isnan(annual_mean_score)).astype("float32")

smoothed_sum = uniform_filter(score_filled, size=FINAL_SMOOTHING_SIZE, mode="constant", cval=0.0)
smoothed_count = uniform_filter(valid_mask_f, size=FINAL_SMOOTHING_SIZE, mode="constant", cval=0.0)

with np.errstate(invalid="ignore", divide="ignore"):
    score_smoothed = smoothed_sum / smoothed_count
score_smoothed = np.where(smoothed_count > 0, score_smoothed, np.nan)

fig, ax = plt.subplots(1, 1, figsize=(8, 7))
im = ax.imshow(score_smoothed, cmap="RdYlGn", vmin=0, vmax=100)
ax.set_title(f"Final Smoothed Suitability Surface (size={FINAL_SMOOTHING_SIZE})")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

# ============================================================
# STEP 3 — Threshold + 4-connectivity labeling, NO dilation
# ============================================================

zone_binary_mask = (score_smoothed >= FINAL_THRESHOLD) & (~np.isnan(score_smoothed))
print(f"Pixels passing threshold >={FINAL_THRESHOLD}: {zone_binary_mask.sum()} "
      f"({zone_binary_mask.sum() / (~np.isnan(score_smoothed)).sum() * 100:.1f}% of valid area)")

structure_4conn = np.array([[0, 1, 0],
                             [1, 1, 1],
                             [0, 1, 0]])

labeled_array, n_raw_zones = ndimage.label(zone_binary_mask, structure=structure_4conn)
print(f"Raw connected components: {n_raw_zones}")

# ============================================================
# STEP 4 — Min-area filter (5 km²)
# ============================================================

min_pixels = int(MIN_AREA_KM2 / PIXEL_AREA_KM2)
zone_sizes = ndimage.sum(zone_binary_mask, labeled_array, range(1, n_raw_zones + 1))
valid_zone_ids_raw = np.where(zone_sizes >= min_pixels)[0] + 1
n_zones_prefilter = len(valid_zone_ids_raw)

final_labeled = np.zeros_like(labeled_array)
for new_id, old_id in enumerate(valid_zone_ids_raw, start=1):
    final_labeled[labeled_array == old_id] = new_id

print(f"Zones after min-area filter: {n_zones_prefilter}")

# --- Visual: raw zone map before attribute computation ---
np.random.seed(42)
zone_colors_lut = np.random.rand(n_zones_prefilter + 1, 3)
zone_colors_lut[0] = [1, 1, 1]
zone_rgb = zone_colors_lut[final_labeled]

fig, ax = plt.subplots(1, 1, figsize=(10, 9))
ax.imshow(zone_rgb)
ax.set_title(f"Delineated Grazing Zones (pre-attribute-filter)\n{n_zones_prefilter} zones "
             f"(threshold≥{FINAL_THRESHOLD}, smoothing={FINAL_SMOOTHING_SIZE}, "
             f"min area {MIN_AREA_KM2}km², 4-connectivity, no dilation)")
ax.axis("off")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 5 — Zone attribute table (NaN-safe from the start)
# ============================================================

print("\nComputing zone attributes (NaN-safe)...")

UTILIZATION_RATE = 0.40
PIXEL_AREA_HA = 100
MONTHLY_INTAKE_PER_TLU = 187.5

transformer_to_wgs84 = Transformer.from_crs(GRID_CRS, "EPSG:4326", always_xy=True)

def quality_from_score(score):
    if score >= 65: return "High"
    elif score >= 45: return "Moderate"
    elif score >= 25: return "Low"
    else: return "Unsuitable"

zone_records = []
skipped_zones = []

for zone_id in range(1, n_zones_prefilter + 1):
    mask = (final_labeled == zone_id)
    n_pixels = mask.sum()
    area_km2 = n_pixels * PIXEL_AREA_KM2

    mean_score = float(np.nanmean(score_smoothed[mask]))
    if np.isnan(mean_score):
        skipped_zones.append(zone_id)
        continue
    quality = quality_from_score(mean_score)

    rows, cols = np.where(mask)
    centroid_row, centroid_col = rows.mean(), cols.mean()
    x_map, y_map = transform_obj * (centroid_col, centroid_row)
    centroid_lon, centroid_lat = transformer_to_wgs84.transform(x_map, y_map)

    monthly_zone_scores = [float(np.nanmean(suitability_score_final[m][mask])) for m in range(12)]
    monthly_zone_biomass = [float(np.nanmean(biomass_monthly_mean[m][mask])) for m in range(12)]

    valid_scores = [s for s in monthly_zone_scores if not np.isnan(s)]
    valid_biomass = [b for b in monthly_zone_biomass if not np.isnan(b)]

    if len(valid_scores) == 0 or len(valid_biomass) == 0:
        skipped_zones.append(zone_id)
        continue

    fallback_score = np.mean(valid_scores)
    fallback_biomass = np.mean(valid_biomass)
    monthly_zone_scores = [fallback_score if np.isnan(s) else s for s in monthly_zone_scores]
    monthly_zone_biomass = [fallback_biomass if np.isnan(b) else b for b in monthly_zone_biomass]

    best_month_idx = int(np.argmax(monthly_zone_scores))
    worst_month_idx = int(np.argmin(monthly_zone_scores))

    monthly_cc_tlu_per_km2 = [(b * UTILIZATION_RATE * PIXEL_AREA_HA) / MONTHLY_INTAKE_PER_TLU
                                for b in monthly_zone_biomass]
    monthly_capacity_tlu = [cc * area_km2 for cc in monthly_cc_tlu_per_km2]

    peak_cattle = int(round(np.nanmax(monthly_capacity_tlu)))
    min_cattle = int(round(np.nanmin(monthly_capacity_tlu)))

    zone_records.append({
        "zone_id": zone_id, "area_km2": round(area_km2, 2), "mean_score": round(mean_score, 2),
        "quality": quality, "centroid_lat": round(centroid_lat, 5), "centroid_lon": round(centroid_lon, 5),
        "best_month": month_names[best_month_idx], "worst_month": month_names[worst_month_idx],
        "peak_cattle": peak_cattle, "min_cattle": min_cattle,
    })

zone_df_attrs = pd.DataFrame(zone_records)
n_final_zones = len(zone_df_attrs)

print(f"✅ Zone attribute table built: {n_final_zones} rows")
if skipped_zones:
    print(f"⚠️ Skipped {len(skipped_zones)} zone(s) with no valid data (boundary fragments): {skipped_zones}")
print(zone_df_attrs.head(10).to_string(index=False))

# ============================================================
# ZONE COUNT VERIFICATION
# ============================================================

print("\n" + "=" * 60)
print("ZONE COUNT VERIFICATION")
print("=" * 60)
print(f"Final zone count: {n_final_zones}")
if 80 <= n_final_zones <= 200:
    print(f"✅ Zone count ({n_final_zones}) is within the expected realistic range (80-200).")
else:
    print(f"⚠️ Zone count ({n_final_zones}) is outside 80-200 — revisit smoothing/threshold.")
print("=" * 60)

# ============================================================
# STEP 6 — Relabel raster to contiguous IDs matching attribute table,
# vectorize, and save
# ============================================================

surviving_zone_ids = sorted(zone_df_attrs["zone_id"].tolist())
final_labeled_clean = np.zeros_like(final_labeled)
old_to_new_id = {}
for new_id, old_id in enumerate(surviving_zone_ids, start=1):
    final_labeled_clean[final_labeled == old_id] = new_id
    old_to_new_id[old_id] = new_id

zone_df_attrs["zone_id"] = zone_df_attrs["zone_id"].map(old_to_new_id)
zone_df_attrs = zone_df_attrs.sort_values("zone_id").reset_index(drop=True)

mask_for_shapes = (final_labeled_clean > 0)
polygons, zone_ids_from_shapes = [], []
for geom, value in shapes(final_labeled_clean.astype("int32"), mask=mask_for_shapes, transform=transform_obj):
    polygons.append(shape(geom))
    zone_ids_from_shapes.append(int(value))

shapes_gdf = gpd.GeoDataFrame({"zone_id": zone_ids_from_shapes, "geometry": polygons}, crs=GRID_CRS)
shapes_dissolved = shapes_gdf.dissolve(by="zone_id", as_index=False)

assert len(shapes_dissolved) == n_final_zones, \
    f"⚠️ Mismatch: {len(shapes_dissolved)} polygons vs {n_final_zones} attribute rows."
print(f"\n✅ Vectorized {len(shapes_dissolved)} zone polygons — matches attribute table")

zones_gdf = shapes_dissolved.merge(zone_df_attrs, on="zone_id", how="left")
assert zones_gdf["mean_score"].isna().sum() == 0, "⚠️ NaN attributes after merge — investigate."

# --- Save all outputs ---
gpkg_path = os.path.join(SUITABILITY_DIR, "grazing_zones.gpkg")
zones_gdf.to_file(gpkg_path, layer="grazing_zones", driver="GPKG")

zone_csv_path = os.path.join(SUITABILITY_DIR, "grazing_zones_attributes.csv")
zone_df_attrs.to_csv(zone_csv_path, index=False)

zone_raster_path = os.path.join(SUITABILITY_DIR, "grazing_zones_raster.tif")
zone_raster_meta = ref_meta.copy()
zone_raster_meta.update({"dtype": "int32", "nodata": 0})
with rasterio.open(zone_raster_path, "w", **zone_raster_meta) as dst:
    dst.write(final_labeled_clean.astype("int32"), 1)

print(f"✅ Saved: {gpkg_path}")
print(f"✅ Saved: {zone_csv_path}")
print(f"✅ Saved: {zone_raster_path}")

# ============================================================
# FINAL VISUALS
# ============================================================

final_zone_sizes_km2 = zone_df_attrs["area_km2"].values
fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.hist(final_zone_sizes_km2, bins=30, color="#2ca02c", edgecolor="black")
ax.set_xlabel("Zone area (km²)")
ax.set_ylabel("Number of zones")
ax.set_title(f"Zone Size Distribution (n={n_final_zones})\n"
             f"Median: {np.median(final_zone_sizes_km2):.1f} km², Max: {final_zone_sizes_km2.max():.0f} km²")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

quality_colors = {"High": "#006400", "Moderate": "#9acd32", "Low": "#ffd700", "Unsuitable": "#8b0000"}
fig, ax = plt.subplots(1, 1, figsize=(11, 10))
for q, color in quality_colors.items():
    subset = zones_gdf[zones_gdf["quality"] == q]
    if len(subset) > 0:
        subset.plot(ax=ax, color=color, edgecolor="black", linewidth=0.3, label=f"{q} ({len(subset)})")
ax.set_title(f"Final Grazing Zones by Quality Class (n={len(zones_gdf)})")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

print("\n📊 Zone quality breakdown:")
print(zone_df_attrs["quality"].value_counts().to_string())
print(f"\n📊 Peak cattle: min={zone_df_attrs['peak_cattle'].min()}, "
      f"max={zone_df_attrs['peak_cattle'].max()}, mean={zone_df_attrs['peak_cattle'].mean():.0f} TLU")
print(f"📊 Zone area: min={zone_df_attrs['area_km2'].min():.1f} km², "
      f"max={zone_df_attrs['area_km2'].max():.1f} km², total={zone_df_attrs['area_km2'].sum():.0f} km²")

print("\n" + "=" * 60)
print(f"✅ PHASE 4 COMPLETE — {n_final_zones} zones finalized and saved")
print("=" * 60)

##Phase 5

### Carrying Capacity Engine (Core Functions)

In [ ]:
# ============================================================
# PHASE 5 — CELL 1
# Carrying Capacity Query Engine — Self-Contained
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

# ============================================================
# SESSION RESTORE
# ============================================================

DRIVE_FOLDER = "pasture-mapping-adamawa"
TARGET_CRS = "EPSG:32633"

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROCESSED_DIR = "/content/data/processed/final"
SUITABILITY_DIR = os.path.join(PROCESSED_DIR, "suitability")
CAPACITY_DIR = os.path.join(PROCESSED_DIR, "capacity")
os.makedirs(CAPACITY_DIR, exist_ok=True)

month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

manifest_path = os.path.join(PROCESSED_DIR, "manifest.json")
with open(manifest_path, "r") as f:
    manifest = json.load(f)

GRID_WIDTH = manifest["grid_width"]
GRID_HEIGHT = manifest["grid_height"]
GRID_TRANSFORM = manifest["grid_transform"]
GRID_CRS = manifest["crs"]

ref_meta = {
    "driver": "GTiff", "height": GRID_HEIGHT, "width": GRID_WIDTH,
    "count": 1, "dtype": "float32", "crs": GRID_CRS,
    "transform": rasterio.Affine(*GRID_TRANSFORM[:6]),
    "nodata": -9999.0, "tiled": True, "blockxsize": 256, "blockysize": 256, "compress": "lzw"
}

PIXEL_AREA_KM2 = 1.0

# --- Load zone attribute table (saved by Phase 4 Cell 3) ---
zone_csv_path = os.path.join(SUITABILITY_DIR, "grazing_zones_attributes.csv")
assert os.path.exists(zone_csv_path), "⚠️ grazing_zones_attributes.csv not found. Run Phase 4 Cell 3 first."
zone_df_attrs = pd.read_csv(zone_csv_path)
print(f"✅ Loaded zone attributes: {len(zone_df_attrs)} zones")

# --- Load zone ID raster (saved by Phase 4 Cell 3) ---
zone_raster_path = os.path.join(SUITABILITY_DIR, "grazing_zones_raster.tif")
assert os.path.exists(zone_raster_path), "⚠️ grazing_zones_raster.tif not found. Run Phase 4 Cell 3 first."
with rasterio.open(zone_raster_path) as src:
    final_labeled_clean = src.read(1)
print(f"✅ Loaded zone raster: shape {final_labeled_clean.shape}, "
      f"{len(np.unique(final_labeled_clean)) - 1} unique zone IDs (excluding 0/nodata)")

# --- Verify raster zone IDs match attribute table zone IDs ---
raster_zone_ids = set(np.unique(final_labeled_clean)) - {0}
attr_zone_ids = set(zone_df_attrs["zone_id"].unique())
assert raster_zone_ids == attr_zone_ids, \
    f"⚠️ MISMATCH: raster has {len(raster_zone_ids)} zone IDs, attributes have {len(attr_zone_ids)}. " \
    f"Diff: {raster_zone_ids.symmetric_difference(attr_zone_ids)}"
print("✅ Raster and attribute table zone IDs match exactly")

# --- Load biomass monthly mean (saved by Phase 3) ---
biomass_path = os.path.join(PROCESSED_DIR, "biomass", "biomass_monthly_mean_2016-2025.tif")
assert os.path.exists(biomass_path), "⚠️ biomass_monthly_mean_2016-2025.tif not found. Run Phase 3 Cell 3."
with rasterio.open(biomass_path) as src:
    biomass_monthly_mean = np.stack([
        np.where(src.read(m + 1) == src.nodata, np.nan, src.read(m + 1))
        for m in range(12)
    ], axis=0)
print(f"✅ Loaded biomass monthly mean: shape {biomass_monthly_mean.shape}, "
      f"range {np.nanmin(biomass_monthly_mean):.1f}-{np.nanmax(biomass_monthly_mean):.1f} kg DM/ha")

# ============================================================
# PATCH — Month normalization helper
# Fixes: string mismatch between full month names ("August") and
# the 3-letter abbreviations actually stored in zone_month_capacity_df ("Aug")
# ============================================================

FULL_MONTH_NAMES = ["January","February","March","April","May","June",
                     "July","August","September","October","November","December"]

MONTH_LOOKUP = {}
for i, (abbr, full) in enumerate(zip(month_names, FULL_MONTH_NAMES), start=1):
    MONTH_LOOKUP[abbr.lower()] = abbr
    MONTH_LOOKUP[full.lower()] = abbr
    MONTH_LOOKUP[str(i)] = abbr

def normalize_month(month):
    """Accepts int (1-12), abbreviation ('Aug'), or full name ('August') -> returns 'Aug' style."""
    if isinstance(month, int):
        return month_names[month - 1]
    key = str(month).strip().lower()
    if key not in MONTH_LOOKUP:
        raise ValueError(f"Unrecognized month '{month}'. Use a number 1-12, "
                          f"an abbreviation like 'Aug', or a full name like 'August'.")
    return MONTH_LOOKUP[key]

# --- Quick test ---
for test_input in ["August", "aug", "Aug", 8]:
    print(f"normalize_month({test_input!r}) -> {normalize_month(test_input)}")

# ============================================================
# STEP 1 — Scientific constants
# ============================================================

CC_CONSTANTS = {
    "TLU_KG": 250,
    "DAILY_INTAKE_PER_TLU_KG": 6.25,
    "MONTHLY_INTAKE_PER_TLU_KG": 187.5,
    "UTILIZATION_RATE": 0.40,
    "PIXEL_AREA_HA": 100,
}

TLU_CONVERSION = {
    "cattle_adult_zebu": 1.00,
    "cattle_young": 0.50,
    "camel": 1.10,
    "horse": 0.80,
    "donkey": 0.50,
    "sheep_goat": 0.10,
}

print("\n📋 Carrying Capacity Constants:")
for k, v in CC_CONSTANTS.items():
    print(f"   {k}: {v}")
print("\n📋 TLU Conversion Factors:")
for k, v in TLU_CONVERSION.items():
    print(f"   {k}: {v}")

# ============================================================
# STEP 2 — Monthly CC (TLU/km²) rasters from biomass
# CC (TLU/km²) = (Biomass x 0.40 x 100) / 187.5
# ============================================================

print("\nComputing monthly carrying capacity rasters (TLU/km²)...")

cc_monthly_tlu_km2 = np.stack([
    (biomass_monthly_mean[m] * CC_CONSTANTS["UTILIZATION_RATE"] * CC_CONSTANTS["PIXEL_AREA_HA"])
    / CC_CONSTANTS["MONTHLY_INTAKE_PER_TLU_KG"]
    for m in range(12)
], axis=0)

print(f"✅ CC raster shape: {cc_monthly_tlu_km2.shape}")
print(f"   Range: {np.nanmin(cc_monthly_tlu_km2):.2f} to {np.nanmax(cc_monthly_tlu_km2):.2f} TLU/km²")

cc_meta = ref_meta.copy()
cc_meta.update({"count": 12})
cc_path = os.path.join(CAPACITY_DIR, "carrying_capacity_monthly_TLU_per_km2.tif")
with rasterio.open(cc_path, "w", **cc_meta) as dst:
    for m in range(12):
        band = np.where(np.isnan(cc_monthly_tlu_km2[m]), -9999.0, cc_monthly_tlu_km2[m]).astype("float32")
        dst.write(band, m + 1)
print(f"✅ Saved: {os.path.basename(cc_path)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
vmax = np.nanpercentile(cc_monthly_tlu_km2[7], 98)
im0 = axes[0].imshow(cc_monthly_tlu_km2[7], cmap="YlGn", vmin=0, vmax=vmax)
axes[0].set_title(f"Carrying Capacity — August\nMean: {np.nanmean(cc_monthly_tlu_km2[7]):.2f} TLU/km²")
plt.colorbar(im0, ax=axes[0], fraction=0.046, label="TLU/km²")

im1 = axes[1].imshow(cc_monthly_tlu_km2[0], cmap="YlOrBr_r", vmin=0, vmax=vmax)
axes[1].set_title(f"Carrying Capacity — January\nMean: {np.nanmean(cc_monthly_tlu_km2[0]):.2f} TLU/km²")
plt.colorbar(im1, ax=axes[1], fraction=0.046, label="TLU/km²")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 3 — Zone x month capacity table (core lookup table)
# ============================================================

print("\nBuilding zone x month capacity table...")

zone_capacity_records = []
for _, zone_row in zone_df_attrs.iterrows():
    zone_id = zone_row["zone_id"]
    area_km2 = zone_row["area_km2"]
    mask = (final_labeled_clean == zone_id)

    for m in range(12):
        cc_density = float(np.nanmean(cc_monthly_tlu_km2[m][mask]))
        if np.isnan(cc_density):
            cc_density = 0.0
        capacity_tlu = cc_density * area_km2

        zone_capacity_records.append({
            "zone_id": zone_id, "month": month_names[m], "month_num": m + 1,
            "cc_tlu_per_km2": round(cc_density, 3), "capacity_tlu": round(capacity_tlu, 1),
        })

zone_month_capacity_df = pd.DataFrame(zone_capacity_records)
print(f"✅ Built table: {len(zone_month_capacity_df)} rows ({len(zone_df_attrs)} zones x 12 months)")

zone_month_capacity_path = os.path.join(CAPACITY_DIR, "zone_month_capacity.csv")
zone_month_capacity_df.to_csv(zone_month_capacity_path, index=False)
print(f"✅ Saved: {os.path.basename(zone_month_capacity_path)}")

# ============================================================
# STEP 4 — TLU conversion helper
# ============================================================

def cattle_to_tlu(n_cattle, livestock_type):
    if livestock_type not in TLU_CONVERSION:
        raise ValueError(f"Unknown livestock_type '{livestock_type}'. Valid: {list(TLU_CONVERSION.keys())}")
    return n_cattle * TLU_CONVERSION[livestock_type]

# ============================================================
# STEP 5 — Core Query Function 1: query_zone_capacity
# ============================================================

def query_zone_capacity(zone_id, month, n_cattle, livestock_type):
    """Can zone `zone_id` support `n_cattle` head of `livestock_type` in `month`?"""
    month_name = month_names[month - 1] if isinstance(month, int) else month

    row = zone_month_capacity_df[
        (zone_month_capacity_df["zone_id"] == zone_id) &
        (zone_month_capacity_df["month"] == month_name)
    ]
    if len(row) == 0:
        return {"error": f"Zone {zone_id} or month '{month_name}' not found."}

    available_capacity_tlu = float(row["capacity_tlu"].values[0])
    requested_tlu = cattle_to_tlu(n_cattle, livestock_type)
    can_sustain = requested_tlu <= available_capacity_tlu
    surplus_deficit = available_capacity_tlu - requested_tlu
    utilization_pct = (requested_tlu / available_capacity_tlu * 100) if available_capacity_tlu > 0 else None

    return {
        "zone_id": zone_id, "month": month_name, "requested_cattle": n_cattle,
        "livestock_type": livestock_type, "requested_tlu": round(requested_tlu, 2),
        "available_capacity_tlu": round(available_capacity_tlu, 2), "can_sustain": bool(can_sustain),
        "surplus_deficit_tlu": round(surplus_deficit, 2),
        "utilization_pct": round(utilization_pct, 1) if utilization_pct is not None else None,
    }

# ============================================================
# STEP 6 — Core Query Function 2: query_best_zones
# ============================================================

def query_best_zones(month, n_cattle, livestock_type="cattle_adult_zebu", top_n=10):
    """Which zones can support `n_cattle` in `month`, ranked by spare capacity?"""
    month_name = month_names[month - 1] if isinstance(month, int) else month
    requested_tlu = cattle_to_tlu(n_cattle, livestock_type)

    month_data = zone_month_capacity_df[zone_month_capacity_df["month"] == month_name].copy()
    month_data["requested_tlu"] = requested_tlu
    month_data["can_sustain"] = month_data["capacity_tlu"] >= requested_tlu
    month_data["surplus_tlu"] = month_data["capacity_tlu"] - requested_tlu

    qualifying = month_data[month_data["can_sustain"]].sort_values("surplus_tlu", ascending=False)
    result = qualifying.merge(
        zone_df_attrs[["zone_id", "quality", "centroid_lat", "centroid_lon", "area_km2"]],
        on="zone_id", how="left"
    )
    return result.head(top_n).reset_index(drop=True)

# ============================================================
# STEP 7 — Core Query Function 3: seasonal carrying capacity calendar
# ============================================================

def get_seasonal_calendar(zone_id=None):
    """Best/worst month per zone (or all zones if zone_id=None)."""
    if zone_id is not None:
        subset = zone_month_capacity_df[zone_month_capacity_df["zone_id"] == zone_id]
        best_row = subset.loc[subset["capacity_tlu"].idxmax()]
        worst_row = subset.loc[subset["capacity_tlu"].idxmin()]
        return {
            "zone_id": zone_id, "best_month": best_row["month"],
            "best_month_capacity_tlu": best_row["capacity_tlu"],
            "worst_month": worst_row["month"], "worst_month_capacity_tlu": worst_row["capacity_tlu"],
        }
    else:
        return pd.DataFrame([get_seasonal_calendar(zid) for zid in zone_df_attrs["zone_id"].unique()])

# ============================================================
# TEST THE QUERY ENGINE
# ============================================================

print("\n" + "=" * 60)
print("TESTING QUERY ENGINE")
print("=" * 60)

test_zone = int(zone_df_attrs.iloc[0]["zone_id"])

print(f"\n🔍 Test 1: query_zone_capacity(zone_id={test_zone}, month='August', n_cattle=50, livestock_type='cattle_adult_zebu')")
result1 = query_zone_capacity(test_zone, "August", 50, "cattle_adult_zebu")
for k, v in result1.items():
    print(f"   {k}: {v}")

print(f"\n🔍 Test 2: query_best_zones(month='August', n_cattle=50, top_n=5)")
best_zones_result = query_best_zones("August", 50, "cattle_adult_zebu", top_n=5)
print(best_zones_result[["zone_id", "quality", "area_km2", "capacity_tlu", "surplus_tlu"]].to_string(index=False))

print(f"\n🔍 Test 3: get_seasonal_calendar(zone_id={test_zone})")
calendar_result = get_seasonal_calendar(test_zone)
for k, v in calendar_result.items():
    print(f"   {k}: {v}")

print(f"\n🔍 Test 4: get_seasonal_calendar() — all zones, first 5")
full_calendar = get_seasonal_calendar()
print(full_calendar.head(5).to_string(index=False))

full_calendar_path = os.path.join(CAPACITY_DIR, "seasonal_calendar_all_zones.csv")
full_calendar.to_csv(full_calendar_path, index=False)
print(f"\n✅ Saved: {os.path.basename(full_calendar_path)}")

# ============================================================
# VISUAL — capacity heatmap: zones (rows) x months (columns)
# ============================================================

pivot_capacity = zone_month_capacity_df.pivot(index="zone_id", columns="month", values="capacity_tlu")
pivot_capacity = pivot_capacity[month_names]

fig, ax = plt.subplots(1, 1, figsize=(10, 16))
im = ax.imshow(pivot_capacity.values, cmap="YlGn", aspect="auto")
ax.set_xticks(range(12)); ax.set_xticklabels(month_names, rotation=45)
ax.set_yticks(range(0, len(pivot_capacity), 5)); ax.set_yticklabels(pivot_capacity.index[::5])
ax.set_xlabel("Month"); ax.set_ylabel("Zone ID")
ax.set_title("Carrying Capacity Calendar — All Zones x All Months (TLU)")
plt.colorbar(im, ax=ax, label="Capacity (TLU)", fraction=0.03)
plt.tight_layout()
plt.show()

total_capacity_by_month = zone_month_capacity_df.groupby("month")["capacity_tlu"].sum().reindex(month_names)

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(month_names, total_capacity_by_month.values, marker="o", color="#2ca02c", linewidth=2)
ax.set_ylabel("Total regional capacity (TLU)")
ax.set_title("Total Adamawa Carrying Capacity by Month (sum of all zones)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📊 Regional capacity range: {total_capacity_by_month.min():.0f} to {total_capacity_by_month.max():.0f} TLU")
print(f"   Peak month: {total_capacity_by_month.idxmax()}, Lowest month: {total_capacity_by_month.idxmin()}")

print("\n" + "=" * 60)
print("✅ PHASE 5 CELL 1 COMPLETE — Query engine built, tested, and self-contained")
print("=" * 60)

###Export parameters.json for Django Backend

In [ ]:
# ============================================================
# PHASE 5 — CELL 2
# Export parameters.json — everything Django needs at startup
# (constants, TLU factors, grid metadata, file paths)
# ============================================================

import json
import os

# --- Session restore check ---
assert 'CC_CONSTANTS' in dir(), "⚠️ Run Phase 5 Cell 1 first."

parameters = {
    "project": "Dynamic Pasture Suitability Mapping — Adamawa, Cameroon",
    "study_area": "Adamaoua, Cameroon",
    "data_years": f"{START_YEAR}-{END_YEAR}" if 'START_YEAR' in dir() else "2016-2025",

    # --- Grid / raster metadata ---
    "grid": {
        "crs": GRID_CRS,
        "width": GRID_WIDTH,
        "height": GRID_HEIGHT,
        "transform": GRID_TRANSFORM,
        "pixel_size_m": 1000,
    },

    # --- Carrying capacity constants ---
    "carrying_capacity_constants": CC_CONSTANTS,

    # --- Livestock TLU conversion factors ---
    "tlu_conversion_factors": TLU_CONVERSION,

    # --- Suitability weights (from Phase 4) ---
    "suitability_weights": {
        "biomass": 0.35,
        "ndvi": 0.20,
        "rainfall": 0.20,
        "water": 0.15,
        "slope": 0.05,
        "population": 0.05,
    },

    # --- Suitability classification thresholds (Phase 4) ---
    "suitability_thresholds": {
        "high": 65,
        "moderate": 45,
        "low": 25,
    },

    # --- Biomass suitability classification thresholds (Phase 3) ---
    "biomass_thresholds_kg_dm_ha": {
        "high": 1000,
        "moderate": 500,
        "low": 187,
        "poor": 50,
    },

    # --- Biomass regression formula ---
    "biomass_formula": "AGB (kg DM/ha) = 3500 * NDVI - 250, clipped to min 0",

    # --- Zone delineation parameters used (Phase 4 Cell 3) ---
    "zone_delineation_params": {
        "smoothing_size": 5,
        "score_threshold": 35,
        "min_area_km2": 5,
        "connectivity": 4,
        "dilation_applied": False,
    },

    # --- Exclusion mask criteria (Phase 4 Cell 1) ---
    "exclusion_criteria": {
        "cropland_landcover_class": 40,
        "builtup_landcover_class": 50,
        "forest_landcover_class": 10,
        "population_threshold_per_km2": 300,
        "slope_threshold_degrees": 25,
    },

    # --- Zone count ---
    "total_zones": int(len(zone_df_attrs)),

    # --- File paths relative to project root (Django will resolve these) ---
    "data_paths": {
        "manifest": "manifest.json",
        "zone_attributes_csv": "suitability/grazing_zones_attributes.csv",
        "zone_raster": "suitability/grazing_zones_raster.tif",
        "zone_geopackage": "suitability/grazing_zones.gpkg",
        "zone_month_capacity_csv": "capacity/zone_month_capacity.csv",
        "seasonal_calendar_csv": "capacity/seasonal_calendar_all_zones.csv",
        "carrying_capacity_raster": "capacity/carrying_capacity_monthly_TLU_per_km2.tif",
        "suitability_score_raster": "suitability/suitability_score_monthly_0-100.tif",
        "biomass_monthly_mean_raster": "biomass/biomass_monthly_mean_2016-2025.tif",
        "exclusion_mask_raster": "suitability/exclusion_mask.tif",
    },
}

parameters_path = os.path.join(PROCESSED_DIR, "parameters.json")
with open(parameters_path, "w") as f:
    json.dump(parameters, f, indent=2)

print(f"✅ Saved: {parameters_path}")
print("\n📋 parameters.json contents preview:")
print(json.dumps(parameters, indent=2)[:2000])
print("...\n")

# --- Sanity check: re-load it back to confirm it's valid JSON and complete ---
with open(parameters_path, "r") as f:
    reloaded = json.load(f)

expected_top_keys = [
    "grid", "carrying_capacity_constants", "tlu_conversion_factors",
    "suitability_weights", "suitability_thresholds", "biomass_thresholds_kg_dm_ha",
    "zone_delineation_params", "exclusion_criteria", "total_zones", "data_paths"
]
missing_keys = [k for k in expected_top_keys if k not in reloaded]

if missing_keys:
    print(f"⚠️ Missing keys after reload: {missing_keys}")
else:
    print("✅ All expected keys present and JSON is valid.")

print(f"\n📊 Total zones recorded: {reloaded['total_zones']}")
print(f"📊 Grid: {reloaded['grid']['width']} x {reloaded['grid']['height']} @ {reloaded['grid']['pixel_size_m']}m, {reloaded['grid']['crs']}")

print("\n" + "=" * 60)
print("✅ PHASE 5 COMPLETE")
print(f"   All carrying capacity outputs saved to: {CAPACITY_DIR}/")
print(f"   Django-ready parameters file: {parameters_path}")
print("=" * 60)

###query_grazing_duration() — Simple Depletion Estimator

In [ ]:
# ============================================================
# PHASE 5 — CELL 3 (FINAL, SELF-CONTAINED)
# query_grazing_duration() — capped at monthly horizon
# ============================================================

import os
import numpy as np
import pandas as pd
import rasterio

# ============================================================
# SESSION RESTORE
# ============================================================

DRIVE_FOLDER = "pasture-mapping-adamawa"
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROCESSED_DIR = "/content/data/processed/final"
SUITABILITY_DIR = os.path.join(PROCESSED_DIR, "suitability")

month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
FULL_MONTH_NAMES = ["January","February","March","April","May","June",
                     "July","August","September","October","November","December"]

MONTH_LOOKUP = {}
for i, (abbr, full) in enumerate(zip(month_names, FULL_MONTH_NAMES), start=1):
    MONTH_LOOKUP[abbr.lower()] = abbr
    MONTH_LOOKUP[full.lower()] = abbr
    MONTH_LOOKUP[str(i)] = abbr

def normalize_month(month):
    if isinstance(month, int):
        return month_names[month - 1]
    key = str(month).strip().lower()
    if key not in MONTH_LOOKUP:
        raise ValueError(f"Unrecognized month '{month}'. Use 1-12, 'Aug', or 'August'.")
    return MONTH_LOOKUP[key]

# --- Load zone attributes ---
zone_csv_path = os.path.join(SUITABILITY_DIR, "grazing_zones_attributes.csv")
assert os.path.exists(zone_csv_path), "⚠️ grazing_zones_attributes.csv not found. Run Phase 4 Cell 3 first."
zone_df_attrs = pd.read_csv(zone_csv_path)

# --- Load zone raster ---
zone_raster_path = os.path.join(SUITABILITY_DIR, "grazing_zones_raster.tif")
assert os.path.exists(zone_raster_path), "⚠️ grazing_zones_raster.tif not found. Run Phase 4 Cell 3 first."
with rasterio.open(zone_raster_path) as src:
    zone_raster = src.read(1)

# --- Load biomass monthly mean ---
biomass_path = os.path.join(PROCESSED_DIR, "biomass", "biomass_monthly_mean_2016-2025.tif")
assert os.path.exists(biomass_path), "⚠️ biomass_monthly_mean_2016-2025.tif not found. Run Phase 3 Cell 3 first."
with rasterio.open(biomass_path) as src:
    biomass_monthly_mean = np.stack([
        np.where(src.read(m + 1) == src.nodata, np.nan, src.read(m + 1))
        for m in range(12)
    ], axis=0)

print(f"✅ Loaded {len(zone_df_attrs)} zones, biomass shape {biomass_monthly_mean.shape}")

# --- Constants ---
CC_CONSTANTS = {
    "DAILY_INTAKE_PER_TLU_KG": 6.25,
    "UTILIZATION_RATE": 0.40,
    "PIXEL_AREA_HA": 100,
}

TLU_CONVERSION = {
    "cattle_adult_zebu": 1.00, "cattle_young": 0.50, "camel": 1.10,
    "horse": 0.80, "donkey": 0.50, "sheep_goat": 0.10,
}

def cattle_to_tlu(n_cattle, livestock_type):
    if livestock_type not in TLU_CONVERSION:
        raise ValueError(f"Unknown livestock_type. Valid: {list(TLU_CONVERSION.keys())}")
    return n_cattle * TLU_CONVERSION[livestock_type]

DAYS_PER_MONTH = 30

# ============================================================
# query_grazing_duration — capped at monthly horizon
# ============================================================

def query_grazing_duration(zone_id, month, n_cattle, livestock_type="cattle_adult_zebu"):
    month_name = normalize_month(month)
    month_idx = month_names.index(month_name)

    zone_row = zone_df_attrs[zone_df_attrs["zone_id"] == zone_id]
    if len(zone_row) == 0:
        return {"error": f"Zone {zone_id} not found."}
    area_km2 = float(zone_row.iloc[0]["area_km2"])

    zone_mask = (zone_raster == zone_id)
    mean_biomass_kg_dm_ha = float(np.nanmean(biomass_monthly_mean[month_idx][zone_mask]))

    if np.isnan(mean_biomass_kg_dm_ha) or mean_biomass_kg_dm_ha <= 0:
        return {"error": f"No valid biomass data for zone {zone_id} in {month_name}."}

    area_ha = area_km2 * CC_CONSTANTS["PIXEL_AREA_HA"]
    total_biomass_kg = mean_biomass_kg_dm_ha * area_ha
    sustainable_biomass_kg = total_biomass_kg * CC_CONSTANTS["UTILIZATION_RATE"]

    requested_tlu = cattle_to_tlu(n_cattle, livestock_type)
    daily_consumption_kg = requested_tlu * CC_CONSTANTS["DAILY_INTAKE_PER_TLU_KG"]

    if daily_consumption_kg <= 0:
        return {"error": "n_cattle must be greater than 0."}

    raw_days_to_ceiling = sustainable_biomass_kg / daily_consumption_kg
    monthly_utilization_pct = (daily_consumption_kg * DAYS_PER_MONTH / sustainable_biomass_kg) * 100

    if raw_days_to_ceiling >= DAYS_PER_MONTH:
        status = "within_monthly_capacity"
        practical_days = DAYS_PER_MONTH
        message = (f"This herd uses only {monthly_utilization_pct:.1f}% of the zone's sustainable "
                   f"monthly forage. It can graze the full month without hitting the utilization "
                   f"ceiling — depletion isn't the binding constraint here.")
    else:
        status = "depletes_within_month"
        practical_days = raw_days_to_ceiling
        message = (f"This herd will reach the sustainable utilization ceiling in "
                   f"{practical_days:.1f} days — should move to a new zone or allow rest before then.")

    return {
        "zone_id": zone_id, "month": month_name, "n_cattle": n_cattle,
        "livestock_type": livestock_type, "requested_tlu": round(requested_tlu, 2),
        "mean_biomass_kg_dm_ha": round(mean_biomass_kg_dm_ha, 1),
        "monthly_utilization_pct": round(monthly_utilization_pct, 2),
        "status": status,
        "practical_grazing_days": round(practical_days, 1),
        "raw_depletion_days_no_regrowth": round(raw_days_to_ceiling, 1),
        "message": message,
    }

# --- Test ---
print("🔍 Zone 15, August, 50 cattle:")
r = query_grazing_duration(15, "August", 50, "cattle_adult_zebu")
for k, v in r.items():
    print(f"   {k}: {v}")

print("\n📊 Sanity sweep:")
for n in [50, 500, 2000, 10000, 30000]:
    r = query_grazing_duration(15, "August", n, "cattle_adult_zebu")
    print(f"   {n:>6} cattle -> {r['status']:<24} practical_days={r['practical_grazing_days']:>5} "
          f"utilization={r['monthly_utilization_pct']:>6.1f}%")

print("\n✅ PHASE 5 CELL 3 COMPLETE")

##Phase 6

### Download OSM Roads + Clip to Study Area

In [ ]:
# ============================================================
# PHASE 6 — CELL 1
# Download Geofabrik Cameroon OSM Roads + Clip to Adamawa
# WITH VISUAL QA
# ============================================================

import os
import json
import glob
import zipfile
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

# ============================================================
# SESSION RESTORE
# ============================================================

DRIVE_FOLDER = "pasture-mapping-adamawa"
TARGET_CRS = "EPSG:32633"

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROCESSED_DIR = "/content/data/processed/final"
SUITABILITY_DIR = os.path.join(PROCESSED_DIR, "suitability")
ROUTING_DIR = os.path.join(PROCESSED_DIR, "routing")
os.makedirs(ROUTING_DIR, exist_ok=True)

RAW_OSM_DIR = "/content/data/raw/osm"
os.makedirs(RAW_OSM_DIR, exist_ok=True)

manifest_path = os.path.join(PROCESSED_DIR, "manifest.json")
with open(manifest_path, "r") as f:
    manifest = json.load(f)

GRID_WIDTH = manifest["grid_width"]
GRID_HEIGHT = manifest["grid_height"]
GRID_TRANSFORM = manifest["grid_transform"]
GRID_CRS = manifest["crs"]
transform_obj = rasterio.Affine(*GRID_TRANSFORM[:6])

# --- Load study area boundary (saved in Phase 2) ---
boundary_path = os.path.join(f"/content/drive/MyDrive/{DRIVE_FOLDER}", "adamaoua_boundary.geojson")
assert os.path.exists(boundary_path), "⚠️ adamaoua_boundary.geojson not found in Drive."
boundary_gdf = gpd.read_file(boundary_path)
boundary_gdf = boundary_gdf.set_crs("EPSG:4326", allow_override=True)
boundary_gdf_utm = boundary_gdf.to_crs(GRID_CRS)
print(f"✅ Loaded study area boundary")

# ============================================================
# STEP 1 — Download Geofabrik Cameroon extract
# ============================================================

GEOFABRIK_URL = "https://download.geofabrik.de/africa/cameroon-latest-free.shp.zip"
zip_path = os.path.join(RAW_OSM_DIR, "cameroon-latest-free.shp.zip")

if not os.path.exists(zip_path):
    print(f"Downloading {GEOFABRIK_URL} ...")
    print("(This is a country-wide OSM extract — may take a few minutes and be several hundred MB)")
    response = requests.get(GEOFABRIK_URL, stream=True, timeout=300)
    response.raise_for_status()
    total_size = int(response.headers.get("content-length", 0))
    downloaded = 0
    with open(zip_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
    print(f"✅ Downloaded: {downloaded / 1e6:.1f} MB")
else:
    print(f"✅ Already downloaded: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

# ============================================================
# STEP 2 — Extract the shapefile archive
# ============================================================

extract_dir = os.path.join(RAW_OSM_DIR, "cameroon_extracted")
os.makedirs(extract_dir, exist_ok=True)

if not os.listdir(extract_dir):
    print("\nExtracting shapefile archive...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    print(f"✅ Extracted to {extract_dir}")
else:
    print(f"✅ Already extracted: {extract_dir}")

# --- List extracted files to find the roads shapefile ---
all_shp_files = glob.glob(os.path.join(extract_dir, "**", "*.shp"), recursive=True)
print(f"\n📂 Shapefiles found in extract:")
for f in all_shp_files:
    print(f"   {os.path.basename(f)}")

# Geofabrik naming convention: gis_osm_roads_free_1.shp
roads_shp_candidates = [f for f in all_shp_files if "roads" in os.path.basename(f).lower()]
assert len(roads_shp_candidates) > 0, \
    "⚠️ No roads shapefile found in extract. Check Geofabrik archive structure — " \
    "it may have changed naming convention."
roads_shp_path = roads_shp_candidates[0]
print(f"\n✅ Roads shapefile identified: {os.path.basename(roads_shp_path)}")

# ============================================================
# STEP 3 — Load roads, reproject, clip to study area
# ============================================================

print("\nLoading roads shapefile (this is Cameroon-wide, may take a moment)...")
roads_gdf = gpd.read_file(roads_shp_path)
print(f"✅ Loaded {len(roads_gdf)} road segments (Cameroon-wide)")
print(f"   Columns: {list(roads_gdf.columns)}")
print(f"   CRS: {roads_gdf.crs}")

# Check highway type field (Geofabrik typically uses 'fclass')
highway_col = "fclass" if "fclass" in roads_gdf.columns else (
    "highway" if "highway" in roads_gdf.columns else None
)
assert highway_col is not None, "⚠️ No 'fclass' or 'highway' column found — inspect roads_gdf.columns manually."
print(f"   Highway type column: '{highway_col}'")
print(f"   Unique highway types (Cameroon-wide): {sorted(roads_gdf[highway_col].unique())}")

# --- Reproject to study CRS ---
roads_gdf_utm = roads_gdf.to_crs(GRID_CRS)

# --- Clip to study area boundary ---
print("\nClipping roads to Adamawa study area...")
roads_clipped = gpd.clip(roads_gdf_utm, boundary_gdf_utm)
print(f"✅ Clipped to {len(roads_clipped)} road segments within Adamawa")

if len(roads_clipped) == 0:
    print("⚠️ WARNING: zero road segments found within the study area. "
          "Check CRS alignment or boundary geometry validity.")

# --- Highway type breakdown within study area ---
print(f"\n📊 Highway type breakdown (Adamawa only):")
type_counts = roads_clipped[highway_col].value_counts()
print(type_counts.to_string())

# --- Save clipped roads ---
roads_clipped_path = os.path.join(ROUTING_DIR, "roads_adamawa.gpkg")
roads_clipped.to_file(roads_clipped_path, layer="roads", driver="GPKG")
print(f"\n✅ Saved: {roads_clipped_path}")

# ============================================================
# VISUAL QA
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(11, 10))
boundary_gdf_utm.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1.5, zorder=1)

# Color by highway type for visual distinction
type_colors = {
    "track": "#8b4513", "unclassified": "#808080", "residential": "#4169e1",
    "primary": "#dc143c", "secondary": "#ff8c00", "tertiary": "#ffd700",
    "path": "#228b22", "footway": "#32cd32", "trunk": "#8b0000",
}
for htype in roads_clipped[highway_col].unique():
    subset = roads_clipped[roads_clipped[highway_col] == htype]
    color = type_colors.get(htype, "#999999")
    subset.plot(ax=ax, color=color, linewidth=0.7, label=htype, zorder=2)

ax.set_title(f"OSM Roads — Adamawa Region\n{len(roads_clipped)} segments, "
             f"{len(roads_clipped[highway_col].unique())} highway types")
ax.legend(loc="upper left", fontsize=8, ncol=2)
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.tight_layout()
plt.show()

# --- Road length by type ---
roads_clipped["length_km"] = roads_clipped.geometry.length / 1000
length_by_type = roads_clipped.groupby(highway_col)["length_km"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
length_by_type.plot(kind="barh", ax=ax, color="#4169e1")
ax.set_xlabel("Total length (km)")
ax.set_title("Road Network Length by Highway Type — Adamawa")
plt.tight_layout()
plt.show()

print(f"\n📊 Total road network length in Adamawa: {roads_clipped['length_km'].sum():.0f} km")

print("\n" + "=" * 60)
print("✅ PHASE 6 CELL 1 COMPLETE")
print(f"   Roads clipped and saved: {roads_clipped_path}")
print(f"   {len(roads_clipped)} segments, {roads_clipped['length_km'].sum():.0f} km total")
print("=" * 60)

###Cost Surface Construction

In [ ]:
# ============================================================
# PHASE 6 — CELL 2
# Cost Surface Construction for Transhumance Routing
# Combines: inverse suitability, slope cost, water distance penalty,
# road/track bonus (by OSM highway type), hard barriers (cropland/settlement)
# WITH VISUAL QA
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ============================================================
# SESSION RESTORE
# ============================================================

DRIVE_FOLDER = "pasture-mapping-adamawa"
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROCESSED_DIR = "/content/data/processed/final"
SUITABILITY_DIR = os.path.join(PROCESSED_DIR, "suitability")
ROUTING_DIR = os.path.join(PROCESSED_DIR, "routing")
os.makedirs(ROUTING_DIR, exist_ok=True)

manifest_path = os.path.join(PROCESSED_DIR, "manifest.json")
with open(manifest_path, "r") as f:
    manifest = json.load(f)

GRID_WIDTH = manifest["grid_width"]
GRID_HEIGHT = manifest["grid_height"]
GRID_TRANSFORM = manifest["grid_transform"]
GRID_CRS = manifest["crs"]
transform_obj = rasterio.Affine(*GRID_TRANSFORM[:6])

ref_meta = {
    "driver": "GTiff", "height": GRID_HEIGHT, "width": GRID_WIDTH,
    "count": 1, "dtype": "float32", "crs": GRID_CRS,
    "transform": transform_obj,
    "nodata": -9999.0, "tiled": True, "blockxsize": 256, "blockysize": 256, "compress": "lzw"
}

def load_raster(path, band=1):
    with rasterio.open(path) as src:
        data = src.read(band).astype("float32")
        nodata = src.nodata
        data = np.where(data == nodata, np.nan, data)
    return data

# --- Load required layers ---
slope = load_raster(os.path.join(PROCESSED_DIR, "static", "slope.tif"))
dist_water = load_raster(os.path.join(PROCESSED_DIR, "static", "distance_to_water.tif"))
landcover = load_raster(os.path.join(PROCESSED_DIR, "static", "landcover.tif"))

# Annual mean suitability score (from Phase 4)
with rasterio.open(os.path.join(SUITABILITY_DIR, "suitability_score_monthly_0-100.tif")) as src:
    suit_score_monthly = np.stack([
        np.where(src.read(m + 1) == src.nodata, np.nan, src.read(m + 1)) for m in range(12)
    ], axis=0)
annual_mean_score = np.nanmean(suit_score_monthly, axis=0)

# Roads (from Phase 6 Cell 1)
roads_path = os.path.join(ROUTING_DIR, "roads_adamawa.gpkg")
assert os.path.exists(roads_path), "⚠️ roads_adamawa.gpkg not found. Run Phase 6 Cell 1 first."
roads_gdf = gpd.read_file(roads_path, layer="roads")
highway_col = "fclass" if "fclass" in roads_gdf.columns else "highway"
print(f"✅ Loaded {len(roads_gdf)} road segments")

valid_pixels = ~np.isnan(annual_mean_score)
print(f"✅ All static layers loaded")

# ============================================================
# STEP 1 — Rasterize roads into a "road quality" grid
# Lower value = better road = lower cost. Assign per your spec:
#   Track/unpaved: very low cost      -> 0.05
#   Primary/secondary: low cost       -> 0.10
#   Path/footway: low cost            -> 0.15
#   (no road / cross-country)         -> handled separately as "no bonus"
# When multiple road types overlap a pixel, the BEST (lowest cost) wins.
# ============================================================

ROAD_COST_BY_TYPE = {
    "track": 0.05,
    "trunk": 0.08,
    "primary": 0.10,
    "secondary": 0.10,
    "tertiary": 0.12,
    "unclassified": 0.15,
    "residential": 0.15,
    "path": 0.15,
    "footway": 0.15,
    "bridleway": 0.15,
}
DEFAULT_ROAD_COST = 0.20  # any other/unrecognized road type still gets a modest bonus

print("\nRasterizing roads by cost priority (best type wins per pixel)...")

# Start with "no road" value = 1.0 (no bonus at all)
road_cost_grid = np.ones((GRID_HEIGHT, GRID_WIDTH), dtype="float32")

# Rasterize from WORST to BEST cost so better roads overwrite worse ones
# where segments overlap on the same pixel
sorted_types_by_cost_desc = sorted(
    roads_gdf[highway_col].unique(),
    key=lambda t: -ROAD_COST_BY_TYPE.get(t, DEFAULT_ROAD_COST)
)

for htype in sorted_types_by_cost_desc:
    subset = roads_gdf[roads_gdf[highway_col] == htype]
    if len(subset) == 0:
        continue
    cost_val = ROAD_COST_BY_TYPE.get(htype, DEFAULT_ROAD_COST)
    burned = rasterize(
        [(geom, cost_val) for geom in subset.geometry],
        out_shape=(GRID_HEIGHT, GRID_WIDTH),
        transform=transform_obj,
        fill=np.nan,
        all_touched=True,  # ensures thin line features aren't missed at 1km resolution
        dtype="float32"
    )
    mask = ~np.isnan(burned)
    road_cost_grid[mask] = burned[mask]

print(f"✅ Road cost grid built. Pixels with any road: "
      f"{(road_cost_grid < 1.0).sum()} ({(road_cost_grid < 1.0).sum() / valid_pixels.sum() * 100:.1f}% of study area)")

# ============================================================
# STEP 2 — Normalize continuous cost components (0-1, higher = worse)
# ============================================================

def normalize_01(array, invert=False, p_low=1, p_high=99):
    valid = array[~np.isnan(array)]
    lo, hi = np.percentile(valid, [p_low, p_high])
    clipped = np.clip(array, lo, hi)
    norm = (clipped - lo) / (hi - lo + 1e-9)
    norm = np.where(np.isnan(array), np.nan, norm)
    if invert:
        norm = np.where(np.isnan(norm), np.nan, 1 - norm)
    return norm

# Inverse suitability: LOW suitability = HIGH cost (invert=True means high raw -> low cost,
# so we do NOT invert here — high score should give LOW cost, so we invert)
inv_suitability_cost = normalize_01(annual_mean_score, invert=True)  # high score -> low cost

# Slope cost: steep = high cost (no invert — high slope = high cost directly)
slope_cost = normalize_01(slope, invert=False)

# Water distance penalty: far = high cost (no invert — high distance = high cost directly)
water_dist_cost = normalize_01(dist_water, invert=False)

print(f"\n✅ Cost components normalized:")
print(f"   Inverse suitability cost range: {np.nanmin(inv_suitability_cost):.3f}-{np.nanmax(inv_suitability_cost):.3f}")
print(f"   Slope cost range: {np.nanmin(slope_cost):.3f}-{np.nanmax(slope_cost):.3f}")
print(f"   Water distance cost range: {np.nanmin(water_dist_cost):.3f}-{np.nanmax(water_dist_cost):.3f}")

# ============================================================
# STEP 3 — Combine into weighted cost surface
# Weights here are a routing-specific weighting (separate from Phase 4's
# suitability weights) — reflecting what matters for WHERE TO WALK,
# not where to graze. Suitability and roads dominate; slope and water
# are secondary modifiers. This split isn't explicitly pinned in your
# spec, so flag if you want different routing weights.
# ============================================================

ROUTING_WEIGHTS = {
    "inv_suitability": 0.30,
    "slope": 0.20,
    "water_distance": 0.15,
    "road_bonus": 0.35,
}
assert abs(sum(ROUTING_WEIGHTS.values()) - 1.0) < 1e-6

combined_cost = (
    ROUTING_WEIGHTS["inv_suitability"] * np.nan_to_num(inv_suitability_cost, nan=1.0) +
    ROUTING_WEIGHTS["slope"]           * np.nan_to_num(slope_cost, nan=1.0) +
    ROUTING_WEIGHTS["water_distance"]  * np.nan_to_num(water_dist_cost, nan=1.0) +
    ROUTING_WEIGHTS["road_bonus"]      * road_cost_grid
)

print(f"\n✅ Combined cost surface (before hard barriers): "
      f"range {np.nanmin(combined_cost):.3f}-{np.nanmax(combined_cost):.3f}")

# ============================================================
# STEP 4 — Hard barriers: cropland, settlement = IMPASSABLE
# Using a large finite value (not np.inf) so NetworkX graph weights
# stay numerically stable — effectively blocks the path unless
# there is truly no alternative route.
# ============================================================

IMPASSABLE_COST = 1000.0  # large finite value, not inf (keeps Dijkstra numerically stable)

cropland_mask = (landcover == 40)
builtup_mask = (landcover == 50)
hard_barrier_mask = cropland_mask | builtup_mask

final_cost_surface = np.where(hard_barrier_mask, IMPASSABLE_COST, combined_cost)
final_cost_surface = np.where(~valid_pixels, np.nan, final_cost_surface)

n_barrier_pixels = (hard_barrier_mask & valid_pixels).sum()
print(f"\n✅ Hard barriers applied: {n_barrier_pixels} pixels "
      f"({n_barrier_pixels / valid_pixels.sum() * 100:.1f}% of study area) set to cost={IMPASSABLE_COST}")

# --- Save cost surface ---
cost_meta = ref_meta.copy()
cost_path = os.path.join(ROUTING_DIR, "cost_surface.tif")
band_out = np.where(np.isnan(final_cost_surface), -9999.0, final_cost_surface).astype("float32")
with rasterio.open(cost_path, "w", **cost_meta) as dst:
    dst.write(band_out, 1)
print(f"✅ Saved: {cost_path}")

# Also save the road cost grid separately (useful for QA / Phase 6 Cell 3 debugging)
road_cost_path = os.path.join(ROUTING_DIR, "road_cost_grid.tif")
with rasterio.open(road_cost_path, "w", **cost_meta) as dst:
    dst.write(np.where(~valid_pixels, -9999.0, road_cost_grid).astype("float32"), 1)
print(f"✅ Saved: {road_cost_path}")

# ============================================================
# VISUAL QA
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

im0 = axes[0, 0].imshow(inv_suitability_cost, cmap="Reds", vmin=0, vmax=1)
axes[0, 0].set_title("Inverse Suitability Cost\n(red = low suitability = high cost)")
plt.colorbar(im0, ax=axes[0, 0], fraction=0.046)

im1 = axes[0, 1].imshow(road_cost_grid, cmap="Greens_r", vmin=0, vmax=1)
axes[0, 1].set_title("Road Cost Grid\n(dark green = on a good road = low cost)")
plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)

# Cost surface clipped for display (exclude the 1000 barrier value so color scale stays useful)
display_cost = np.where(final_cost_surface >= IMPASSABLE_COST, np.nan, final_cost_surface)
im2 = axes[1, 0].imshow(display_cost, cmap="RdYlGn_r", vmin=0, vmax=1)
axes[1, 0].set_title("Combined Cost Surface (excl. hard barriers)\n(red = high cost/avoid, green = low cost/prefer)")
plt.colorbar(im2, ax=axes[1, 0], fraction=0.046)

barrier_display = np.where(hard_barrier_mask & valid_pixels, 1, np.where(valid_pixels, 0, np.nan))
im3 = axes[1, 1].imshow(barrier_display, cmap="Reds", vmin=0, vmax=1)
axes[1, 1].set_title(f"Hard Barriers (cropland + built-up)\n{n_barrier_pixels} pixels impassable")
plt.colorbar(im3, ax=axes[1, 1], fraction=0.046)

plt.tight_layout()
plt.show()

# --- Cost distribution histogram (excluding barriers, for visibility) ---
fig, ax = plt.subplots(1, 1, figsize=(9, 5))
valid_costs = combined_cost[valid_pixels & ~hard_barrier_mask]
ax.hist(valid_costs, bins=50, color="#4169e1", edgecolor="black", alpha=0.7)
ax.set_xlabel("Combined cost (excl. hard barriers)")
ax.set_ylabel("Pixel count")
ax.set_title("Cost Surface Distribution — Traversable Area")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("✅ PHASE 6 CELL 2 COMPLETE")
print(f"   Cost surface saved: {cost_path}")
print(f"   Traversable pixels: {(valid_pixels & ~hard_barrier_mask).sum()}")
print(f"   Impassable pixels: {n_barrier_pixels}")
print("=" * 60)

###NetworkX Graph + Dijkstra Routing + GeoJSON Export

In [ ]:
# ============================================================
# PHASE 6 — CELL 3
# NetworkX Graph (8-connectivity) + Dijkstra Routing
# find_optimal_route(start_lat, start_lon, n_cattle, month, livestock_type)
# WITH VISUAL QA + GEOJSON EXPORT
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import networkx as nx
from pyproj import Transformer
from shapely.geometry import LineString, Point
import matplotlib.pyplot as plt

# ============================================================
# SESSION RESTORE
# ============================================================

DRIVE_FOLDER = "pasture-mapping-adamawa"
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROCESSED_DIR = "/content/data/processed/final"
SUITABILITY_DIR = os.path.join(PROCESSED_DIR, "suitability")
CAPACITY_DIR = os.path.join(PROCESSED_DIR, "capacity")
ROUTING_DIR = os.path.join(PROCESSED_DIR, "routing")

month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

manifest_path = os.path.join(PROCESSED_DIR, "manifest.json")
with open(manifest_path, "r") as f:
    manifest = json.load(f)

GRID_WIDTH = manifest["grid_width"]
GRID_HEIGHT = manifest["grid_height"]
GRID_TRANSFORM = manifest["grid_transform"]
GRID_CRS = manifest["crs"]
transform_obj = rasterio.Affine(*GRID_TRANSFORM[:6])
PIXEL_SIZE_M = abs(transform_obj[0])  # 1000 m

# --- Load cost surface (Phase 6 Cell 2) ---
cost_path = os.path.join(ROUTING_DIR, "cost_surface.tif")
assert os.path.exists(cost_path), "⚠️ cost_surface.tif not found. Run Phase 6 Cell 2 first."
with rasterio.open(cost_path) as src:
    cost_surface = src.read(1)
    cost_surface = np.where(cost_surface == src.nodata, np.nan, cost_surface)
print(f"✅ Loaded cost surface: shape {cost_surface.shape}")

# --- Load zone raster + attributes (Phase 4/5) ---
zone_raster_path = os.path.join(SUITABILITY_DIR, "grazing_zones_raster.tif")
with rasterio.open(zone_raster_path) as src:
    zone_raster = src.read(1)

zone_csv_path = os.path.join(SUITABILITY_DIR, "grazing_zones_attributes.csv")
zone_df_attrs = pd.read_csv(zone_csv_path)
print(f"✅ Loaded {len(zone_df_attrs)} zone attributes")

# --- Load zone x month capacity table (Phase 5) ---
zone_month_capacity_path = os.path.join(CAPACITY_DIR, "zone_month_capacity.csv")
zone_month_capacity_df = pd.read_csv(zone_month_capacity_path)
print(f"✅ Loaded zone-month capacity table: {len(zone_month_capacity_df)} rows")

TLU_CONVERSION = {
    "cattle_adult_zebu": 1.00, "cattle_young": 0.50, "camel": 1.10,
    "horse": 0.80, "donkey": 0.50, "sheep_goat": 0.10,
}

def cattle_to_tlu(n_cattle, livestock_type):
    return n_cattle * TLU_CONVERSION[livestock_type]

# --- Coordinate transformers ---
transformer_to_wgs84 = Transformer.from_crs(GRID_CRS, "EPSG:4326", always_xy=True)
transformer_to_utm = Transformer.from_crs("EPSG:4326", GRID_CRS, always_xy=True)

def latlon_to_rowcol(lat, lon):
    x, y = transformer_to_utm.transform(lon, lat)
    col, row = ~transform_obj * (x, y)
    return int(round(row)), int(round(col))

def rowcol_to_latlon(row, col):
    x, y = transform_obj * (col, row)
    lon, lat = transformer_to_wgs84.transform(x, y)
    return lat, lon

# ============================================================
# STEP 1 — Build the routable graph (8-connectivity, vectorized)
# ============================================================

print("\nBuilding routing graph (8-connectivity)...")

valid_mask = ~np.isnan(cost_surface)
node_ids = np.full((GRID_HEIGHT, GRID_WIDTH), -1, dtype="int64")
node_ids[valid_mask] = np.arange(valid_mask.sum())
n_nodes = valid_mask.sum()
print(f"   Valid nodes: {n_nodes}")

# 4 unique directions avoid double-adding undirected edges: (dr, dc, distance_m)
directions = [
    (0, 1, PIXEL_SIZE_M),                          # right
    (1, 0, PIXEL_SIZE_M),                          # down
    (1, 1, PIXEL_SIZE_M * np.sqrt(2)),             # down-right (diagonal)
    (1, -1, PIXEL_SIZE_M * np.sqrt(2)),            # down-left (diagonal)
]

edges_u, edges_v, edges_weight, edges_dist = [], [], [], []

for dr, dc, dist_m in directions:
    # Shift grids to find valid neighbor pairs, vectorized (no python-level pixel loop)
    r0, r1 = max(0, -dr), GRID_HEIGHT - max(0, dr)
    c0, c1 = max(0, -dc), GRID_WIDTH - max(0, dc)

    src_ids = node_ids[r0:r1, c0:c1]
    dst_ids = node_ids[r0+dr:r1+dr, c0+dc:c1+dc]
    src_cost = cost_surface[r0:r1, c0:c1]
    dst_cost = cost_surface[r0+dr:r1+dr, c0+dc:c1+dc]

    both_valid = (src_ids >= 0) & (dst_ids >= 0)

    u = src_ids[both_valid]
    v = dst_ids[both_valid]
    avg_cost = (src_cost[both_valid] + dst_cost[both_valid]) / 2.0
    w = avg_cost * dist_m  # cost-weighted edge weight (used for routing)
    d = np.full(u.shape, dist_m)  # physical distance (used for trek-length/day estimate)

    edges_u.append(u); edges_v.append(v)
    edges_weight.append(w); edges_dist.append(d)

edges_u = np.concatenate(edges_u)
edges_v = np.concatenate(edges_v)
edges_weight = np.concatenate(edges_weight)
edges_dist = np.concatenate(edges_dist)

print(f"   Edges built: {len(edges_u)}")

G = nx.Graph()
G.add_nodes_from(range(n_nodes))
edge_tuples = [
    (int(u), int(v), {"weight": float(w), "dist_m": float(d)})
    for u, v, w, d in zip(edges_u, edges_v, edges_weight, edges_dist)
]
G.add_edges_from(edge_tuples)

print(f"✅ Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# --- Zone centroid -> nearest valid node lookup ---
zone_centroid_nodes = {}
for _, row in zone_df_attrs.iterrows():
    zid = int(row["zone_id"])
    zone_mask = (zone_raster == zid)
    rows_idx, cols_idx = np.where(zone_mask)
    center_row = int(round(rows_idx.mean()))
    center_col = int(round(cols_idx.mean()))
    if node_ids[center_row, center_col] >= 0:
        zone_centroid_nodes[zid] = int(node_ids[center_row, center_col])
    else:
        # fallback: nearest valid pixel within the zone itself
        dists = (rows_idx - center_row)**2 + (cols_idx - center_col)**2
        nearest_idx = np.argmin(dists)
        zone_centroid_nodes[zid] = int(node_ids[rows_idx[nearest_idx], cols_idx[nearest_idx]])

print(f"✅ Mapped {len(zone_centroid_nodes)} zone centroids to graph nodes")

# ============================================================
# STEP 2 — find_optimal_route: core routing function
# ============================================================

TRANSHUMANCE_SPEED_KM_PER_DAY = 15  # assumption: typical herded cattle walking pace

def find_nearest_valid_node(lat, lon, search_radius=5):
    """Snaps a GPS point to the nearest valid graph node, searching outward if needed."""
    row, col = latlon_to_rowcol(lat, lon)
    if 0 <= row < GRID_HEIGHT and 0 <= col < GRID_WIDTH and node_ids[row, col] >= 0:
        return node_ids[row, col], row, col

    for radius in range(1, search_radius + 1):
        for dr in range(-radius, radius + 1):
            for dc in range(-radius, radius + 1):
                rr, cc = row + dr, col + dc
                if 0 <= rr < GRID_HEIGHT and 0 <= cc < GRID_WIDTH and node_ids[rr, cc] >= 0:
                    return node_ids[rr, cc], rr, cc
    return None, row, col

def find_optimal_route(start_lat, start_lon, n_cattle, month, livestock_type="cattle_adult_zebu"):
    """
    Finds the nearest reachable grazing zone that can sustain n_cattle in `month`,
    and returns the optimal (least-cost) route from the herder's GPS location.
    """
    month_name = normalize_month(month)
    requested_tlu = cattle_to_tlu(n_cattle, livestock_type)

    start_node, start_row, start_col = find_nearest_valid_node(start_lat, start_lon)
    if start_node is None:
        return {"error": "Could not snap start location to the routable grid — "
                          "point may be far outside the study area."}

    # Qualifying zones for this month
    month_data = zone_month_capacity_df[zone_month_capacity_df["month"] == month_name]
    qualifying_zone_ids = month_data[month_data["capacity_tlu"] >= requested_tlu]["zone_id"].tolist()

    if len(qualifying_zone_ids) == 0:
        return {"error": f"No zone can sustain {n_cattle} {livestock_type} in {month_name}."}

    # Single Dijkstra run from start — then look up cost to each qualifying zone centroid
    distances, paths = nx.single_source_dijkstra(G, source=start_node, weight="weight")

    reachable_candidates = [
        (zid, distances[zone_centroid_nodes[zid]])
        for zid in qualifying_zone_ids
        if zone_centroid_nodes.get(zid) in distances
    ]

    if len(reachable_candidates) == 0:
        return {"error": "No qualifying zone is reachable from this location "
                          "(may be cut off by hard barriers)."}

    best_zone_id, best_cost = min(reachable_candidates, key=lambda x: x[1])
    best_node = zone_centroid_nodes[best_zone_id]
    path_nodes = paths[best_node]

    # Convert path node IDs back to row/col -> lat/lon waypoints
    inv_node_ids = np.full(n_nodes, -1, dtype="int64")
    rows_all, cols_all = np.where(valid_mask)
    inv_node_ids[node_ids[valid_mask]] = np.arange(len(rows_all))
    # simpler direct mapping: build once
    node_to_rc = {int(node_ids[r, c]): (r, c) for r, c in zip(rows_all, cols_all)}

    waypoints = []
    total_physical_dist_m = 0.0
    for i, node in enumerate(path_nodes):
        r, c = node_to_rc[node]
        lat, lon = rowcol_to_latlon(r, c)
        waypoints.append({"lat": round(lat, 6), "lon": round(lon, 6)})
        if i > 0:
            total_physical_dist_m += G[path_nodes[i-1]][node]["dist_m"]

    total_distance_km = total_physical_dist_m / 1000
    estimated_days = total_distance_km / TRANSHUMANCE_SPEED_KM_PER_DAY

    zone_info = zone_df_attrs[zone_df_attrs["zone_id"] == best_zone_id].iloc[0].to_dict()
    zone_capacity_row = month_data[month_data["zone_id"] == best_zone_id].iloc[0]

    return {
        "zone_id": int(best_zone_id),
        "zone_info": {
            "quality": zone_info["quality"],
            "area_km2": zone_info["area_km2"],
            "mean_score": zone_info["mean_score"],
            "best_month": zone_info["best_month"],
            "worst_month": zone_info["worst_month"],
        },
        "capacity_info": {
            "month": month_name,
            "capacity_tlu": float(zone_capacity_row["capacity_tlu"]),
            "requested_tlu": round(requested_tlu, 2),
            "surplus_tlu": round(float(zone_capacity_row["capacity_tlu"]) - requested_tlu, 2),
        },
        "route": {
            "waypoints": waypoints,
            "n_waypoints": len(waypoints),
            "total_distance_km": round(total_distance_km, 1),
            "estimated_travel_days": round(estimated_days, 1),
            "network_cost": round(best_cost, 2),
        },
    }

# ============================================================
# STEP 3 — Test the routing function
# ============================================================

print("\n" + "=" * 60)
print("TESTING find_optimal_route()")
print("=" * 60)

# Test point: a location inside the study area, offset from the grid edge
# (using a point 15% into the grid from top-left as a stand-in "herder GPS location")
test_row = int(GRID_HEIGHT * 0.2)
test_col = int(GRID_WIDTH * 0.2)
# find nearest valid pixel near that location
while node_ids[test_row, test_col] < 0:
    test_row += 1
test_lat, test_lon = rowcol_to_latlon(test_row, test_col)

print(f"\n🔍 Test herder location: lat={test_lat:.5f}, lon={test_lon:.5f}")
print(f"   Query: 50 adult zebu cattle, month='August'")

route_result = find_optimal_route(test_lat, test_lon, 50, "August", "cattle_adult_zebu")

if "error" in route_result:
    print(f"❌ {route_result['error']}")
else:
    print(f"\n✅ Route found to Zone {route_result['zone_id']}:")
    print(f"   Quality: {route_result['zone_info']['quality']}")
    print(f"   Zone area: {route_result['zone_info']['area_km2']} km²")
    print(f"   Capacity: {route_result['capacity_info']['capacity_tlu']} TLU "
          f"(requested: {route_result['capacity_info']['requested_tlu']} TLU, "
          f"surplus: {route_result['capacity_info']['surplus_tlu']} TLU)")
    print(f"   Route distance: {route_result['route']['total_distance_km']} km")
    print(f"   Estimated travel time: {route_result['route']['estimated_travel_days']} days")
    print(f"   Waypoints: {route_result['route']['n_waypoints']}")

# ============================================================
# STEP 4 — Export route as GeoJSON
# ============================================================

if "error" not in route_result:
    coords = [(wp["lon"], wp["lat"]) for wp in route_result["route"]["waypoints"]]
    route_line = LineString(coords)

    route_gdf = gpd.GeoDataFrame(
        [{
            "zone_id": route_result["zone_id"],
            "quality": route_result["zone_info"]["quality"],
            "distance_km": route_result["route"]["total_distance_km"],
            "travel_days": route_result["route"]["estimated_travel_days"],
            "capacity_tlu": route_result["capacity_info"]["capacity_tlu"],
            "geometry": route_line,
        }],
        crs="EPSG:4326"
    )

    route_geojson_path = os.path.join(ROUTING_DIR, "test_route.geojson")
    route_gdf.to_file(route_geojson_path, driver="GeoJSON")
    print(f"\n✅ Saved: {route_geojson_path}")

# ============================================================
# PATCH — Visualization: rebuild path from waypoints (lat/lon),
# not internal function variables (path_nodes/node_to_rc don't
# exist outside find_optimal_route's scope)
# ============================================================

if "error" not in route_result:
    fig, ax = plt.subplots(1, 1, figsize=(11, 10))
    ax.imshow(cost_surface, cmap="RdYlGn_r", vmin=0, vmax=1, extent=[0, GRID_WIDTH, GRID_HEIGHT, 0])

    # Convert each waypoint's lat/lon back to row/col for plotting
    path_rows, path_cols = [], []
    for wp in route_result["route"]["waypoints"]:
        r, c = latlon_to_rowcol(wp["lat"], wp["lon"])
        path_rows.append(r)
        path_cols.append(c)

    ax.plot(path_cols, path_rows, color="blue", linewidth=2.5, label="Optimal route")
    ax.scatter([test_col], [test_row], color="black", s=100, marker="*", label="Herder start", zorder=5)

    zone_mask_best = (zone_raster == route_result["zone_id"])
    ax.contour(zone_mask_best, colors="cyan", linewidths=1)

    ax.set_title(f"Optimal Route to Zone {route_result['zone_id']}\n"
                 f"{route_result['route']['total_distance_km']} km, "
                 f"~{route_result['route']['estimated_travel_days']} days")
    ax.legend(loc="upper right")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

print("\n" + "=" * 60)
print("✅ PHASE 6 COMPLETE — Routing engine built, tested, and exported")
print("=" * 60)

##Phase 7

###Feature Engineering

In [ ]:
# ============================================================
# PHASE 7 — CELL 1
# Feature Engineering for Biomass Prediction
# Features: NDVI lag1/lag2, rainfall lag1/lag2, month, elevation,
# slope, water distance, landcover, previous-month biomass
# Target: current-month biomass (kg DM/ha)
# WITH VISUAL QA
# ============================================================

import os
import json
import glob
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

!pip install -q pyarrow

# ============================================================
# SESSION RESTORE
# ============================================================

DRIVE_FOLDER = "pasture-mapping-adamawa"
START_YEAR = 2016
END_YEAR = 2025

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROCESSED_DIR = "/content/data/processed/final"
ML_DIR = os.path.join(PROCESSED_DIR, "ml")
os.makedirs(ML_DIR, exist_ok=True)

month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

manifest_path = os.path.join(PROCESSED_DIR, "manifest.json")
with open(manifest_path, "r") as f:
    manifest = json.load(f)

GRID_WIDTH = manifest["grid_width"]
GRID_HEIGHT = manifest["grid_height"]
NDVI_YEARS = manifest["ndvi_years_available"]
CHIRPS_YEARS = manifest["chirps_years_available"]

print(f"📋 Grid: {GRID_WIDTH} x {GRID_HEIGHT}")
print(f"📋 NDVI years: {NDVI_YEARS}")
print(f"📋 CHIRPS years: {CHIRPS_YEARS}")

# ============================================================
# STEP 1 — Reload NDVI 4D array (years, 12, H, W)
# ============================================================

print("\nLoading NDVI...")
n_years = len(NDVI_YEARS)
ndvi_4d = np.full((n_years, 12, GRID_HEIGHT, GRID_WIDTH), np.nan, dtype="float32")

for y_idx, year in enumerate(NDVI_YEARS):
    path = os.path.join(PROCESSED_DIR, "ndvi", f"NDVI_{year}.tif")
    with rasterio.open(path) as src:
        data = src.read()
        nodata = src.nodata if src.nodata is not None else -9999.0
        ndvi_4d[y_idx] = np.where(data == nodata, np.nan, data)

print(f"✅ NDVI loaded: shape {ndvi_4d.shape}")

# ============================================================
# STEP 2 — Recompute biomass 4D from NDVI (same formula as Phase 3)
# ============================================================

biomass_4d = 3500 * ndvi_4d - 250
biomass_4d = np.where(np.isnan(ndvi_4d), np.nan, np.clip(biomass_4d, 0, None))
print(f"✅ Biomass recomputed: range {np.nanmin(biomass_4d):.1f}-{np.nanmax(biomass_4d):.1f} kg DM/ha")

# ============================================================
# STEP 3 — Reload CHIRPS rainfall 4D, pad missing 2025 months
# with 2016-2024 climatology mean (per error note 11)
# ============================================================

print("\nLoading CHIRPS rainfall...")
n_chirps_years = len(CHIRPS_YEARS)
rainfall_4d = np.full((n_chirps_years, 12, GRID_HEIGHT, GRID_WIDTH), np.nan, dtype="float32")

for y_idx, year in enumerate(CHIRPS_YEARS):
    path = os.path.join(PROCESSED_DIR, "chirps", f"CHIRPS_{year}.tif")
    with rasterio.open(path) as src:
        data = src.read()
        nodata = src.nodata if src.nodata is not None else -9999.0
        n_bands = data.shape[0]
        rainfall_4d[y_idx, :n_bands] = np.where(data == nodata, np.nan, data)

# Identify missing months (all-NaN slices, e.g. late 2025)
missing_month_mask = np.all(np.isnan(rainfall_4d), axis=(2, 3))  # (n_years, 12)
n_missing = missing_month_mask.sum()
print(f"   Missing rainfall year-months detected: {n_missing}")

if n_missing > 0:
    # Climatology from years that DO have data for each month (typically 2016-2024)
    rainfall_climatology = np.nanmean(rainfall_4d, axis=0)  # (12, H, W)
    for y_idx in range(n_chirps_years):
        for m in range(12):
            if missing_month_mask[y_idx, m]:
                rainfall_4d[y_idx, m] = rainfall_climatology[m]
                print(f"   Padded {CHIRPS_YEARS[y_idx]}-{month_names[m]} with climatology mean")

print(f"✅ Rainfall loaded + padded: shape {rainfall_4d.shape}")

# Align rainfall years to NDVI years (should already match, but guard against mismatch)
assert NDVI_YEARS == CHIRPS_YEARS, \
    f"⚠️ Year mismatch: NDVI={NDVI_YEARS}, CHIRPS={CHIRPS_YEARS}. Using NDVI years as reference."

# ============================================================
# STEP 4 — Load static layers
# ============================================================

def load_static(name):
    path = os.path.join(PROCESSED_DIR, "static", name)
    with rasterio.open(path) as src:
        data = src.read(1).astype("float32")
        nodata = src.nodata
        return np.where(data == nodata, np.nan, data)

elevation = load_static("elevation.tif")
slope = load_static("slope.tif")
dist_water = load_static("distance_to_water.tif")
landcover = load_static("landcover.tif")

print(f"\n✅ Static layers loaded")

# ============================================================
# STEP 5 — Build continuous time axis + valid pixel mask
# ============================================================

n_timesteps = n_years * 12  # e.g. 120 for 10 years

ndvi_flat_t = ndvi_4d.reshape(n_timesteps, GRID_HEIGHT, GRID_WIDTH)
rain_flat_t = rainfall_4d.reshape(n_timesteps, GRID_HEIGHT, GRID_WIDTH)
biomass_flat_t = biomass_4d.reshape(n_timesteps, GRID_HEIGHT, GRID_WIDTH)

# Valid pixel = has static data AND has NDVI data in every timestep
valid_pixel_mask = (
    ~np.isnan(elevation) & ~np.isnan(slope) & ~np.isnan(dist_water) & ~np.isnan(landcover)
    & ~np.any(np.isnan(ndvi_flat_t), axis=0)
)
n_valid_pixels = valid_pixel_mask.sum()
print(f"✅ Valid pixels for ML dataset: {n_valid_pixels}")

valid_rows, valid_cols = np.where(valid_pixel_mask)

# ============================================================
# STEP 6 — Vectorized feature construction
# For each timestep t (t>=2), predict biomass(t) using data up to t-1
# ============================================================

print("\nBuilding feature dataset (vectorized across time, per pixel)...")

# --- Optional subsampling to keep the dataset tractable in Colab ---
MAX_PIXELS = 15000  # None to disable subsampling and use all valid pixels
np.random.seed(42)

if MAX_PIXELS is not None and n_valid_pixels > MAX_PIXELS:
    sample_idx = np.random.choice(n_valid_pixels, size=MAX_PIXELS, replace=False)
    sample_rows = valid_rows[sample_idx]
    sample_cols = valid_cols[sample_idx]
    print(f"   Subsampling {MAX_PIXELS} of {n_valid_pixels} valid pixels for tractability")
else:
    sample_rows, sample_cols = valid_rows, valid_cols
    print(f"   Using all {n_valid_pixels} valid pixels")

n_sample_pixels = len(sample_rows)

# Extract per-pixel static values once
elev_vals = elevation[sample_rows, sample_cols]
slope_vals = slope[sample_rows, sample_cols]
water_vals = dist_water[sample_rows, sample_cols]
lc_vals = landcover[sample_rows, sample_cols]

records = []
for t in range(2, n_timesteps):
    year_idx = t // 12
    month_idx = t % 12

    ndvi_lag1 = ndvi_flat_t[t-1, sample_rows, sample_cols]
    ndvi_lag2 = ndvi_flat_t[t-2, sample_rows, sample_cols]
    rain_lag1 = rain_flat_t[t-1, sample_rows, sample_cols]
    rain_lag2 = rain_flat_t[t-2, sample_rows, sample_cols]
    prev_biomass = biomass_flat_t[t-1, sample_rows, sample_cols]
    target_biomass = biomass_flat_t[t, sample_rows, sample_cols]

    chunk = pd.DataFrame({
        "year": NDVI_YEARS[year_idx],
        "month": month_idx + 1,
        "row": sample_rows,
        "col": sample_cols,
        "elevation": elev_vals,
        "slope": slope_vals,
        "dist_water": water_vals,
        "landcover": lc_vals,
        "ndvi_lag1": ndvi_lag1,
        "ndvi_lag2": ndvi_lag2,
        "rainfall_lag1": rain_lag1,
        "rainfall_lag2": rain_lag2,
        "prev_month_biomass": prev_biomass,
        "target_biomass": target_biomass,
    })
    records.append(chunk)

feature_df = pd.concat(records, ignore_index=True)
print(f"✅ Raw feature rows before cleaning: {len(feature_df)}")

# --- Drop rows with any NaN (e.g. target/lag pixels that were nodata) ---
before = len(feature_df)
feature_df = feature_df.dropna().reset_index(drop=True)
print(f"✅ Rows after dropping NaN: {len(feature_df)} ({before - len(feature_df)} dropped)")

# --- Cyclical month encoding (helps tree models less, but essential for LSTM/regression) ---
feature_df["month_sin"] = np.sin(2 * np.pi * feature_df["month"] / 12)
feature_df["month_cos"] = np.cos(2 * np.pi * feature_df["month"] / 12)

# --- Save ---
feature_path = os.path.join(ML_DIR, "biomass_prediction_features.parquet")
feature_df.to_parquet(feature_path, index=False)
print(f"✅ Saved: {feature_path}")

# ============================================================
# VISUAL QA
# ============================================================

print("\n📊 Feature dataset summary:")
print(feature_df.describe().T[["mean", "std", "min", "max"]].to_string())

# --- Feature correlation with target ---
corr_cols = ["elevation", "slope", "dist_water", "landcover", "ndvi_lag1", "ndvi_lag2",
             "rainfall_lag1", "rainfall_lag2", "prev_month_biomass", "month_sin", "month_cos",
             "target_biomass"]
corr_matrix = feature_df[corr_cols].corr()

fig, ax = plt.subplots(1, 1, figsize=(9, 7))
im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols))); ax.set_xticklabels(corr_cols, rotation=90)
ax.set_yticks(range(len(corr_cols))); ax.set_yticklabels(corr_cols)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr_matrix.values[i,j]:.2f}", ha="center", va="center",
                fontsize=7, color="black")
ax.set_title("Feature Correlation Matrix (incl. target)")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

print(f"\n📊 Correlation with target_biomass:")
print(corr_matrix["target_biomass"].sort_values(ascending=False).to_string())

# --- Sample pixel time series: prev_month_biomass vs target_biomass ---
sample_pixel_row, sample_pixel_col = sample_rows[0], sample_cols[0]
pixel_series = feature_df[(feature_df["row"] == sample_pixel_row) & (feature_df["col"] == sample_pixel_col)]
pixel_series = pixel_series.sort_values(["year", "month"])

fig, ax = plt.subplots(1, 1, figsize=(12, 5))
x_labels = [f"{y}-{m:02d}" for y, m in zip(pixel_series["year"], pixel_series["month"])]
ax.plot(range(len(pixel_series)), pixel_series["prev_month_biomass"], label="prev_month_biomass (feature)", alpha=0.7)
ax.plot(range(len(pixel_series)), pixel_series["target_biomass"], label="target_biomass (label)", alpha=0.7)
ax.set_xticks(range(0, len(pixel_series), 12))
ax.set_xticklabels(x_labels[::12], rotation=45)
ax.set_ylabel("Biomass (kg DM/ha)")
ax.set_title(f"Sample Pixel Time Series (row={sample_pixel_row}, col={sample_pixel_col})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- NDVI lag1 vs target biomass scatter (sanity check the core physical relationship) ---
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
sample_for_plot = feature_df.sample(min(20000, len(feature_df)), random_state=42)
ax.scatter(sample_for_plot["ndvi_lag1"], sample_for_plot["target_biomass"], s=2, alpha=0.2, color="#2ca02c")
ax.set_xlabel("NDVI (1 month lag)")
ax.set_ylabel("Target biomass (kg DM/ha)")
ax.set_title("NDVI Lag1 vs Target Biomass — should show clear positive relationship")
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("✅ PHASE 7 CELL 1 COMPLETE")
print(f"   Feature dataset: {len(feature_df)} rows, {len(feature_df.columns)} columns")
print(f"   Saved: {feature_path}")
print("=" * 60)

###Train and Compare Random Forest, XGBoost and LSTM

In [ ]:
# ============================================================
# PHASE 7 — CELL 2
# Train and Compare: Random Forest, XGBoost, LSTM
# WITH VISUAL QA + VISIBLE TRAINING PROGRESS
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

!pip install -q xgboost
import xgboost as xgb

!pip install -q tensorflow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM as LSTMLayer, Dense
from tensorflow.keras.callbacks import EarlyStopping

import joblib

# ============================================================
# SESSION RESTORE
# ============================================================

DRIVE_FOLDER = "pasture-mapping-adamawa"
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROCESSED_DIR = "/content/data/processed/final"
ML_DIR = os.path.join(PROCESSED_DIR, "ml")

feature_path = os.path.join(ML_DIR, "biomass_prediction_features.parquet")
assert os.path.exists(feature_path), "⚠️ Run Phase 7 Cell 1 first."
feature_df = pd.read_parquet(feature_path)
print(f"✅ Loaded feature dataset: {len(feature_df)} rows")

FEATURE_COLS = ["elevation", "slope", "dist_water", "landcover",
                 "ndvi_lag1", "ndvi_lag2", "rainfall_lag1", "rainfall_lag2",
                 "prev_month_biomass", "month_sin", "month_cos"]
TARGET_COL = "target_biomass"

# ============================================================
# STEP 1 — Temporal train/test split (NOT random — last year held out)
# ============================================================

test_year = feature_df["year"].max()  # 2025 — most recent year
train_df = feature_df[feature_df["year"] != test_year].copy()
test_df = feature_df[feature_df["year"] == test_year].copy()

print(f"\n📊 Temporal split:")
print(f"   Train: {len(train_df)} rows (years {sorted(train_df['year'].unique())})")
print(f"   Test:  {len(test_df)} rows (year {test_year})")

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET_COL]

# ============================================================
# STEP 2 — Random Forest (verbose=2 shows tree-building progress)
# ============================================================

print("\n" + "=" * 60)
print("TRAINING RANDOM FOREST")
print("=" * 60)

rf_model = RandomForestRegressor(
    n_estimators=200, max_depth=15, min_samples_leaf=5,
    n_jobs=-1, random_state=42,
    verbose=2
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print(f"\n✅ Random Forest — MAE: {rf_mae:.1f}, RMSE: {rf_rmse:.1f}, R²: {rf_r2:.4f}")

# ============================================================
# STEP 3 — XGBoost (eval_set + verbose prints RMSE every 20 rounds)
# ============================================================

print("\n" + "=" * 60)
print("TRAINING XGBOOST")
print("=" * 60)

xgb_model = xgb.XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42,
    eval_metric="rmse"
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=20
)
xgb_pred = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_r2 = r2_score(y_test, xgb_pred)

print(f"\n✅ XGBoost — MAE: {xgb_mae:.1f}, RMSE: {xgb_rmse:.1f}, R²: {xgb_r2:.4f}")

# ============================================================
# STEP 4 — LSTM (verbose=1 shows per-epoch progress bar)
# ============================================================

print("\n" + "=" * 60)
print("TRAINING LSTM")
print("=" * 60)

def build_lstm_sequences(df):
    n = len(df)
    seq = np.zeros((n, 3, 7), dtype="float32")  # 3 timesteps, 7 features per step

    static_feats = df[["elevation", "slope", "dist_water", "landcover"]].values

    # timestep t-2
    seq[:, 0, 0] = df["ndvi_lag2"].values
    seq[:, 0, 1] = df["rainfall_lag2"].values
    seq[:, 0, 2:6] = static_feats
    seq[:, 0, 6] = 0

    # timestep t-1
    seq[:, 1, 0] = df["ndvi_lag1"].values
    seq[:, 1, 1] = df["rainfall_lag1"].values
    seq[:, 1, 2:6] = static_feats
    seq[:, 1, 6] = df["prev_month_biomass"].values

    # timestep t (target month calendar position only — no leakage)
    seq[:, 2, 0] = df["month_sin"].values
    seq[:, 2, 1] = df["month_cos"].values
    seq[:, 2, 2:6] = static_feats
    seq[:, 2, 6] = 0

    return seq

feature_means = X_train.mean()
feature_stds = X_train.std() + 1e-9

train_df_norm = train_df.copy()
test_df_norm = test_df.copy()
for col in FEATURE_COLS:
    train_df_norm[col] = (train_df[col] - feature_means[col]) / feature_stds[col]
    test_df_norm[col] = (test_df[col] - feature_means[col]) / feature_stds[col]

X_train_seq = build_lstm_sequences(train_df_norm)
X_test_seq = build_lstm_sequences(test_df_norm)

target_mean, target_std = y_train.mean(), y_train.std()
y_train_norm = (y_train - target_mean) / target_std
y_test_norm = (y_test - target_mean) / target_std

lstm_model = Sequential([
    LSTMLayer(32, input_shape=(3, 7), return_sequences=False),
    Dense(16, activation="relu"),
    Dense(1)
])
lstm_model.compile(optimizer="adam", loss="mse")

early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history = lstm_model.fit(
    X_train_seq, y_train_norm,
    validation_split=0.1,
    epochs=30, batch_size=512,
    callbacks=[early_stop],
    verbose=1
)

lstm_pred_norm = lstm_model.predict(X_test_seq, verbose=0).flatten()
lstm_pred = lstm_pred_norm * target_std + target_mean

lstm_mae = mean_absolute_error(y_test, lstm_pred)
lstm_rmse = np.sqrt(mean_squared_error(y_test, lstm_pred))
lstm_r2 = r2_score(y_test, lstm_pred)

print(f"\n✅ LSTM — MAE: {lstm_mae:.1f}, RMSE: {lstm_rmse:.1f}, R²: {lstm_r2:.4f}")

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(history.history["loss"], label="Train loss")
ax.plot(history.history["val_loss"], label="Val loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE (normalized)")
ax.set_title("LSTM Training Curve")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# STEP 5 — Model comparison
# ============================================================

comparison_df = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost", "LSTM"],
    "MAE": [rf_mae, xgb_mae, lstm_mae],
    "RMSE": [rf_rmse, xgb_rmse, lstm_rmse],
    "R2": [rf_r2, xgb_r2, lstm_r2],
})
print("\n" + "=" * 60)
print("MODEL COMPARISON (test year = 2025)")
print("=" * 60)
print(comparison_df.to_string(index=False))

best_model_name = comparison_df.loc[comparison_df["R2"].idxmax(), "Model"]
print(f"\n🏆 Best model by R²: {best_model_name}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric, better in zip(axes, ["MAE", "RMSE", "R2"], ["lower", "lower", "higher"]):
    best_idx = comparison_df[metric].idxmin() if better == "lower" else comparison_df[metric].idxmax()
    colors = ["#2ca02c" if i == best_idx else "#4169e1" for i in range(len(comparison_df))]
    ax.bar(comparison_df["Model"], comparison_df[metric], color=colors)
    ax.set_title(f"{metric} ({better} is better)")
    ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

pred_map = {"Random Forest": rf_pred, "XGBoost": xgb_pred, "LSTM": lstm_pred}
best_pred = pred_map[best_model_name]

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
sample_idx = np.random.choice(len(y_test), min(10000, len(y_test)), replace=False)
ax.scatter(y_test.values[sample_idx], best_pred[sample_idx], s=3, alpha=0.2, color="#2ca02c")
lims = [0, max(y_test.max(), best_pred.max())]
ax.plot(lims, lims, "r--", linewidth=1, label="Perfect prediction")
ax.set_xlabel("Actual biomass (kg DM/ha)")
ax.set_ylabel("Predicted biomass (kg DM/ha)")
ax.set_title(f"{best_model_name} — Predicted vs Actual (test year {test_year})")
ax.legend()
plt.tight_layout()
plt.show()

# ============================================================
# STEP 6 — Feature importance (RF and XGBoost)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

rf_importance = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values()
rf_importance.plot(kind="barh", ax=axes[0], color="#2ca02c")
axes[0].set_title("Random Forest — Feature Importance")

xgb_importance = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values()
xgb_importance.plot(kind="barh", ax=axes[1], color="#4169e1")
axes[1].set_title("XGBoost — Feature Importance")

plt.tight_layout()
plt.show()

print("\n⚠️ Reminder: ndvi_lag1 and prev_month_biomass are near-collinear "
      "(both derived from the same NDVI signal). Their importance is split "
      "between them — sum them together mentally when judging true impact.")
print(f"   Combined importance (RF): {rf_importance['ndvi_lag1'] + rf_importance['prev_month_biomass']:.3f}")
print(f"   Combined importance (XGB): {xgb_importance['ndvi_lag1'] + xgb_importance['prev_month_biomass']:.3f}")

# ============================================================
# STEP 7 — Save all three models
# ============================================================

rf_path = os.path.join(ML_DIR, "biomass_model_rf.pkl")
joblib.dump(rf_model, rf_path)

xgb_path = os.path.join(ML_DIR, "biomass_model_xgb.pkl")
joblib.dump(xgb_model, xgb_path)

lstm_path = os.path.join(ML_DIR, "biomass_model_lstm.h5")
lstm_model.save(lstm_path)

norm_stats_path = os.path.join(ML_DIR, "lstm_normalization_stats.json")
with open(norm_stats_path, "w") as f:
    json.dump({
        "feature_means": feature_means.to_dict(),
        "feature_stds": feature_stds.to_dict(),
        "target_mean": float(target_mean),
        "target_std": float(target_std),
    }, f, indent=2)

comparison_path = os.path.join(ML_DIR, "model_comparison.csv")
comparison_df.to_csv(comparison_path, index=False)

print(f"\n✅ Saved: {rf_path}")
print(f"✅ Saved: {xgb_path}")
print(f"✅ Saved: {lstm_path}")
print(f"✅ Saved: {norm_stats_path}")
print(f"✅ Saved: {comparison_path}")

print("\n" + "=" * 60)
print(f"✅ PHASE 7 CELL 2 COMPLETE — Best model: {best_model_name} (R²={comparison_df['R2'].max():.4f})")
print("=" * 60)

###Cross Validation

In [ ]:
# ============================================================
# PHASE 7 — CELL 3 (FAST VERSION)
# Cross-Validation: Temporal (4 representative years) + Spatial (4 blocks)
# Using Random Forest as the fast validation proxy model
# Speed optimizations: fewer trees, shallower depth, subsampled data
# WITH VISUAL QA
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================================
# SESSION RESTORE
# ============================================================

DRIVE_FOLDER = "pasture-mapping-adamawa"
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROCESSED_DIR = "/content/data/processed/final"
ML_DIR = os.path.join(PROCESSED_DIR, "ml")

feature_path = os.path.join(ML_DIR, "biomass_prediction_features.parquet")
feature_df = pd.read_parquet(feature_path)
print(f"✅ Loaded feature dataset: {len(feature_df)} rows")

FEATURE_COLS = ["elevation", "slope", "dist_water", "landcover",
                 "ndvi_lag1", "ndvi_lag2", "rainfall_lag1", "rainfall_lag2",
                 "prev_month_biomass", "month_sin", "month_cos"]
TARGET_COL = "target_biomass"

print("\n⚠️ Using Random Forest (fast to retrain) as the validation proxy model.")
print("   Tests whether FEATURES generalize across time/space — a necessary")
print("   condition regardless of which final model (RF/XGBoost/LSTM) is deployed.")

# --- Subsample for CV speed (validation doesn't need the full dataset) ---
CV_SAMPLE_SIZE = 300000
feature_df_cv = feature_df.sample(n=min(CV_SAMPLE_SIZE, len(feature_df)), random_state=42).copy()
print(f"✅ CV working dataset: {len(feature_df_cv)} rows (subsampled from {len(feature_df)})")

RF_CV_PARAMS = dict(n_estimators=30, max_depth=10, min_samples_leaf=20, n_jobs=-1, random_state=42)

# ============================================================
# PART A — TEMPORAL CROSS-VALIDATION
# 4 representative years: earliest, middle, a drought year, most recent
# ============================================================

print("\n" + "=" * 60)
print("TEMPORAL CROSS-VALIDATION (4 representative years)")
print("=" * 60)

all_years_available = sorted(feature_df_cv["year"].unique())
print(f"All years in data: {all_years_available}")

# Pick 4 representative years: earliest, ~1/3, ~2/3, latest
n = len(all_years_available)
selected_idx = sorted(set([0, n // 3, (2 * n) // 3, n - 1]))
temporal_test_years = [all_years_available[i] for i in selected_idx]
# Ensure exactly 4 (pad if rounding collapsed any duplicates)
while len(temporal_test_years) < 4:
    for y in all_years_available:
        if y not in temporal_test_years:
            temporal_test_years.append(y)
            break
temporal_test_years = sorted(temporal_test_years)[:4]

print(f"Selected years for temporal CV: {temporal_test_years}")

temporal_results = []

for held_out_year in temporal_test_years:
    train_fold = feature_df_cv[feature_df_cv["year"] != held_out_year]
    test_fold = feature_df_cv[feature_df_cv["year"] == held_out_year]

    X_train_f, y_train_f = train_fold[FEATURE_COLS], train_fold[TARGET_COL]
    X_test_f, y_test_f = test_fold[FEATURE_COLS], test_fold[TARGET_COL]

    model_f = RandomForestRegressor(**RF_CV_PARAMS)
    model_f.fit(X_train_f, y_train_f)
    pred_f = model_f.predict(X_test_f)

    mae = mean_absolute_error(y_test_f, pred_f)
    rmse = np.sqrt(mean_squared_error(y_test_f, pred_f))
    r2 = r2_score(y_test_f, pred_f)

    temporal_results.append({"held_out_year": held_out_year, "n_test_rows": len(test_fold),
                               "MAE": mae, "RMSE": rmse, "R2": r2})
    print(f"   Held out {held_out_year}: MAE={mae:.1f}, RMSE={rmse:.1f}, R²={r2:.4f} (n={len(test_fold)})")

temporal_cv_df = pd.DataFrame(temporal_results)
temporal_cv_path = os.path.join(ML_DIR, "temporal_cv_results.csv")
temporal_cv_df.to_csv(temporal_cv_path, index=False)

print(f"\n📊 Temporal CV summary:")
print(f"   Mean R²: {temporal_cv_df['R2'].mean():.4f} ± {temporal_cv_df['R2'].std():.4f}")
print(f"   Min R²: {temporal_cv_df['R2'].min():.4f} (year {temporal_cv_df.loc[temporal_cv_df['R2'].idxmin(), 'held_out_year']})")
print(f"   Max R²: {temporal_cv_df['R2'].max():.4f} (year {temporal_cv_df.loc[temporal_cv_df['R2'].idxmax(), 'held_out_year']})")
print(f"✅ Saved: {temporal_cv_path}")

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
colors = ["#d62728" if r2 < temporal_cv_df["R2"].mean() - temporal_cv_df["R2"].std() else "#2ca02c"
          for r2 in temporal_cv_df["R2"]]
ax.bar(temporal_cv_df["held_out_year"].astype(str), temporal_cv_df["R2"], color=colors)
ax.axhline(temporal_cv_df["R2"].mean(), color="black", linestyle="--", linewidth=1, label="Mean R²")
ax.set_xlabel("Held-out year")
ax.set_ylabel("R²")
ax.set_title("Temporal Cross-Validation — R² per Held-Out Year (4 sampled years)")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# ============================================================
# PART B — SPATIAL CROSS-VALIDATION (leave-one-block-out, 4 quadrants)
# ============================================================

print("\n" + "=" * 60)
print("SPATIAL CROSS-VALIDATION (4 quadrants)")
print("=" * 60)

row_mid = feature_df_cv["row"].median()
col_mid = feature_df_cv["col"].median()

def assign_block(row, col):
    r_half = "N" if row < row_mid else "S"
    c_half = "W" if col < col_mid else "E"
    return r_half + c_half

feature_df_cv["spatial_block"] = feature_df_cv.apply(lambda x: assign_block(x["row"], x["col"]), axis=1)
print(f"Spatial blocks: {feature_df_cv['spatial_block'].value_counts().to_dict()}")

spatial_results = []
all_blocks = sorted(feature_df_cv["spatial_block"].unique())

for held_out_block in all_blocks:
    train_fold = feature_df_cv[feature_df_cv["spatial_block"] != held_out_block]
    test_fold = feature_df_cv[feature_df_cv["spatial_block"] == held_out_block]

    X_train_f, y_train_f = train_fold[FEATURE_COLS], train_fold[TARGET_COL]
    X_test_f, y_test_f = test_fold[FEATURE_COLS], test_fold[TARGET_COL]

    model_f = RandomForestRegressor(**RF_CV_PARAMS)
    model_f.fit(X_train_f, y_train_f)
    pred_f = model_f.predict(X_test_f)

    mae = mean_absolute_error(y_test_f, pred_f)
    rmse = np.sqrt(mean_squared_error(y_test_f, pred_f))
    r2 = r2_score(y_test_f, pred_f)

    spatial_results.append({"held_out_block": held_out_block, "n_test_rows": len(test_fold),
                              "MAE": mae, "RMSE": rmse, "R2": r2})
    print(f"   Held out block {held_out_block}: MAE={mae:.1f}, RMSE={rmse:.1f}, R²={r2:.4f} (n={len(test_fold)})")

spatial_cv_df = pd.DataFrame(spatial_results)
spatial_cv_path = os.path.join(ML_DIR, "spatial_cv_results.csv")
spatial_cv_df.to_csv(spatial_cv_path, index=False)

print(f"\n📊 Spatial CV summary:")
print(f"   Mean R²: {spatial_cv_df['R2'].mean():.4f} ± {spatial_cv_df['R2'].std():.4f}")
print(f"✅ Saved: {spatial_cv_path}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

block_colors = {"NW": "#1f77b4", "NE": "#ff7f0e", "SW": "#2ca02c", "SE": "#d62728"}
sample_plot = feature_df_cv.sample(min(20000, len(feature_df_cv)), random_state=1)
for block, color in block_colors.items():
    subset = sample_plot[sample_plot["spatial_block"] == block]
    axes[0].scatter(subset["col"], subset["row"], s=1, color=color, label=block, alpha=0.5)
axes[0].invert_yaxis()
axes[0].set_title("Spatial CV Blocks (pixel locations)")
axes[0].legend(markerscale=10)
axes[0].set_xlabel("Column"); axes[0].set_ylabel("Row")

axes[1].bar(spatial_cv_df["held_out_block"], spatial_cv_df["R2"],
            color=[block_colors[b] for b in spatial_cv_df["held_out_block"]])
axes[1].axhline(spatial_cv_df["R2"].mean(), color="black", linestyle="--", label="Mean R²")
axes[1].set_title("Spatial CV — R² per Held-Out Block")
axes[1].set_ylabel("R²")
axes[1].legend()
axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("✅ PHASE 7 CELL 3 COMPLETE (fast version: 4 temporal + 4 spatial folds)")
print("=" * 60)
print(f"Temporal CV — Mean R²: {temporal_cv_df['R2'].mean():.4f} (std: {temporal_cv_df['R2'].std():.4f})")
print(f"Spatial CV  — Mean R²: {spatial_cv_df['R2'].mean():.4f} (std: {spatial_cv_df['R2'].std():.4f})")

if temporal_cv_df["R2"].std() < 0.05 and spatial_cv_df["R2"].std() < 0.05:
    print("✅ Low variance across folds — model performance is STABLE across both time and space.")
else:
    print("⚠️ Noticeable variance across folds — performance is somewhat sensitive to which "
          "year/region is held out. Worth discussing as a limitation in your defense.")

###3-Month Forward Forecasting + Final Model Save

In [ ]:
# ============================================================
# PHASE 7 — CELL 4
# 3-Month Forward Biomass Forecast + Final Model Package
# ============================================================

import os, json
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

DRIVE_FOLDER = "pasture-mapping-adamawa"
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROCESSED_DIR = "/content/data/processed/final"
ML_DIR = os.path.join(PROCESSED_DIR, "ml")
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

manifest_path = os.path.join(PROCESSED_DIR, "manifest.json")
with open(manifest_path) as f: manifest = json.load(f)
GRID_WIDTH, GRID_HEIGHT = manifest["grid_width"], manifest["grid_height"]
GRID_TRANSFORM, GRID_CRS = manifest["grid_transform"], manifest["crs"]
transform_obj = rasterio.Affine(*GRID_TRANSFORM[:6])
ref_meta = {"driver": "GTiff", "height": GRID_HEIGHT, "width": GRID_WIDTH, "count": 1,
            "dtype": "float32", "crs": GRID_CRS, "transform": transform_obj,
            "nodata": -9999.0, "tiled": True, "blockxsize": 256, "blockysize": 256, "compress": "lzw"}

lstm_model = load_model(
    os.path.join(ML_DIR, "biomass_model_lstm.h5"),
    compile=False
)
lstm_model.compile(optimizer="adam", loss="mse")  # fresh, minimal recompile — training config not needed for inference
print("✅ Loaded LSTM model (compile=False, recompiled fresh for inference)")
with open(os.path.join(ML_DIR, "lstm_normalization_stats.json")) as f:
    norm_stats = json.load(f)
print("✅ Loaded LSTM model + normalization stats")

def load_static(name):
    with rasterio.open(os.path.join(PROCESSED_DIR, "static", name)) as src:
        d = src.read(1).astype("float32")
        return np.where(d == src.nodata, np.nan, d)

elevation, slope = load_static("elevation.tif"), load_static("slope.tif")
dist_water, landcover = load_static("distance_to_water.tif"), load_static("landcover.tif")

NDVI_YEARS = manifest["ndvi_years_available"]
with rasterio.open(os.path.join(PROCESSED_DIR, "ndvi", f"NDVI_{NDVI_YEARS[-1]}.tif")) as src:
    ndvi_last_year = np.where(src.read() == src.nodata, np.nan, src.read())
with rasterio.open(os.path.join(PROCESSED_DIR, "chirps", f"CHIRPS_{NDVI_YEARS[-1]}.tif")) as src:
    rain_last_year = np.where(src.read() == src.nodata, np.nan, src.read())

biomass_last_year = np.clip(3500 * ndvi_last_year - 250, 0, None)
biomass_last_year = np.where(np.isnan(ndvi_last_year), np.nan, biomass_last_year)

valid_mask = ~np.isnan(elevation) & ~np.isnan(slope) & ~np.isnan(dist_water) & ~np.isnan(landcover)
rows, cols = np.where(valid_mask)
print(f"✅ Forecasting for {len(rows)} valid pixels")

def normalize_feat(vals, name):
    return (vals - norm_stats["feature_means"][name]) / norm_stats["feature_stds"][name]

forecast_maps = []
cur_ndvi_lag2 = ndvi_last_year[-2][rows, cols]
cur_ndvi_lag1 = ndvi_last_year[-1][rows, cols]
cur_rain_lag2 = rain_last_year[-2][rows, cols] if rain_last_year.shape[0] >= 2 else rain_last_year[-1][rows, cols]
cur_rain_lag1 = rain_last_year[-1][rows, cols]
cur_prev_biomass = biomass_last_year[-1][rows, cols]
last_month_idx = 11  # December, index of final month in the last year

elev_v, slope_v, water_v, lc_v = elevation[rows, cols], slope[rows, cols], dist_water[rows, cols], landcover[rows, cols]

print("\nForecasting 3 months forward...")
for step in range(3):
    month_idx = (last_month_idx + step + 1) % 12
    m_sin, m_cos = np.sin(2*np.pi*(month_idx+1)/12), np.cos(2*np.pi*(month_idx+1)/12)

    seq = np.zeros((len(rows), 3, 7), dtype="float32")
    seq[:,0,0] = normalize_feat(cur_ndvi_lag2, "ndvi_lag2")
    seq[:,0,1] = normalize_feat(cur_rain_lag2, "rainfall_lag2")
    seq[:,0,2] = normalize_feat(elev_v, "elevation"); seq[:,0,3] = normalize_feat(slope_v, "slope")
    seq[:,0,4] = normalize_feat(water_v, "dist_water"); seq[:,0,5] = normalize_feat(lc_v, "landcover")
    seq[:,1,0] = normalize_feat(cur_ndvi_lag1, "ndvi_lag1")
    seq[:,1,1] = normalize_feat(cur_rain_lag1, "rainfall_lag1")
    seq[:,1,2] = normalize_feat(elev_v, "elevation"); seq[:,1,3] = normalize_feat(slope_v, "slope")
    seq[:,1,4] = normalize_feat(water_v, "dist_water"); seq[:,1,5] = normalize_feat(lc_v, "landcover")
    seq[:,1,6] = normalize_feat(cur_prev_biomass, "prev_month_biomass")
    seq[:,2,0], seq[:,2,1] = m_sin, m_cos
    seq[:,2,2] = normalize_feat(elev_v, "elevation"); seq[:,2,3] = normalize_feat(slope_v, "slope")
    seq[:,2,4] = normalize_feat(water_v, "dist_water"); seq[:,2,5] = normalize_feat(lc_v, "landcover")

    pred_norm = lstm_model.predict(seq, verbose=0).flatten()
    pred_biomass = np.clip(pred_norm * norm_stats["target_std"] + norm_stats["target_mean"], 0, None)

    fmap = np.full((GRID_HEIGHT, GRID_WIDTH), np.nan, dtype="float32")
    fmap[rows, cols] = pred_biomass
    forecast_maps.append(fmap)

    # Roll forward: this step's prediction becomes next step's "prev_month_biomass"
    # NDVI proxy for next step derived from predicted biomass (inverse of AGB formula)
    pred_ndvi_proxy = np.clip((pred_biomass + 250) / 3500, -0.2, 1.0)
    cur_ndvi_lag2, cur_ndvi_lag1 = cur_ndvi_lag1, pred_ndvi_proxy
    cur_rain_lag2, cur_rain_lag1 = cur_rain_lag1, cur_rain_lag1  # no future rainfall known -> persist last value
    cur_prev_biomass = pred_biomass

    print(f"   Month +{step+1} ({month_names[month_idx]}): mean predicted biomass = {np.nanmean(fmap):.1f} kg DM/ha")

ML_DIR_OUT = ML_DIR
forecast_meta = ref_meta.copy(); forecast_meta.update({"count": 3})
forecast_path = os.path.join(ML_DIR_OUT, "biomass_forecast_next_3_months.tif")
with rasterio.open(forecast_path, "w", **forecast_meta) as dst:
    for i, fmap in enumerate(forecast_maps):
        dst.write(np.where(np.isnan(fmap), -9999.0, fmap).astype("float32"), i+1)
print(f"\n✅ Saved: {forecast_path}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, fmap in enumerate(forecast_maps):
    month_idx = (last_month_idx + i + 1) % 12
    im = axes[i].imshow(fmap, cmap="YlGn", vmin=0, vmax=3200)
    axes[i].set_title(f"Forecast: {month_names[month_idx]} (+{i+1} months)")
    axes[i].axis("off")
    plt.colorbar(im, ax=axes[i], fraction=0.046)
plt.suptitle("3-Month Forward Biomass Forecast", y=1.02)
plt.tight_layout(); plt.show()

print("\n" + "="*60)
print("✅ PHASE 7 COMPLETE")
print(f"   Forecast raster: {forecast_path}")
print(f"   Models saved: biomass_model_rf.pkl, biomass_model_xgb.pkl, biomass_model_lstm.h5")
print("="*60)

In [ ]:
import shutil
shutil.make_archive('/content/pasture_project_data', 'zip', '/content/data/processed/final')
print("✅ Zipped. Now download it:")

from google.colab import files
files.download('/content/pasture_project_data.zip')